# Post-feature model comparison and ensemble studies

This notebook contains post-feature-extraction experiments for the Swin Transformer, EVA-02, and CoAtNet representations.

## Experimental protocol and data-leakage prevention

All data-driven model selection, hyperparameter tuning, ensemble-weight optimization, and probability-calibration decisions in the executable selection procedures use training and validation data only. Configurations explicitly designated as fixed or pre-specified are treated as comparison configurations and are not optimized using test results. Before any fixed configuration is presented as the final proposed model, its origin should be confirmed against the experiment records.

The test split is used to report comparative performance for validation-ranked candidates and pre-specified configurations. Test metrics are descriptive only and must not be used to select or revise model architectures, hyperparameters, ensemble weights, metadata options, calibration parameters, or the final model. Test comparisons are interpretable under this protocol only when the evaluated configurations were defined or selected without reference to test performance.

## Data-partition policy

- **Training split:** fit estimators and training-only preprocessing steps.
- **Validation split:** choose candidate feature combinations, model variants, ensemble weights, and calibration temperatures. Validation metrics may be used for these decisions only as specified by each experiment.
- **Test split:** report final metrics for configurations fixed using the training/validation protocol. Do not choose or revise model members, hyperparameters, ensemble weights, metadata options, or calibration temperatures based on test results.

Test summaries are kept in validation-selected or pre-specified order, not ranked by test performance. The test-based ensemble-weight search from the source notebook has been replaced by a validation-only weight search. The notebook outputs have been cleared before publication.

**Important reproducibility note:** some cells contain fixed model hyperparameters or fixed ensemble weights as comparison configurations. Their provenance cannot be reconstructed from code alone. Confirm that any configuration used as the final proposed model was fixed using validation-only evidence or a pre-specified protocol, and do not change it after inspecting test metrics.

## Research questions by code cell

| Section | Research question | Selection/configuration rule | Role of TEST |
|---|---|---|---|
| 1A — Stage 1A concatenation study | How do concatenating one, two, or three backbone feature sets affect classical classifiers and voting? | Rank candidate configurations on validation Macro-F1, then Macro-AUC and MCC. | Report validation-selected candidates only; do not re-rank by TEST. |
| 1B — Stage 1B per-feature ML | Which classifier per feature and which pre-defined ensemble/voting scheme perform best? | Select the per-feature classifier and Top-5 candidates using validation metrics. Automatic weights are estimated from training data via cross-validation. | Report-only; preserve validation rank. |
| 2 — Representation and aggregation | Which preprocessing, aggregation, SMOTE, and voting settings are strongest? | Rank settings on validation Macro-F1, Macro-AUC, and MCC; select the Top-5 from validation. | Evaluate the selected shortlist without sorting candidates by TEST metrics. |
| 3A — Fixed-weight ensemble | How does a pre-specified three-member ensemble perform, and how does validation-calibrated temperature scaling affect it? | Members and weights are fixed as a comparison configuration; temperature is selected by validation NLL. | Report final metrics only; no test-label-based oracle/rescue analysis. |
| 3 — A/B/C/D variants | How do fixed weights, validation-tuned weights, voting rules, and an expanded member set compare? | Dynamic weights and temperatures are selected using validation only; A uses a fixed-weight reference configuration. | Report pre-defined variants; do not use TEST metrics to select a winner. |
| A — Dynamic ensemble weights | Can validation Macro-F1-based weight selection improve the three-member ensemble? | Search ensemble weights on validation Macro-F1; choose temperature on validation NLL. | Apply the frozen weights and temperature to the test predictions. |
| A — Fixed global weights | What is the performance of a fixed-weight ensemble with validation-selected temperature calibration? | Weights are pre-specified; temperature is selected by validation NLL. | Report-only. Do not revise fixed weights based on TEST. |
| Metadata-aware weight search | Which metadata-enabled ensemble weights maximize validation Macro-F1? | Grid search uses validation probabilities/labels only; temperature is selected by validation NLL. | One final evaluation using frozen weights and temperature. |
| Baseline comparison | Which pre-defined KNN/Logistic Regression baseline is strongest for each backbone feature set on validation? | Select one baseline per feature using validation Macro-F1, then Macro-AUC and MCC. | Refit the selected baseline on train + validation and evaluate once on TEST. |

## Expected local data layout

All code cells use `/content/skin_lesion_project` as the default project root and read `SKIN_LESION_PROJECT_ROOT` when that environment variable is set. The datasets, metadata CSV, split files, feature arrays, checkpoints, and generated metrics are not included in this repository.

```text
skin_lesion_project/
├── data/
│   ├── splits/
│   │   ├── train_split.csv
│   │   ├── val_split.csv
│   │   └── test_split.csv
│   └── metadata/
│       └── ISIC_2019_Training_Metadata.csv
├── features/
│   ├── swin_ft768/
│   ├── eva02/
│   └── coatnet_ft672/
└── outputs/
```

Review each section's configuration paths before running it. Some sections require additional Python packages, including LightGBM and PyTorch. This notebook was statically inspected and cleaned, but the complete GPU workflow has not been executed end to end in this environment.


## Section 1A — Stage 1A: feature concatenation, classical classifiers, and voting

**Research question:** Does concatenating one, two, or three backbone feature representations improve validation performance across KNN, Logistic Regression, LightGBM, and the pre-defined voting rules?

**Selection rule:** the candidate table is sorted using validation Macro-F1, then Macro-AUC and MCC. The Top-3 test rows are determined from that validation ranking. Test metrics are report-only and must not be used to change the selected configuration. The oracle-pair result is a validation diagnostic, not a deployable classifier.

In [ ]:
# STAGE 1A
# ================== Stage-1 (Verbose): Concat (1/2/3) × {KNN, LogReg, LGBM} + Hard/Soft + Oracle + Diversity ==================
# Leak-free: scaler fit ONLY on train → transform val/test.
# Prints each combination and saves outputs under the configurable OUT_DIR.

import os, json, math, warnings, itertools, random
from pathlib import Path
warnings.filterwarnings("ignore")

# ---------- CONFIG ----------
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
DRIVE_ROOT = str(PROJECT_ROOT / "outputs")
STAGE_NAME = "Stage1"
REGISTRY_JSON = f"{DRIVE_ROOT}/stage_registry.json"
OUT_DIR      = f"{DRIVE_ROOT}/{STAGE_NAME}"
SPLIT_DIR = str(PROJECT_ROOT / "data" / "splits")
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin_ft768"),
    "EVA": str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet_ft672"),
}
CLASS_NAMES = ['AK','BCC','BKL','DF','MEL','NV','SCC','VASC']
RNG_SEED = 42

# ---------- Imports ----------
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
import lightgbm as lgb
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, classification_report,
                             confusion_matrix, matthews_corrcoef, roc_curve, auc)
import matplotlib.pyplot as plt
from IPython.display import display

# ---------- Utils ----------
def ensure_dir(p): Path(p).mkdir(parents=True, exist_ok=True)
def set_seed(s=RNG_SEED):
    random.seed(s); np.random.seed(s)

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True)
    return s.values

def _read_split_csv(path):
    df = pd.read_csv(path)
    id_col=None
    for c in ["image_id","image","Image","filename","fname","img_id","name"]:
        if c in df.columns: id_col=c; break
    assert id_col is not None, f"{path}: id column missing"
    ids = _normalize_ids(df[id_col])
    if "label" in df.columns:
        y = df["label"].astype(int).values
    elif "class_name" in df.columns:
        classes = sorted(df["class_name"].unique().tolist())
        mp={c:i for i,c in enumerate(classes)}
        y = df["class_name"].map(mp).astype(int).values
    else:
        raise ValueError(f"{path}: label/class_name missing")
    return ids, y

def load_splits(split_dir):
    p = Path(split_dir)
    tr_ids, ytr = _read_split_csv(p/"train_split.csv")
    va_ids, yva = _read_split_csv(p/"val_split.csv")
    te_ids, yte = _read_split_csv(p/"test_split.csv")
    return (tr_ids,ytr), (va_ids,yva), (te_ids,yte)

def _load_features(dir_path, split):
    p = Path(dir_path)
    for nm in [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"]:
        f=p/nm
        if f.exists(): return np.load(f)
    raise FileNotFoundError(f"{dir_path}: {split} features missing")

def macro_auc_ovr(y, proba):
    return roc_auc_score(y, proba, multi_class="ovr", average="macro")

def micro_f1(y, yhat):
    return f1_score(y, yhat, average="micro", zero_division=0)

def compute_cr_df(y, yhat, class_names):
    rep = classification_report(y, yhat, target_names=class_names, zero_division=0, output_dict=True)
    df = pd.DataFrame(rep).T.reset_index().rename(columns={"index":"class"})
    return df[df["class"].isin(class_names)][["class","precision","recall","f1-score","support"]].reset_index(drop=True)

def fit_scaler_on_train_transform(Xtr, Xva, Xte):
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)

def make_concat(parts, FEATS):
    Xtr = np.concatenate([FEATS[p]["train"] for p in parts], axis=1)
    Xva = np.concatenate([FEATS[p]["val"] for p in parts], axis=1)
    Xte = np.concatenate([FEATS[p]["test"] for p in parts], axis=1)
    return Xtr, Xva, Xte

# Diversity metrics (per pair)
def pair_diversity(y, yhat_a, yhat_b):
    correct_a = (yhat_a == y).astype(int)
    correct_b = (yhat_b == y).astype(int)
    N = len(y)
    N11 = int(((correct_a==1) & (correct_b==1)).sum())
    N00 = int(((correct_a==0) & (correct_b==0)).sum())
    N10 = int(((correct_a==1) & (correct_b==0)).sum())
    N01 = int(((correct_a==0) & (correct_b==1)).sum())
    denom = (N11*N00 + N10*N01)
    Q = ((N11*N00 - N10*N01) / denom) if denom>0 else np.nan
    disagreement = (N10 + N01) / N
    if np.std(correct_a)>0 and np.std(correct_b)>0:
        corr = float(np.corrcoef(correct_a, correct_b)[0,1])
    else:
        corr = np.nan
    return {"Q": Q, "disagreement": float(disagreement), "corr_correctness": corr,
            "N11":N11, "N00":N00, "N10":N10, "N01":N01}

# Oracle utilities
def majority_vote(preds_list, n_classes):
    K,N = len(preds_list), len(preds_list[0])
    out = np.zeros(N, dtype=int)
    votes = np.stack(preds_list, axis=0)
    for i in range(N):
        cnt = np.bincount(votes[:,i], minlength=n_classes)
        out[i] = int(np.argmax(cnt))
    return out

def oracle_any_correct(y, preds_list, fallback=None):
    any_ok = np.any(np.stack([(p==y) for p in preds_list], axis=0), axis=0)
    if fallback is None:
        fallback = majority_vote(preds_list, len(np.unique(y)))
    pred = fallback.copy()
    pred[any_ok] = y[any_ok]
    return pred

def best_oracle_pair(y, preds_list, names):  # <<< fixed: returns NAME pair instead of index pair
    best_acc=-1; best_pair_names=None; best_pred=None
    for (i,j) in itertools.combinations(range(len(preds_list)), 2):
        p_or = oracle_any_correct(y, [preds_list[i], preds_list[j]], fallback=None)
        acc = accuracy_score(y, p_or)
        if acc > best_acc:
            best_acc, best_pair_names, best_pred = acc, (names[i], names[j]), p_or
    return best_pair_names, best_pred

# ---------- Load data ----------
set_seed(RNG_SEED)
ensure_dir(DRIVE_ROOT); ensure_dir(OUT_DIR)
(TR_IDS, Y_TR), (VA_IDS, Y_VA), (TE_IDS, Y_TE) = load_splits(SPLIT_DIR)
C = len(np.unique(Y_TR))
assert list(np.unique(Y_TR)) == list(range(C)), "Labels must be 0..C-1"

# Features
FEATS = {}
for name, d in FEATURE_DIRS.items():
    FEATS[name] = {
        "train": _load_features(d, "train"),
        "val"  : _load_features(d, "val"),
        "test" : _load_features(d, "test"),
    }

# Concat sets
CONCAT_SETS = {
    "BASE1:SWIN": ["SWIN"],
    "BASE1:EVA" : ["EVA"],
    "BASE1:COAT": ["COAT"],
    "BASE2:SWIN+EVA" : ["SWIN","EVA"],
    "BASE2:SWIN+COAT": ["SWIN","COAT"],
    "BASE2:EVA+COAT" : ["EVA","COAT"],
    "BASE3:SWIN+EVA+COAT": ["SWIN","EVA","COAT"],
}

# ---------- Train+Eval helpers ----------
def train_and_eval_models(Xtr, ytr, Xva, yva, verbose_prefix=""):
    Ztr, Zva, _ = fit_scaler_on_train_transform(Xtr, Xva, Xva)

    models = {
        "KNN":    KNeighborsClassifier(n_neighbors=15, weights="distance", p=2),
        "LOGREG": LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", n_jobs=-1, multi_class="auto"),
        "LGBM":   lgb.LGBMClassifier(n_estimators=600, num_leaves=48, learning_rate=0.05,
                                     subsample=0.9, colsample_bytree=0.7, reg_lambda=1.0,
                                     random_state=RNG_SEED, verbosity=-1),
    }

    preds_val, probas_val = {}, {}

    for name, mdl in models.items():
        mdl.fit(Ztr, ytr)
        P = mdl.predict_proba(Zva)
        yhat = np.argmax(P, axis=1)
        preds_val[name]  = yhat
        probas_val[name] = P

        # ---- print per-model VAL summary ----
        acc  = accuracy_score(yva, yhat)
        f1Ma = f1_score(yva, yhat, average="macro", zero_division=0)
        f1Mi = f1_score(yva, yhat, average="micro", zero_division=0)
        mcc  = matthews_corrcoef(yva, yhat)
        aucM = macro_auc_ovr(yva, P)

        print(f"\n[{verbose_prefix}] VAL — {name}")
        print(f"Acc={acc:.4f} | Macro-F1={f1Ma:.4f} | Micro-F1={f1Mi:.4f} | Macro-AUC={aucM:.4f} | MCC={mcc:.4f}")

        # CM (normalized)
        cm = confusion_matrix(yva, yhat, labels=list(range(C)))
        cm_norm = cm / cm.sum(axis=1, keepdims=True)
        cm_df = pd.DataFrame(cm_norm, index=[f"true_{c}" for c in CLASS_NAMES], columns=[f"pred_{c}" for c in CLASS_NAMES])
        print("Confusion Matrix (normalized):")
        display(cm_df.round(2))

        # CR horizontal
        cr_df = compute_cr_df(yva, yhat, CLASS_NAMES)
        print("Classification Report (per-class):")
        display(cr_df)

    return preds_val, probas_val

def metrics_block(y, yhat, proba):
    out = {}
    out["accuracy"]  = accuracy_score(y, yhat)
    out["macro_f1"]  = f1_score(y, yhat, average="macro", zero_division=0)
    out["micro_f1"]  = f1_score(y, yhat, average="micro", zero_division=0)
    out["mcc"]       = matthews_corrcoef(y, yhat)
    out["auc_macro"] = macro_auc_ovr(y, proba) if proba is not None else np.nan
    return out

def hard_vote(preds_list):
    return majority_vote(preds_list, C)

def soft_vote(probas_list):
    S = np.zeros_like(probas_list[0], dtype=float)
    for P in probas_list: S += P
    S /= len(probas_list)
    return S, np.argmax(S, axis=1)

# ---------- Run Stage-1 (verbose printing enabled) ----------
rows, cr_rows, diversity_rows = [], [], []

ensure_dir(OUT_DIR)

for key, parts in CONCAT_SETS.items():
    print("\n" + "="*80)
    print(f"CONCAT: {key}  → Parts: {parts}")
    print("="*80)

    # data
    Xtr, Xva, Xte = make_concat(parts, FEATS)

    # train & eval base models
    preds_val, probas_val = train_and_eval_models(Xtr, Y_TR, Xva, Y_VA, verbose_prefix=key)

    base_names = ["KNN","LOGREG","LGBM"]
    # record base metrics & CR (VAL)
    for nm in base_names:
        yhat = preds_val[nm]; P = probas_val[nm]
        mb = metrics_block(Y_VA, yhat, P)
        rows.append({"stage": STAGE_NAME, "concat": key, "model": nm, "voting": "NA", **mb})
        cr = compute_cr_df(Y_VA, yhat, CLASS_NAMES)
        for _,r in cr.iterrows():
            cr_rows.append({
                "stage":STAGE_NAME, "concat":key, "model":nm, "voting":"NA",
                "class": r["class"], "precision": r["precision"], "recall": r["recall"],
                "f1": r["f1-score"], "support": int(r["support"])
            })

    # diversity (VAL)
    print("\nPairwise Diversity (VAL):")
    div_show=[]
    for a,b in itertools.combinations(base_names, 2):
        div = pair_diversity(Y_VA, preds_val[a], preds_val[b])
        rec = {"pair":f"{a}-{b}", **div}
        diversity_rows.append({"stage":STAGE_NAME, "concat":key, **rec})
        div_show.append(rec)
    display(pd.DataFrame(div_show))

    # voting (VAL) — print and record
    y_hard = hard_vote([preds_val[n] for n in base_names])
    P_soft, y_soft = soft_vote([probas_val[n] for n in base_names])

    mb_h = metrics_block(Y_VA, y_hard, None)
    mb_s = metrics_block(Y_VA, y_soft, P_soft)
    print("\nVoting — HARD (VAL):", {k:round(v,4) for k,v in mb_h.items()})
    print("Voting — SOFT (VAL):", {k:round(v,4) for k,v in mb_s.items()})

    # show CM/CR for voting
    for tag, yv, Pv in [("HARD", y_hard, None), ("SOFT", y_soft, P_soft)]:
        cm = confusion_matrix(Y_VA, yv, labels=list(range(C)))
        cm_norm = cm / cm.sum(axis=1, keepdims=True)
        print(f"\n{tag} Voting — Confusion Matrix (VAL):")
        display(pd.DataFrame(cm_norm, index=[f"true_{c}" for c in CLASS_NAMES],
                             columns=[f"pred_{c}" for c in CLASS_NAMES]).round(2))
        cr_df = compute_cr_df(Y_VA, yv, CLASS_NAMES)
        print(f"{tag} Voting — Classification Report (VAL):")
        display(cr_df)

    rows.append({"stage":STAGE_NAME, "concat":key, "model":"+".join(base_names), "voting":"HARD", **mb_h})
    rows.append({"stage":STAGE_NAME, "concat":key, "model":"+".join(base_names), "voting":"SOFT", **mb_s})

    # Oracle-3 (VAL)
    y_or3 = oracle_any_correct(Y_VA, [preds_val[n] for n in base_names], fallback=y_hard)
    mb_o3 = metrics_block(Y_VA, y_or3, None)
    print("\nOracle-3 (any-correct) — VAL:", {k:round(v,4) for k,v in mb_o3.items()})
    rows.append({"stage":STAGE_NAME, "concat":key, "model":"+".join(base_names), "voting":"ORACLE_3", **mb_o3})

    # Oracle-2 (best pair) — FIX: return names, not indices  # <<<
    best_pair_names, y_or2 = best_oracle_pair(Y_VA, [preds_val[n] for n in base_names], base_names)
    mb_o2 = metrics_block(Y_VA, y_or2, None)
    print(f"Oracle-2 (best pair={'+'.join(best_pair_names)}) — VAL:", {k:round(v,4) for k,v in mb_o2.items()})
    rows.append({"stage":STAGE_NAME, "concat":key, "model":"+".join(best_pair_names), "voting":"ORACLE_2", **mb_o2})

# ---------- Save VAL tables & diversity ----------
ensure_dir(OUT_DIR)
df_val = pd.DataFrame(rows)
df_cr  = pd.DataFrame(cr_rows)
df_div = pd.DataFrame(diversity_rows)

df_val_sorted = df_val.sort_values(["macro_f1","auc_macro","mcc"], ascending=False).reset_index(drop=True)
df_val_sorted.to_csv(f"{OUT_DIR}/stage1_val_summary.csv", index=False)
df_cr.to_csv(f"{OUT_DIR}/stage1_val_CR_horizontal.csv", index=False)
df_div.to_csv(f"{OUT_DIR}/stage1_val_diversity.csv", index=False)

print("\n=== VAL summary (top 15 by Macro-F1) ===")
display_cols = ["concat","model","voting","accuracy","macro_f1","micro_f1","auc_macro","mcc"]
display(df_val_sorted[display_cols].head(15))

# ---------- Update JSON registry ----------
reg = {}
if Path(REGISTRY_JSON).exists():
    try:
        reg = json.loads(Path(REGISTRY_JSON).read_text())
    except Exception:
        reg = {}
if "stages" not in reg: reg["stages"] = {}
reg["stages"][STAGE_NAME] = {
    "val_summary_csv": f"{OUT_DIR}/stage1_val_summary.csv",
    "val_cr_csv": f"{OUT_DIR}/stage1_val_CR_horizontal.csv",
    "val_diversity_csv": f"{OUT_DIR}/stage1_val_diversity.csv",
    "sorted_by": "macro_f1(desc), auc_macro(desc), mcc(desc)"
}
Path(REGISTRY_JSON).write_text(json.dumps(reg, indent=2))
print(f"\nRegistry updated: {REGISTRY_JSON}")

# ---------- Select Top-3 by VAL Macro-F1 and evaluate on TEST (CM/CR/ROC/AUC/MCC) ----------
top3 = df_val_sorted[(df_val_sorted["voting"].isin(["NA","HARD","SOFT"]))].head(3).copy()
print("\n=== Top-3 by VAL Macro-F1 (evaluate on TEST) ===")
display(top3[display_cols])

# Reproduce and test helpers
def reproduce_val_models_and_get_test_preds(row):
    key = row["concat"]; parts = CONCAT_SETS[key]
    Xtr, Xva, Xte = make_concat(parts, FEATS)
    Ztr, Zva, Zte = fit_scaler_on_train_transform(Xtr, Xva, Xte)
    voting = row["voting"]; mdl = row["model"]
    if voting == "NA":
        if mdl=="KNN":
            M = KNeighborsClassifier(n_neighbors=15, weights="distance", p=2)
        elif mdl=="LOGREG":
            M = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", n_jobs=-1, multi_class="auto")
        elif mdl=="LGBM":
            M = lgb.LGBMClassifier(n_estimators=600, num_leaves=48, learning_rate=0.05,
                                   subsample=0.9, colsample_bytree=0.7, reg_lambda=1.0,
                                   random_state=RNG_SEED, verbosity=-1)
        else:
            raise ValueError("Unknown model")
        M.fit(Ztr, Y_TR)
        P = M.predict_proba(Zte); yhat = np.argmax(P, axis=1)
        return yhat, P
    else:
        members = {
            "KNN":    KNeighborsClassifier(n_neighbors=15, weights="distance", p=2),
            "LOGREG": LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", n_jobs=-1, multi_class="auto"),
            "LGBM":   lgb.LGBMClassifier(n_estimators=600, num_leaves=48, learning_rate=0.05,
                                         subsample=0.9, colsample_bytree=0.7, reg_lambda=1.0,
                                         random_state=RNG_SEED, verbosity=-1),
        }
        P_list=[]; Yhat_list=[]
        for nm,M in members.items():
            M.fit(Ztr, Y_TR)
            Pm = M.predict_proba(Zte)
            P_list.append(Pm); Yhat_list.append(np.argmax(Pm, axis=1))
        if voting=="HARD":
            yhat = majority_vote(Yhat_list, C)
            P = np.mean(np.stack(P_list,0), axis=0)  
            return yhat, P
        elif voting=="SOFT":
            P = np.mean(np.stack(P_list,0), axis=0)
            yhat = np.argmax(P, axis=1)
            return yhat, P
        else:
            raise ValueError("Unexpected voting for Top-3")

def plot_and_save_cm_cr_roc(prefix_dir, label, y_true, y_pred, proba, class_names):
    ensure_dir(prefix_dir)
    # CM
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_norm = cm / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(6,5))
    im = plt.imshow(cm_norm, interpolation='nearest', aspect='auto')
    plt.title(f"Confusion Matrix — {label}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(class_names)), class_names, rotation=45)
    plt.yticks(range(len(class_names)), class_names)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cm_norm[i,j]
            plt.text(j,i,f"{val:.2f}",ha="center",va="center",fontsize=8,color="white" if val>0.5 else "black")
    plt.tight_layout()
    plt.savefig(f"{prefix_dir}/CM.png", dpi=300, bbox_inches="tight"); plt.close()

    # CR
    cr_df = compute_cr_df(y_true, y_pred, class_names)
    cr_df.to_csv(f"{prefix_dir}/CR.csv", index=False)

    # ROC (macro-ish)
    fpr, tpr, _ = roc_curve(pd.get_dummies(y_true).values.ravel(), proba.ravel())
    rocA = auc(fpr, tpr)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"macro-ish ROC (AUC≈{rocA:.3f})")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"ROC — {label}"); plt.legend()
    plt.tight_layout()
    plt.savefig(f"{prefix_dir}/ROC.png", dpi=300, bbox_inches="tight"); plt.close()

# Evaluate Top-3 on TEST and save
test_summ_rows=[]
for idx, row in top3.iterrows():
    yhat_te, P_te = reproduce_val_models_and_get_test_preds(row)
    acc  = accuracy_score(Y_TE, yhat_te)
    f1Ma = f1_score(Y_TE, yhat_te, average="macro", zero_division=0)
    f1Mi = f1_score(Y_TE, yhat_te, average="micro", zero_division=0)
    mcc  = matthews_corrcoef(Y_TE, yhat_te)
    aucM = macro_auc_ovr(Y_TE, P_te)

    exp_dir = f"{OUT_DIR}/Top3_TEST/{idx:02d}_{row['concat']}_{row['model']}_{row['voting']}"
    plot_and_save_cm_cr_roc(exp_dir, f"{row['concat']} | {row['model']} | {row['voting']}", Y_TE, yhat_te, P_te, CLASS_NAMES)

    print(f"\n[TEST] {row['concat']} | {row['model']} | {row['voting']}")
    print(f"Acc={acc:.4f} | Macro-F1={f1Ma:.4f} | Micro-F1={f1Mi:.4f} | Macro-AUC={aucM:.4f} | MCC={mcc:.4f}")

    test_summ_rows.append({
        "rank_by_val": int(idx),
        "concat": row["concat"], "model": row["model"], "voting": row["voting"],
        "TEST_accuracy": acc, "TEST_macro_f1": f1Ma, "TEST_micro_f1": f1Mi, "TEST_auc_macro": aucM, "TEST_mcc": mcc,
        "cm_png": f"{exp_dir}/CM.png", "cr_csv": f"{exp_dir}/CR.csv", "roc_png": f"{exp_dir}/ROC.png"
    })

df_test_top3 = pd.DataFrame(test_summ_rows).sort_values("rank_by_val", ascending=True)
df_test_top3.to_csv(f"{OUT_DIR}/stage1_top3_TEST_summary.csv", index=False)
print(f"\nAll saved under: {OUT_DIR}")
print(f"Registry JSON: {REGISTRY_JSON}")

## Section 1B — Stage 1B: per-feature classifiers and ensemble strategies

**Research question:** Which pre-defined classifier is strongest for each backbone feature set, and how do hard, soft, average, and training-CV-weighted ensembles compare?

**Selection rule:** choose the per-feature classifier and Top-5 ensemble rows using validation Macro-F1 (with the existing validation tie-breakers). Automatic ensemble weights are estimated from training probabilities using cross-validation, not from test predictions. Test summaries retain validation rank.

In [ ]:
#STAGE 1B
# ============================ Stage-1b: Per-Feature ML → Probas → 1/2/3 Ensemble (Hard/Soft/Avg/Auto-Weighted) ============================
# Algorithms: FAST_SVM (LinearSVC+Calibrated), KNN, GaussianNB, LogisticRegression, RobustMLP, LightGBM, ExtraTrees
# Prints each experiment (metrics, confusion matrix, and classification report) and saves outputs under the configurable OUT_DIR. Updates a local JSON registry.
# ========================================================================================================================================

# --- installs (quiet) ---
import sys, warnings, os, json, random, itertools, math
from pathlib import Path
warnings.filterwarnings("ignore")
!pip -q install lightgbm imbalanced-learn torch torchvision --extra-index-url https://download.pytorch.org/whl/cpu >/dev/null

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, confusion_matrix,
                             classification_report, matthews_corrcoef, roc_curve, auc, log_loss)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import ExtraTreesClassifier
import lightgbm as lgb
from IPython.display import display

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# ---------------- CONFIG ----------------
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
DRIVE_ROOT = str(PROJECT_ROOT / "outputs")
STAGE_NAME = "Stage1b"
OUT_DIR    = f"{DRIVE_ROOT}/{STAGE_NAME}"
REGISTRY_JSON = f"{DRIVE_ROOT}/stage_registry.json"

SPLIT_DIR = str(PROJECT_ROOT / "data" / "splits")
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin_ft768"),
    "EVA": str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet_ft672"),
}
CLASS_NAMES = ['AK','BCC','BKL','DF','MEL','NV','SCC','VASC']
RNG_SEED = 42
set_seed = lambda s=RNG_SEED: (random.seed(s), np.random.seed(s), torch.manual_seed(s))

# Lightweight RobustMLP
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_cls, hidden=(1024,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_cls)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def train_mlp(Xtr, ytr, n_cls, hidden=(1024,), dropout=0.3, lr=3e-4, wd=1e-4, epochs=20, bs=256, device="cpu"):
    ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32), torch.tensor(ytr, dtype=torch.long))
    dl = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=0)
    model = RobustMLP(Xtr.shape[1], n_cls, hidden, dropout).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    crit = nn.CrossEntropyLoss(label_smoothing=0.05)
    model.train()
    for ep in range(epochs):
        for xb, yb in dl:
            xb = xb.to(device); yb = yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb); loss.backward(); opt.step()
    return model

def mlp_proba(model, X, device="cpu", T=1.0):
    model.eval()
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X)))
    dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0)
    logits=[]
    with torch.no_grad():
        for xb,_ in dl:
            xb = xb.to(device)
            logits.append(model(xb).cpu().numpy())
    L = np.concatenate(logits,0)
    P = torch.softmax(torch.tensor(L)/T, dim=1).numpy()
    return P

# ---------------- Utils ----------------
def ensure_dir(p): Path(p).mkdir(parents=True, exist_ok=True)

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True)
    return s.values

def _read_split_csv(path):
    df = pd.read_csv(path)
    id_col=None
    for c in ["image_id","image","Image","filename","fname","img_id","name"]:
        if c in df.columns: id_col=c; break
    assert id_col is not None, f"{path}: id column missing"
    ids = _normalize_ids(df[id_col])
    if "label" in df.columns:
        y = df["label"].astype(int).values
    elif "class_name" in df.columns:
        classes = sorted(df["class_name"].unique().tolist())
        mp={c:i for i,c in enumerate(classes)}
        y = df["class_name"].map(mp).astype(int).values
    else:
        raise ValueError(f"{path}: label/class_name missing")
    return ids, y

def load_splits(split_dir):
    p = Path(split_dir)
    tr_ids, ytr = _read_split_csv(p/"train_split.csv")
    va_ids, yva = _read_split_csv(p/"val_split.csv")
    te_ids, yte = _read_split_csv(p/"test_split.csv")
    return (tr_ids,ytr), (va_ids,yva), (te_ids,yte)

def _load_features(dir_path, split):
    p = Path(dir_path)
    for nm in [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"]:
        f=p/nm
        if f.exists(): return np.load(f)
    raise FileNotFoundError(f"{dir_path}: {split} features missing")

def macro_auc_ovr(y, proba):
    return roc_auc_score(y, proba, multi_class="ovr", average="macro")

def compute_cr_df(y, yhat, class_names):
    rep = classification_report(y, yhat, target_names=class_names, zero_division=0, output_dict=True)
    df = pd.DataFrame(rep).T.reset_index().rename(columns={"index":"class"})
    return df[df["class"].isin(class_names)][["class","precision","recall","f1-score","support"]].reset_index(drop=True)

def fit_scaler_on_train_transform(Xtr, Xva, Xte):
    sc = StandardScaler().fit(Xtr)
    return sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)

def majority_vote(preds_list, n_classes):
    K,N = len(preds_list), len(preds_list[0])
    out = np.zeros(N, dtype=int)
    votes = np.stack(preds_list, axis=0)
    for i in range(N):
        cnt = np.bincount(votes[:,i], minlength=n_classes)
        out[i] = int(np.argmax(cnt))
    return out

def soft_mean(probas_list, weights=None):
    P_list = [P for P in probas_list]
    if weights is None:
        W = np.ones(len(P_list), dtype=float)
    else:
        W = np.array(weights, dtype=float)
    S = np.zeros_like(P_list[0], dtype=float)
    for P, w in zip(P_list, W): S += w * P
    S /= W.sum()
    return S

def weighted_vote_train_cv(P_tr_list, y_tr, n_splits=5, n_trials=256, seed=RNG_SEED):
    # Learn ensemble weights (w≥0, Σw=1) that maximize macro-AUC on TRAIN via 5-fold CV
    rng = np.random.RandomState(seed)
    K = len(P_tr_list)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    def score_w(w):
        w = np.maximum(w, 1e-9); w = w/np.sum(w)
        aucs=[]
        for tr_idx, va_idx in skf.split(P_tr_list[0], y_tr):
            Ps = [P[va_idx] for P in P_tr_list]
            P  = soft_mean(Ps, w)
            aucs.append(macro_auc_ovr(y_tr[va_idx], P))
        return float(np.mean(aucs))
    best_w, best_s = None, -np.inf
    for _ in range(n_trials):
        w = rng.dirichlet(alpha=np.ones(K))
        s = score_w(w)
        if s > best_s:
            best_w, best_s = w, s
    return best_w

# Diversity (pairwise, VAL)
def pair_diversity(y, yhat_a, yhat_b):
    correct_a = (yhat_a == y).astype(int)
    correct_b = (yhat_b == y).astype(int)
    N = len(y)
    N11 = int(((correct_a==1) & (correct_b==1)).sum())
    N00 = int(((correct_a==0) & (correct_b==0)).sum())
    N10 = int(((correct_a==1) & (correct_b==0)).sum())
    N01 = int(((correct_a==0) & (correct_b==1)).sum())
    denom = (N11*N00 + N10*N01)
    Q = ((N11*N00 - N10*N01) / denom) if denom>0 else np.nan
    disagreement = (N10 + N01) / N
    corr = np.corrcoef(correct_a, correct_b)[0,1] if (np.std(correct_a)>0 and np.std(correct_b)>0) else np.nan
    return {"Q": Q, "disagreement": float(disagreement), "corr_correctness": float(corr) if not np.isnan(corr) else np.nan}

# ---------------- Load data & features ----------------
set_seed()
ensure_dir(DRIVE_ROOT); ensure_dir(OUT_DIR)
(TR_IDS, Y_TR), (VA_IDS, Y_VA), (TE_IDS, Y_TE) = load_splits(SPLIT_DIR)
C = len(np.unique(Y_TR)); assert list(np.unique(Y_TR)) == list(range(C)), "Labels must be 0..C-1"

FEATS = {}
for name, d in FEATURE_DIRS.items():
    FEATS[name] = {
        "train": _load_features(d, "train"),
        "val"  : _load_features(d, "val"),
        "test" : _load_features(d, "test"),
    }

# ---------------- Per-feature ML training (VAL probas) ----------------
# Each feature will be trained with all MLs → store VAL proba/preds and TRAIN proba (for weight search)
ML_TAGS = ["KNN","NB","LOGREG","MLP","LGBM","ET","FAST_SVM"]

def train_one_ml(tag, Xtr_raw, ytr, Xva_raw, yva):
    # scaled/unscaled routing
    need_scaled = tag in ["KNN","LOGREG","MLP","NB","FAST_SVM"]  # NB ok with scaled; trees use raw
    if need_scaled:
        Ztr, Zva, _ = fit_scaler_on_train_transform(Xtr_raw, Xva_raw, Xva_raw)
    else:
        Ztr, Zva = Xtr_raw, Xva_raw

    if tag=="FAST_SVM":
        base = LinearSVC(C=1.0, max_iter=2000, tol=1e-3)
        clf  = CalibratedClassifierCV( cv=5, method="sigmoid")
        clf.fit(Ztr, ytr)
        P_val = clf.predict_proba(Zva)
        P_tr  = clf.predict_proba(Ztr)
    elif tag=="KNN":
        clf = KNeighborsClassifier(n_neighbors=15, weights="distance", p=2)
        clf.fit(Ztr, ytr); P_val = clf.predict_proba(Zva); P_tr = clf.predict_proba(Ztr)
    elif tag=="NB":
        clf = GaussianNB()
        clf.fit(Ztr, ytr); P_val = clf.predict_proba(Zva); P_tr = clf.predict_proba(Ztr)
    elif tag=="LOGREG":
        clf = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", n_jobs=-1, multi_class="auto")
        clf.fit(Ztr, ytr); P_val = clf.predict_proba(Zva); P_tr = clf.predict_proba(Ztr)
    elif tag=="MLP":
        device = "cuda" if torch.cuda.is_available() else "cpu"
        mdl = train_mlp(Ztr, ytr, C, hidden=(1024,), dropout=0.3, lr=3e-4, wd=1e-4, epochs=20, bs=256, device=device)
        P_val = mlp_proba(mdl, Zva, device=device, T=1.0)
        P_tr  = mlp_proba(mdl, Ztr, device=device, T=1.0)
        clf = mdl  # for API symmetry
    elif tag=="LGBM":
        clf = lgb.LGBMClassifier(n_estimators=600, num_leaves=48, learning_rate=0.05,
                                 subsample=0.9, colsample_bytree=0.7, reg_lambda=1.0,
                                 random_state=RNG_SEED, verbosity=-1)
        clf.fit(Xtr_raw, ytr)  # raw
        P_val = clf.predict_proba(Xva_raw); P_tr = clf.predict_proba(Xtr_raw)
    elif tag=="ET":
        clf = ExtraTreesClassifier(n_estimators=600, max_depth=None, n_jobs=-1, random_state=RNG_SEED)
        clf.fit(Xtr_raw, ytr)
        P_val = clf.predict_proba(Xva_raw); P_tr = clf.predict_proba(Xtr_raw)
    else:
        raise ValueError("Unknown tag")
    yhat_val = np.argmax(P_val, axis=1)
    return clf, P_val, yhat_val, P_tr

# Train all features × MLs
per_feat_results = {}  # feature -> {tag: {...}}
for feat in ["SWIN","EVA","COAT"]:
    print("\n" + "="*90)
    print(f"[TRAIN] Feature: {feat}")
    print("="*90)
    Xtr = FEATS[feat]["train"]; Xva = FEATS[feat]["val"]
    per_feat_results[feat] = {}
    for tag in ML_TAGS:
        clf, P_val, yhat_val, P_tr = train_one_ml(tag, Xtr, Y_TR, Xva, Y_VA)
        acc  = accuracy_score(Y_VA, yhat_val)
        f1Ma = f1_score(Y_VA, yhat_val, average="macro", zero_division=0)
        f1Mi = f1_score(Y_VA, yhat_val, average="micro", zero_division=0)
        mcc  = matthews_corrcoef(Y_VA, yhat_val)
        aucM = macro_auc_ovr(Y_VA, P_val)
        print(f"VAL — {feat}|{tag}: Acc={acc:.4f} | Macro-F1={f1Ma:.4f} | Micro-F1={f1Mi:.4f} | Macro-AUC={aucM:.4f} | MCC={mcc:.4f}")
        # CM & CR to screen
        cm = confusion_matrix(Y_VA, yhat_val, labels=list(range(C)))
        cmn = cm / cm.sum(axis=1, keepdims=True)
        print("CM (VAL, normalized):")
        display(pd.DataFrame(cmn, index=[f"true_{c}" for c in CLASS_NAMES],
                             columns=[f"pred_{c}" for c in CLASS_NAMES]).round(2))
        print("CR (VAL):")
        display(compute_cr_df(Y_VA, yhat_val, CLASS_NAMES))
        per_feat_results[feat][tag] = {
            "clf": clf, "P_val": P_val, "yhat_val": yhat_val, "P_tr": P_tr,
            "metrics": {"accuracy":acc,"macro_f1":f1Ma,"micro_f1":f1Mi,"mcc":mcc,"auc_macro":aucM}
        }

# ---------------- Build ensembles from 1/2/3 features ----------------
def best_ml_for_feature_on_val(feat):
    # choose the single best ML per feature by VAL macro-F1
    best_tag = None; best_score=-1; best_entry=None
    for tag, d in per_feat_results[feat].items():
        sc = d["metrics"]["macro_f1"]
        if sc > best_score:
            best_score, best_tag, best_entry = sc, tag, d
    return best_tag, best_entry

# Gather candidate ensembles:
# For each size k in {1,2,3}, build a member set by picking the best ML per chosen feature.
FEATURES = ["SWIN","EVA","COAT"]
ENSEMBLES = []  # list of dicts with keys: name, members [{feat, tag, P_val, yhat_val, P_tr}]
for k in [1,2,3]:
    for combo in itertools.combinations(FEATURES, k):
        members=[]
        for feat in combo:
            tag, entry = best_ml_for_feature_on_val(feat)
            members.append({"feat":feat, "tag":tag, **entry})
        name = f"{'+'.join(combo)} | " + "+".join([m["tag"] for m in members])
        ENSEMBLES.append({"name":name, "features":combo, "members":members})

# Voting & reporting
def metrics_block(y, yhat, proba):
    return {
        "accuracy": accuracy_score(y, yhat),
        "macro_f1": f1_score(y, yhat, average="macro", zero_division=0),
        "micro_f1": f1_score(y, yhat, average="micro", zero_division=0),
        "mcc": matthews_corrcoef(y, yhat),
        "auc_macro": macro_auc_ovr(y, proba) if proba is not None else np.nan
    }

VAL_ROWS=[]
DIV_ROWS=[]

for ens in ENSEMBLES:
    print("\n" + "-"*90)
    print(f"[ENSEMBLE VAL] {ens['name']}")
    print("-"*90)
    P_list = [m["P_val"] for m in ens["members"]]
    Yhat_list = [m["yhat_val"] for m in ens["members"]]

    # Diversity among members
    pairs=[]
    for i,j in itertools.combinations(range(len(Yhat_list)),2):
        d = pair_diversity(Y_VA, Yhat_list[i], Yhat_list[j])
        pair_name = f"{ens['members'][i]['feat']}-{ens['members'][i]['tag']}__{ens['members'][j]['feat']}-{ens['members'][j]['tag']}"
        pairs.append({"ensemble":ens["name"], "pair":pair_name, **d})
    if pairs:
        display(pd.DataFrame(pairs))
        DIV_ROWS.extend(pairs)

    # HARD
    y_hard = majority_vote(Yhat_list, C)
    mb_h = metrics_block(Y_VA, y_hard, None)
    print("HARD:", {k:round(v,4) for k,v in mb_h.items()})
    # show CM/CR
    cm = confusion_matrix(Y_VA, y_hard); cmn = cm/ cm.sum(axis=1, keepdims=True)
    display(pd.DataFrame(cmn, index=[f"true_{c}" for c in CLASS_NAMES],
                         columns=[f"pred_{c}" for c in CLASS_NAMES]).round(2))
    display(compute_cr_df(Y_VA, y_hard, CLASS_NAMES))
    VAL_ROWS.append({"stage":STAGE_NAME,"ensemble":ens["name"],"voting":"HARD",**mb_h})

    # SOFT (equal)
    P_soft = soft_mean(P_list, None); y_soft = np.argmax(P_soft, axis=1)
    mb_s = metrics_block(Y_VA, y_soft, P_soft)
    print("SOFT:", {k:round(v,4) for k,v in mb_s.items()})
    cm = confusion_matrix(Y_VA, y_soft); cmn = cm/ cm.sum(axis=1, keepdims=True)
    display(pd.DataFrame(cmn, index=[f"true_{c}" for c in CLASS_NAMES],
                         columns=[f"pred_{c}" for c in CLASS_NAMES]).round(2))
    display(compute_cr_df(Y_VA, y_soft, CLASS_NAMES))
    VAL_ROWS.append({"stage":STAGE_NAME,"ensemble":ens["name"],"voting":"SOFT",**mb_s})

    # AVG (alias of SOFT here)
    VAL_ROWS.append({"stage":STAGE_NAME,"ensemble":ens["name"],"voting":"AVG",**mb_s})

    # AUTO-WEIGHTED (learn w on TRAIN via 5-fold)
    P_tr_list = [m["P_tr"] for m in ens["members"]]
    w = weighted_vote_train_cv(P_tr_list, Y_TR, n_splits=5, n_trials=256, seed=RNG_SEED)
    P_w = soft_mean(P_list, w); y_w = np.argmax(P_w, axis=1)
    mb_w = metrics_block(Y_VA, y_w, P_w)
    print(f"AUTO-WEIGHTED (w={np.round(w,3).tolist()}):", {k:round(v,4) for k,v in mb_w.items()})
    cm = confusion_matrix(Y_VA, y_w); cmn = cm/ cm.sum(axis=1, keepdims=True)
    display(pd.DataFrame(cmn, index=[f"true_{c}" for c in CLASS_NAMES],
                         columns=[f"pred_{c}" for c in CLASS_NAMES]).round(2))
    display(compute_cr_df(Y_VA, y_w, CLASS_NAMES))
    VAL_ROWS.append({"stage":STAGE_NAME,"ensemble":ens["name"],"voting":"WEIGHTED",**mb_w, "weights": "|".join([f"{m['feat']}-{m['tag']}={w[i]:.3f}" for i,m in enumerate(ens['members'])])})

# ---------------- Save VAL summary, diversity & registry ----------------
ensure_dir(OUT_DIR)
df_val = pd.DataFrame(VAL_ROWS).sort_values(["macro_f1","auc_macro","mcc"], ascending=False).reset_index(drop=True)
df_div = pd.DataFrame(DIV_ROWS)
df_val.to_csv(f"{OUT_DIR}/stage1b_val_summary.csv", index=False)
df_div.to_csv(f"{OUT_DIR}/stage1b_val_diversity.csv", index=False)

print("\n=== TOP-20 by VAL Macro-F1 ===")
display(df_val[["ensemble","voting","accuracy","macro_f1","micro_f1","auc_macro","mcc"]].head(20))

# update registry
reg = {}
if Path(REGISTRY_JSON).exists():
    try: reg = json.loads(Path(REGISTRY_JSON).read_text())
    except: reg = {}
if "stages" not in reg: reg["stages"]={}
reg["stages"][STAGE_NAME] = {
    "val_summary_csv": f"{OUT_DIR}/stage1b_val_summary.csv",
    "val_diversity_csv": f"{OUT_DIR}/stage1b_val_diversity.csv",
    "note": "Per-feature best-ML members; 1/2/3 ensembles with Hard/Soft/Avg/Auto-Weighted"
}
Path(REGISTRY_JSON).write_text(json.dumps(reg, indent=2))
print(f"Registry updated: {REGISTRY_JSON}")

# ---------------- TEST evaluation for Top-5 (by VAL Macro-F1) ----------------
TOP5 = df_val.head(5).copy()
print("\n=== Validation-selected Top-5; TEST metrics are report-only ===")
display(TOP5[["ensemble","voting","macro_f1","auc_macro","mcc"]])

def get_test_proba_for_member(member):
    feat, tag = member["feat"], member["tag"]
    Xtr = FEATS[feat]["train"]; Xva = FEATS[feat]["val"]; Xte = FEATS[feat]["test"]
    
    need_scaled = tag in ["FAST_SVM","KNN","LOGREG","MLP","NB"]
    if need_scaled:
        Ztr, Zva, Zte = fit_scaler_on_train_transform(Xtr, Xva, Xte)
    else:
        Ztr, Zva, Zte = Xtr, Xva, Xte

    if tag=="FAST_SVM":
        base = LinearSVC(C=1.0, max_iter=2000, tol=1e-3)
        clf  = CalibratedClassifierCV( cv=5, method="sigmoid")
        clf.fit(Ztr, Y_TR); P_te = clf.predict_proba(Zte)
    elif tag=="KNN":
        clf = KNeighborsClassifier(n_neighbors=15, weights="distance", p=2)
        clf.fit(Ztr, Y_TR); P_te = clf.predict_proba(Zte)
    elif tag=="NB":
        clf = GaussianNB(); clf.fit(Ztr, Y_TR); P_te = clf.predict_proba(Zte)
    elif tag=="LOGREG":
        clf = LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", n_jobs=-1, multi_class="auto")
        clf.fit(Ztr, Y_TR); P_te = clf.predict_proba(Zte)
    elif tag=="MLP":
        device = "cuda" if torch.cuda.is_available() else "cpu"
        mdl = train_mlp(Ztr, Y_TR, C, hidden=(1024,), dropout=0.3, lr=3e-4, wd=1e-4, epochs=20, bs=256, device=device)
        P_te = mlp_proba(mdl, Zte, device=device, T=1.0)
    elif tag=="LGBM":
        clf = lgb.LGBMClassifier(n_estimators=600, num_leaves=48, learning_rate=0.05,
                                 subsample=0.9, colsample_bytree=0.7, reg_lambda=1.0,
                                 random_state=RNG_SEED, verbosity=-1)
        clf.fit(Xtr, Y_TR); P_te = clf.predict_proba(Xte)
    elif tag=="ET":
        clf = ExtraTreesClassifier(n_estimators=600, max_depth=None, n_jobs=-1, random_state=RNG_SEED)
        clf.fit(Xtr, Y_TR); P_te = clf.predict_proba(Xte)
    else:
        raise ValueError("Unknown tag")
    return P_te

def save_cm_cr_roc(prefix_dir, label, y_true, y_pred, proba, class_names):
    ensure_dir(prefix_dir)
    # CM
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cmn = cm / cm.sum(axis=1, keepdims=True)
    plt.figure(figsize=(6,5))
    im = plt.imshow(cmn, interpolation='nearest', aspect='auto')
    plt.title(f"Confusion Matrix — {label}")
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(len(class_names)), class_names, rotation=45)
    plt.yticks(range(len(class_names)), class_names)
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            val = cmn[i,j]
            plt.text(j,i,f"{val:.2f}",ha="center",va="center",fontsize=8,color="white" if val>0.5 else "black")
    plt.tight_layout(); plt.savefig(f"{prefix_dir}/CM.png", dpi=300, bbox_inches="tight"); plt.close()

    # CR
    cr_df = compute_cr_df(y_true, y_pred, class_names)
    cr_df.to_csv(f"{prefix_dir}/CR.csv", index=False)

    # ROC (macro-ish)
    fpr, tpr, _ = roc_curve(pd.get_dummies(y_true).values.ravel(), proba.ravel())
    rocA = auc(fpr, tpr)
    plt.figure(figsize=(6,5))
    plt.plot(fpr, tpr, label=f"macro-ish ROC (AUC≈{rocA:.3f})")
    plt.plot([0,1],[0,1],'--')
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title(f"ROC — {label}"); plt.legend()
    plt.tight_layout(); plt.savefig(f"{prefix_dir}/ROC.png", dpi=300, bbox_inches="tight"); plt.close()

TEST_ROWS=[]
for i, r in TOP5.iterrows():
    name   = r["ensemble"]; voting = r["voting"]
    # reproduce TEST probs for ensemble members
    ens = next(e for e in ENSEMBLES if e["name"]==name)
    Pte_list = [ get_test_proba_for_member(m) for m in ens["members"] ]
    if voting=="HARD":
        y_list = [np.argmax(P,1) for P in Pte_list]
        y_pred = majority_vote(y_list, C)
        P_ens  = np.mean(np.stack(Pte_list,0), axis=0)
    elif voting in ["SOFT","AVG"]:
        P_ens = soft_mean(Pte_list, None); y_pred = np.argmax(P_ens,1)
    elif voting=="WEIGHTED":
        # Learn weights again on TRAIN (members’ P_tr) — reuse from previous step:
        P_tr_list = [ m["P_tr"] for m in ens["members"] ]
        w = weighted_vote_train_cv(P_tr_list, Y_TR, n_splits=5, n_trials=256, seed=RNG_SEED)
        P_ens = soft_mean(Pte_list, w); y_pred = np.argmax(P_ens,1)
    else:
        raise ValueError("Unknown voting")

    acc  = accuracy_score(Y_TE, y_pred)
    f1Ma = f1_score(Y_TE, y_pred, average="macro", zero_division=0)
    f1Mi = f1_score(Y_TE, y_pred, average="micro", zero_division=0)
    mcc  = matthews_corrcoef(Y_TE, y_pred)
    aucM = macro_auc_ovr(Y_TE, P_ens)

    test_dir = f"{OUT_DIR}/Top5_TEST/{i:02d}"
    save_cm_cr_roc(test_dir, f"{name} | {voting}", Y_TE, y_pred, P_ens, CLASS_NAMES)

    print(f"\n[TEST] {name} | {voting} → Acc={acc:.4f} | Macro-F1={f1Ma:.4f} | Micro-F1={f1Mi:.4f} | Macro-AUC={aucM:.4f} | MCC={mcc:.4f}")

    TEST_ROWS.append({
        "validation_rank": int(i) + 1,
        "ensemble": name, "voting": voting,
        "TEST_accuracy":acc, "TEST_macro_f1":f1Ma, "TEST_micro_f1":f1Mi, "TEST_auc_macro":aucM, "TEST_mcc":mcc,
        "paths": {"cm": f"{test_dir}/CM.png", "cr": f"{test_dir}/CR.csv", "roc": f"{test_dir}/ROC.png"}
    })

df_test = pd.DataFrame(TEST_ROWS).sort_values("validation_rank", ascending=True).reset_index(drop=True)
df_test.to_csv(f"{OUT_DIR}/stage1b_top5_TEST_summary.csv", index=False)

print(f"\nAll saved under: {OUT_DIR}")
print(f"Registry JSON: {REGISTRY_JSON}")

## Section 2 — Representation and aggregation study

**Research question:** Which pre-defined dimensionality-reduction, aggregation, SMOTE, and voting settings achieve the best validation performance?

**Selection rule:** rank every candidate on validation Macro-F1, Macro-AUC, and MCC; take the Top-5 from that validation table. The final test loop evaluates those pre-selected candidates and retains their validation rank; it must not be used to select a new variant.

In [ ]:
#STAGE 2
# =========================== STAGE-2: Representation & Aggregation Study ===========================
# Q1-ready: leak-free VAL selection, Train∪Val refit for TEST, full logging & diversity metrics
# -----------------------------------------------------------------------------------------------
# Paths & splits match your earlier setup.

!pip -q install imbalanced-learn lightgbm > /dev/null

import os, json, math, random, time, itertools, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, matthews_corrcoef,
                             confusion_matrix, classification_report, log_loss)
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from imblearn.over_sampling import SMOTE

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import matplotlib.pyplot as plt

# ================== CONFIG ==================
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
SPLIT_DIR = str(PROJECT_ROOT / "data" / "splits")  # train_split.csv / val_split.csv / test_split.csv
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin_ft768"),
    "EVA": str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet_ft672"),
}
META_CSV_PATH = str(PROJECT_ROOT / "data" / "metadata" / "ISIC_2019_Training_Metadata.csv")

SAVE_ROOT = PROJECT_ROOT / "outputs" / "Stage2"
SAVE_ROOT.mkdir(parents=True, exist_ok=True)
REGISTRY_JSON = SAVE_ROOT/"stage2_registry.json"

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)

CLASS_NAMES = ['AK','BCC','BKL','DF','MEL','NV','SCC','VASC']

# Ensembles under study (fixed best base-ML per feature from Stage-1b)
ENSEMBLES = [
    ("SWIN+EVA",       [("SWIN","MLP"), ("EVA","FAST_SVM")]),
    ("SWIN+EVA+COAT",  [("SWIN","MLP"), ("EVA","FAST_SVM"), ("COAT","MLP")]),
]

# Search space
SEARCH_SPACE = {
    "pca": [None, 0.95, 0.99],
    "dea_bottlenecks": [256, 512, 1024, 1536, 2048],  # can shrink for faster runs
    "smote": [False, True],
    "aggregation": ["hard", "soft", "avg", "weighted"],   # weighted = grid on soft
    "weighted_step": 0.05
}

# Optional TTA-lite (feature-space jitter)
TTA_ENABLE = False
TTA_N = 5
TTA_SIGMA = 0.01

# MLP presets from your Stage-1b (slightly generalized)
MLP_HP = {
    "SWIN": dict(hidden=[2048], dropout=0.16, lr=3.4e-4, weight_decay=1.49e-4, batch_size=128, epochs=50, T=0.9),
    "EVA" : dict(hidden=[2048], dropout=0.32, lr=9.77e-4, weight_decay=8.21e-5, batch_size=128, epochs=50, T=0.9),
    "COAT": dict(hidden=[2048], dropout=0.25, lr=5.0e-4, weight_decay=1.0e-4, batch_size=128, epochs=50, T=0.9),
}
SVM_HP = dict(C=2.0, max_iter=5000)  # fast & calibrated

# ================== IO helpers ==================
def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True)
    return s.values

def _read_split_csv(path):
    df = pd.read_csv(path)
    id_col=None
    for c in ["image_id","image","Image","filename","fname","img_id","name"]:
        if c in df.columns: id_col=c; break
    assert id_col is not None, f"{path}: id column not found"
    ids = _normalize_ids(df[id_col])
    if "label" in df.columns: y = df["label"].astype(int).values
    elif "class_name" in df.columns:
        classes = sorted(df["class_name"].unique().tolist()); mp={c:i for i,c in enumerate(classes)}
        y = df["class_name"].map(mp).astype(int).values
    else:
        raise ValueError(f"{path}: label/class_name not found")
    return ids, y

def load_global_splits(split_dir):
    p = Path(split_dir)
    tr_ids, ytr = _read_split_csv(p/"train_split.csv")
    va_ids, yva = _read_split_csv(p/"val_split.csv")
    te_ids, yte = _read_split_csv(p/"test_split.csv")
    return (tr_ids,ytr), (va_ids,yva), (te_ids,yte)

def _load_features(dir_path, split):
    p = Path(dir_path)
    for nm in [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"]:
        f=p/nm
        if f.exists(): return np.load(f)
    raise FileNotFoundError(f"{dir_path}: {split} features not found")

# ================== Metrics ==================
def ece_score(y_true, proba, n_bins=15):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); accs = (pred == y_true).astype(float)
    bins = np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        if m.sum(): ece += m.mean() * abs(accs[m].mean() - conf[m].mean())
    return float(ece)

def brier_score(y_true, proba):
    C = proba.shape[1]; Y = np.eye(C)[y_true]
    return float(np.mean(np.sum((proba - Y)**2, axis=1)))

def macro_auc(y_true, proba):
    try:
        return float(roc_auc_score(y_true, proba, multi_class="ovr", average="macro"))
    except Exception:
        return float("nan")

def eval_block(y, proba):
    yhat = proba.argmax(1)
    return {
        "accuracy": float(accuracy_score(y, yhat)),
        "macro_f1": float(f1_score(y, yhat, average="macro", zero_division=0)),
        "micro_f1": float(f1_score(y, yhat, average="micro", zero_division=0)),
        "auc_macro": macro_auc(y, proba),
        "mcc": float(matthews_corrcoef(y, yhat)),
        "ece": ece_score(y, proba),
        "brier": brier_score(y, proba),
    }

def save_cm_cr(prefix, y, proba, class_names, outdir, print_console=True):
    yhat = proba.argmax(1)
    cm = confusion_matrix(y, yhat, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in class_names],
                         columns=[f"pred_{c}" for c in class_names])
    cr = classification_report(y, yhat, target_names=class_names, zero_division=0, output_dict=True)
    cr_df = pd.DataFrame(cr)
    cm_df.to_csv(outdir/f"{prefix}_confusion_matrix.csv")
    cr_df.to_csv(outdir/f"{prefix}_classification_report.csv")
    if print_console:
        print("\nCM (normalized):")
        cm_norm = (cm.T / cm.sum(axis=1)).T
        print(pd.DataFrame(np.round(cm_norm,2), index=[f"true_{c}" for c in class_names],
                           columns=[f"pred_{c}" for c in class_names]).to_string())
        print("\nCR:")
        print(cr_df.to_string())

# ================== Diversity (pairwise on predictions) ==================
def diversity_pairwise(y, P1, P2):
    p1 = P1.argmax(1); p2 = P2.argmax(1)
    c1 = (p1==y).astype(int); c2 = (p2==y).astype(int)
    N11 = int(((p1==y) & (p2==y)).sum()); N00 = int(((p1!=y) & (p2!=y)).sum())
    N10 = int(((p1==y) & (p2!=y)).sum()); N01 = int(((p1!=y) & (p2==y)).sum())
    denom = (N11+N10)*(N01+N00)
    denom2= (N11+N01)*(N10+N00)
    Q = (N11*N00 - N01*N10) / max(1e-9, (N11*N00 + N01*N10))
    disagreement = (N01 + N10) / max(1, (N11+N10+N01+N00))
    corr = np.corrcoef(c1, c2)[0,1] if np.std(c1)>0 and np.std(c2)>0 else 0.0
    return float(Q), float(disagreement), float(corr)

# ================== Transformers: PCA / DEA ==================
class AutoEncoder(nn.Module):
    def __init__(self, in_dim, bottleneck):
        super().__init__()
        h1 = max(1024, min(4096, in_dim*2))
        h2 = max(512, bottleneck*2)
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, h1), nn.BatchNorm1d(h1), nn.ReLU(),
            nn.Linear(h1, h2),    nn.BatchNorm1d(h2), nn.ReLU(),
            nn.Linear(h2, bottleneck),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck, h2), nn.BatchNorm1d(h2), nn.ReLU(),
            nn.Linear(h2, h1),         nn.BatchNorm1d(h1), nn.ReLU(),
            nn.Linear(h1, in_dim)
        )
    def forward(self, x):
        z = self.encoder(x); xh = self.decoder(z); return xh, z

def fit_dea(Xtr, bottleneck, max_epochs=200, bs=256, lr=1e-3, wd=1e-4, patience=10, delta=1e-4, device="cuda" if torch.cuda.is_available() else "cpu"):
    model = AutoEncoder(Xtr.shape[1], bottleneck).to(device)
    ds = TensorDataset(torch.tensor(Xtr, dtype=torch.float32))
    dl = DataLoader(ds, batch_size=bs, shuffle=True, num_workers=0, drop_last=False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    best = 1e9; bad=0
    for ep in range(1, max_epochs+1):
        model.train(); loss_epoch=0.0
        for (xb,) in dl:
            xb = xb.to(device); opt.zero_grad()
            xh, _ = model(xb); loss = torch.nn.functional.mse_loss(xh, xb)
            loss.backward(); opt.step(); loss_epoch += loss.item()*len(xb)
        loss_epoch /= len(ds)
        if loss_epoch < best - delta: best = loss_epoch; bad=0
        else: bad += 1
        if ep % 10 == 0 or ep==1:
            print(f"[DEA] ep{ep:03d} | recon={loss_epoch:.6f}")
        if bad >= patience: break
    model.eval()
    @torch.no_grad()
    def transform(X):
        dl = DataLoader(TensorDataset(torch.tensor(X, dtype=torch.float32)), batch_size=512, shuffle=False)
        Z=[]
        for (xb,) in dl:
            xb=xb.to(device); _, z = model(xb); Z.append(z.cpu().numpy())
        return np.concatenate(Z, axis=0)
    return transform

def get_transformer(kind, Xtr, pca_var=None, dea_bottleneck=None):
    scaler = StandardScaler().fit(Xtr)
    if kind == "pca":
        if pca_var is None:
            def tr(X): return scaler.transform(X)
            return tr, f"STD"
        pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(scaler.transform(Xtr))
        def tr(X): return pca.transform(scaler.transform(X))
        return tr, f"STD+PCA{pca_var}"
    elif kind == "dea":
        Ztr = scaler.transform(Xtr)
        dea = fit_dea(Ztr, dea_bottleneck)
        def tr(X): return dea(scaler.transform(X))
        return tr, f"STD+DEA{dea_bottleneck}"
    else:
        raise ValueError("unknown transformer")

# ================== Models (MLP / FAST_SVM[Calibrated]) ==================
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(1024,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def train_mlp(X, y, n_classes, hp, device="cuda" if torch.cuda.is_available() else "cpu"):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    dl = DataLoader(ds, batch_size=hp["batch_size"], shuffle=True, num_workers=0, drop_last=False)
    model = RobustMLP(X.shape[1], n_classes, tuple(hp["hidden"]), hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
    model.train()
    for ep in range(1, hp["epochs"]+1):
        for xb,yb in dl:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(); loss=crit(model(xb), yb); loss.backward(); opt.step()
        if ep==1 or ep%5==0:
            with torch.no_grad():
                lg = model(torch.tensor(X, dtype=torch.float32, device=device)).cpu().numpy()
                proba = torch.softmax(torch.tensor(lg), dim=1).numpy()
                acc = accuracy_score(y, proba.argmax(1))
                f1m = f1_score(y, proba.argmax(1), average="macro", zero_division=0)
            print(f"[MLP] ep {ep:03d} | Acc {acc:.4f} | MacroF1 {f1m:.4f}")
    return model

def predict_proba_mlp(model, X, T=1.0, device="cuda" if torch.cuda.is_available() else "cpu"):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X)))
    dl = DataLoader(ds, batch_size=512, shuffle=False, num_workers=0, drop_last=False)
    out=[]
    model.eval()
    with torch.no_grad():
        for (xb,_) in dl:
            xb=xb.to(device); lg = model(xb); out.append(lg.cpu().numpy())
    logits = np.concatenate(out,0)
    proba = torch.softmax(torch.tensor(logits)/T, dim=1).numpy()
    return proba

def build_classifier(name, in_dim, n_classes, feat_key):
    if name == "MLP":
        hp = MLP_HP[feat_key]
        return ("MLP", hp)
    elif name == "FAST_SVM":
        base = LinearSVC(C=SVM_HP["C"], max_iter=SVM_HP["max_iter"], tol=1e-3, dual=True)
        clf = CalibratedClassifierCV(base, method="sigmoid", cv=3)
        return ("FAST_SVM", clf)
    else:
        raise ValueError("unknown model")

def fit_predict(name, obj, Xtr, ytr, Xva, n_classes, feat_key):
    if name == "MLP":
        hp = obj
        mdl = train_mlp(Xtr, ytr, n_classes, hp)
        Pva = predict_proba_mlp(mdl, Xva, T=hp["T"])
        return mdl, Pva
    elif name == "FAST_SVM":
        clf = obj
        clf.fit(Xtr, ytr)
        Pva = clf.predict_proba(Xva)
        return clf, Pva

def refit_predict(name, obj, Xfit, yfit, Xte, n_classes, feat_key):
    if name == "MLP":
        hp = obj
        mdl = train_mlp(Xfit, yfit, n_classes, hp)
        Pte = predict_proba_mlp(mdl, Xte, T=hp["T"])
        return Pte
    elif name == "FAST_SVM":
        clf = obj.__class__(**obj.get_params())
        clf.fit(Xfit, yfit)
        return clf.predict_proba(Xte)

# ================== Aggregation ==================
def agg_hard(probas):
    preds = [P.argmax(1) for P in probas]
    votes = np.stack(preds, axis=0)
    N = votes.shape[1]; C = probas[0].shape[1]
    out = np.zeros((N,C), dtype=float)
    for i in range(N):
        cnt = np.bincount(votes[:,i], minlength=C)
        out[i, cnt.argmax()] = 1.0
    return out

def agg_soft(probas, weights=None):
    if weights is None: weights = np.ones(len(probas))
    S = np.zeros_like(probas[0], dtype=float)
    wsum = 0.0
    for P,w in zip(probas, weights):
        S += w * P; wsum += w
    return S / max(wsum, 1e-12)

def agg_avg(probas):
    return agg_soft(probas, None)

def weight_grid(k, step=0.05):
    # yields tuples of length k summing to 1 with increments of step
    m = int(round(1/step))
    if k == 2:
        for i in range(m+1):
            w1 = i*step; yield (w1, 1.0-w1)
    else:
        # simple integer partition with granularity m
        def rec(prefix, remain, slots):
            if slots==1:
                yield tuple(prefix+[remain])
            else:
                for x in range(remain+1):
                    yield from rec(prefix+[x], remain-x, slots-1)
        for parts in rec([], m, k):
            yield tuple(p/m for p in parts)

# ================== Loader (features & splits) ==================
(tr_ids, y_tr), (va_ids, y_va), (te_ids, y_te) = load_global_splits(SPLIT_DIR)
Y_TR, Y_VA, Y_TE = y_tr, y_va, y_te
NC = len(np.unique(Y_TR))

DATA = {}
for feat, d in FEATURE_DIRS.items():
    DATA[feat] = {
        "Xtr": _load_features(d, "train"),
        "Xva": _load_features(d, "val"),
        "Xte": _load_features(d, "test"),
        "ytr": Y_TR, "yva": Y_VA, "yte": Y_TE
    }

print("Loaded features:", {k: {s: v.shape for s,v in DATA[k].items() if s.startswith("X")} for k in DATA})

# ================== Core runner ==================
def maybe_tta(X, proba_func):
    if not TTA_ENABLE: return proba_func(X)
    out = []
    for _ in range(TTA_N):
        noise = np.random.normal(0.0, TTA_SIGMA, size=X.shape).astype(np.float32)
        out.append(proba_func(X + noise))
    return np.mean(out, axis=0)

def run_one_setting(ens_name, members, transf_kind, transf_tag, transform, use_smote, agg, weights=None, save_dir=None):
    # 1) Transform (fit on TRAIN only!)
    # 2) Optional SMOTE on TRAIN
    # 3) Train member models on TRAIN; get VAL probas
    # 4) Aggregate; eval; diversity logs

    os.makedirs(save_dir, exist_ok=True)

    member_val_probas=[]; member_names=[]
    for feat, model_name in members:
        Xtr, Xva = DATA[feat]["Xtr"], DATA[feat]["Xva"]
        Ztr, Zva = transform(Xtr), transform(Xva)

        # SMOTE only on TRAIN
        ytr = Y_TR
        if use_smote:
            try:
                sm = SMOTE(k_neighbors=5, random_state=RANDOM_STATE)
                Ztr, ytr = sm.fit_resample(Ztr, ytr)
            except Exception as e:
                print(f"[WARN] SMOTE failed: {e}")

        # Build model & fit
        mtype, obj = build_classifier(model_name, Ztr.shape[1], NC, feat)
        mdl, Pva = fit_predict(mtype, obj, Ztr, ytr, Zva, NC, feat)
        member_val_probas.append(Pva); member_names.append(f"{feat}|{model_name}")

        # Save per-member VAL report
        sub = Path(save_dir)/f"{feat}_{model_name}"
        sub.mkdir(parents=True, exist_ok=True)
        met = eval_block(Y_VA, Pva)
        pd.DataFrame([met]).to_csv(sub/"VAL_metrics.csv", index=False)
        save_cm_cr("VAL", Y_VA, Pva, CLASS_NAMES, sub, print_console=False)

    # Diversity (pairwise)
    div_rows=[]
    for (i,(n1,P1)), (j,(n2,P2)) in itertools.combinations(enumerate(zip(member_names, member_val_probas)), 2):
        Q,dis,corr = diversity_pairwise(Y_VA, P1, P2)
        div_rows.append({"pair": f"{n1}__{n2}", "Q":Q, "disagreement":dis, "corr_correctness":corr})
    div_df = pd.DataFrame(div_rows);
    if len(div_df): div_df.to_csv(Path(save_dir)/"VAL_diversity_pairs.csv", index=False)

    # Aggregations
    results=[]
    if agg in ("hard","soft","avg"):
        if agg=="hard":
            Pv = agg_hard(member_val_probas)
        elif agg=="avg":
            Pv = agg_avg(member_val_probas)
        else:
            Pv = agg_soft(member_val_probas)
        met = eval_block(Y_VA, Pv)
        results.append(("VAL", agg.upper(), None, met))
        # Save CM/CR + metrics
        pd.DataFrame([met]).to_csv(Path(save_dir)/f"VAL_{agg}_metrics.csv", index=False)
        save_cm_cr(f"VAL_{agg}", Y_VA, Pv, CLASS_NAMES, Path(save_dir), print_console=True)

    elif agg=="weighted":
        best = (-1, None, None, None)  # macro_f1, weights, met, Pv
        K = len(member_val_probas)
        for w in weight_grid(K, step=SEARCH_SPACE["weighted_step"]):
            Pv = agg_soft(member_val_probas, w)
            met = eval_block(Y_VA, Pv)
            results.append(("VAL","WEIGHTED", w, met))
            if met["macro_f1"] > best[0]:
                best = (met["macro_f1"], w, met, Pv)
        # Save best-weighted
        if best[1] is not None:
            w = [float(x) for x in best[1]]
            pd.DataFrame([best[2]]).to_csv(Path(save_dir)/"VAL_WEIGHTED_best_metrics.csv", index=False)
            json.dump({"weights": w}, open(Path(save_dir)/"VAL_WEIGHTED_best_weights.json","w"), indent=2)
            save_cm_cr("VAL_WEIGHTED_best", Y_VA, best[3], CLASS_NAMES, Path(save_dir), print_console=True)

    # Return summary rows
    rows=[]
    for tag, what, w, met in results:
        rows.append({
            "ensemble": ens_name,
            "members": "+".join(member_names),
            "transform": transf_tag,
            "smote": use_smote,
            "voting": what,
            "weights": None if w is None else list(map(float,w)),
            **met
        })
    return rows, member_val_probas, member_names

def refit_and_test(ens_name, members, transform, use_smote, voting, weights, save_dir):
    # Refit pipeline on TRAIN∪VAL, evaluate TEST
    member_test_probas=[]; member_names=[]
    for feat, model_name in members:
        Xtr, Xva, Xte = DATA[feat]["Xtr"], DATA[feat]["Xva"], DATA[feat]["Xte"]
        Ztr = transform(Xtr); Zva = transform(Xva); Zte = transform(Xte)
        Xfit = np.vstack([Ztr,Zva]); yfit = np.concatenate([Y_TR, Y_VA])

        if use_smote:
            try:
                sm = SMOTE(k_neighbors=5, random_state=RANDOM_STATE)
                Xfit, yfit = sm.fit_resample(Xfit, yfit)
            except Exception as e:
                print(f"[WARN] SMOTE failed on refit: {e}")

        mtype, obj = build_classifier(model_name, Xfit.shape[1], NC, feat)
        Pte = refit_predict(mtype, obj, Xfit, yfit, Zte, NC, feat)
        member_test_probas.append(Pte); member_names.append(f"{feat}|{model_name}")

    # Aggregate
    if voting=="HARD":
        P = agg_hard(member_test_probas)
    elif voting in ("SOFT","AVG"):
        P = agg_soft(member_test_probas) if voting=="SOFT" else agg_avg(member_test_probas)
    elif voting=="WEIGHTED":
        P = agg_soft(member_test_probas, weights)
    else:
        raise ValueError("unknown voting")

    met = eval_block(Y_TE, P)
    pd.DataFrame([met]).to_csv(Path(save_dir)/"TEST_metrics.csv", index=False)
    save_cm_cr("TEST", Y_TE, P, CLASS_NAMES, Path(save_dir), print_console=True)
    print(f"\n[TEST] {ens_name} | {voting}{'' if weights is None else ' w='+str(weights)} → "
          f"Acc={met['accuracy']:.4f} | Macro-F1={met['macro_f1']:.4f} | Macro-AUC={met['auc_macro']:.4f} | MCC={met['mcc']:.4f}")
    return met

# ================== MAIN SWEEP ==================
all_val_rows=[]

for ens_name, members in ENSEMBLES:
    print("\n" + "="*100)
    print(f"[STAGE-2] Ensemble: {ens_name}  | Members: {members}")
    print("="*100)

    # Load a base to know input dim for DEA scaler; we fit per-feature anyway inside get_transformer
    # TRANSFORMS: PCA(None/0.95/0.99) + DEA(bn list)
    transform_specs = [("pca", None, None), ("pca", 0.95, None), ("pca", 0.99, None)]
    for bn in SEARCH_SPACE["dea_bottlenecks"]:
        transform_specs.append(("dea", None, bn))

    for kind, pvar, bn in transform_specs:
        # For leak-free, transformer must be fitted on TRAIN for each feature separately.
        # But we need a single transform function that uses TRAIN of the *first* feature as reference
        # → To keep it fair, we build one transformer PER-FEATURE inside run_one_setting (already done).
        # Here we only pass a "factory" that will be constructed per-feature.

        # We pass a small wrapper that caches per-feature transformer functions:
        per_feat_cache = {}
        def transform_factory(X, feat_key):
            key = (feat_key, kind, pvar, bn)
            if key not in per_feat_cache:
                if kind=="pca":
                    tr, tag = get_transformer("pca", X, pca_var=pvar)
                else:
                    tr, tag = get_transformer("dea", X, dea_bottleneck=bn)
                per_feat_cache[key] = (tr, tag)
            return per_feat_cache[key]

        for sm in SEARCH_SPACE["smote"]:
            for agg in SEARCH_SPACE["aggregation"]:
                tag = f"{'PCA-'+str(pvar) if kind=='pca' else 'DEA-'+str(bn)} | SMOTE={sm} | {agg.upper()}"
                out_dir = SAVE_ROOT/f"{ens_name}/{tag.replace(' ','_').replace('|','_')}"
                out_dir.mkdir(parents=True, exist_ok=True)

                # Build a per-feature transform wrapper to keep API the same
                def transform_per_feature_builder(feat_key):
                    Xtr = DATA[feat_key]["Xtr"]
                    tr, inner_tag = transform_factory(Xtr, feat_key)
                    return tr, inner_tag

                # Prepare a transform that picks per-feature cached version
                def transform_picker(feat_key):
                    tr,_ = transform_per_feature_builder(feat_key)
                    return tr

                # Run setting
                # We need a transform function compatible with run_one_setting: transform(X) signature.
                # So we wrap it to accept only X but capture feature name via closure during call.
                # Simpler: call run_one_setting per-feature by partially applying transform for that feature inside.
                # We'll implement a small adapter:

                class PerFeatureTransform:
                    def __init__(self): self.current_feat=None
                    def set_feat(self, feat): self.current_feat=feat
                    def __call__(self, X):
                        tr = transform_picker(self.current_feat)
                        return tr(X)

                PFT = PerFeatureTransform()

                # Monkey-run: inside run_one_setting we need to set current_feat before each member call.
                # We'll slightly adjust run_one_setting here (local override) to set feature before transform:
                def run_one_setting_local(ens_name, members, use_smote, agg, weights, save_dir):
                    member_val_probas=[]; member_names=[]
                    for feat, model_name in members:
                        tr, inner_tag = transform_per_feature_builder(feat)
                        PFT.set_feat(feat)
                        Xtr, Xva = DATA[feat]["Xtr"], DATA[feat]["Xva"]
                        Ztr, Zva = PFT(Xtr), PFT(Xva)
                        ytr = Y_TR
                        if use_smote:
                            try:
                                sm_ = SMOTE(k_neighbors=5, random_state=RANDOM_STATE)
                                Ztr, ytr = sm_.fit_resample(Ztr, ytr)
                            except Exception as e:
                                print(f"[WARN] SMOTE failed: {e}")

                        mtype, obj = build_classifier(model_name, Ztr.shape[1], NC, feat)
                        mdl, Pva = fit_predict(mtype, obj, Ztr, ytr, Zva, NC, feat)
                        member_val_probas.append(Pva); member_names.append(f"{feat}|{model_name}")

                        # save per-member
                        sub = Path(save_dir)/f"{feat}_{model_name}"
                        sub.mkdir(parents=True, exist_ok=True)
                        met = eval_block(Y_VA, Pva)
                        pd.DataFrame([met]).to_csv(sub/"VAL_metrics.csv", index=False)
                        save_cm_cr("VAL", Y_VA, Pva, CLASS_NAMES, sub, print_console=False)

                    # diversity
                    div_rows=[]
                    for (i,(n1,P1)), (j,(n2,P2)) in itertools.combinations(enumerate(zip(member_names, member_val_probas)), 2):
                        Q,dis,corr = diversity_pairwise(Y_VA, P1, P2)
                        div_rows.append({"pair": f"{n1}__{n2}", "Q":Q, "disagreement":dis, "corr_correctness":corr})
                    if len(div_rows):
                        pd.DataFrame(div_rows).to_csv(Path(save_dir)/"VAL_diversity_pairs.csv", index=False)

                    # aggregation
                    out_rows=[]
                    if agg in ("hard","soft","avg"):
                        Pv = agg_hard(member_val_probas) if agg=="hard" else (agg_avg(member_val_probas) if agg=="avg" else agg_soft(member_val_probas))
                        met = eval_block(Y_VA, Pv)
                        out_rows.append(("VAL", agg.upper(), None, met))
                        pd.DataFrame([met]).to_csv(Path(save_dir)/f"VAL_{agg}_metrics.csv", index=False)
                        print(f"\n[{ens_name}] {tag}")
                        save_cm_cr(f"VAL_{agg}", Y_VA, Pv, CLASS_NAMES, Path(save_dir), print_console=True)
                    else:
                        best=(-1,None,None,None)
                        K=len(member_val_probas)
                        for w in weight_grid(K, step=SEARCH_SPACE["weighted_step"]):
                            Pv = agg_soft(member_val_probas, w)
                            met = eval_block(Y_VA, Pv)
                            out_rows.append(("VAL","WEIGHTED", w, met))
                            if met["macro_f1"] > best[0]:
                                best=(met["macro_f1"], w, met, Pv)
                        if best[1] is not None:
                            pd.DataFrame([best[2]]).to_csv(Path(save_dir)/"VAL_WEIGHTED_best_metrics.csv", index=False)
                            json.dump({"weights":[float(x) for x in best[1]]}, open(Path(save_dir)/"VAL_WEIGHTED_best_weights.json","w"), indent=2)
                            print(f"\n[{ens_name}] {tag} | BEST-WEIGHTED w={best[1]}")
                            save_cm_cr("VAL_WEIGHTED_best", Y_VA, best[3], CLASS_NAMES, Path(save_dir), print_console=True)
                    rows=[]
                    for _,what,w,met in out_rows:
                        rows.append({
                            "ensemble": ens_name,
                            "members": "+".join(member_names),
                            "transform": f"{'PCA-'+str(pvar) if kind=='pca' else 'DEA-'+str(bn)}",
                            "smote": use_smote,
                            "voting": what,
                            "weights": None if w is None else list(map(float,w)),
                            **met
                        })
                    return rows, member_val_probas, member_names

                rows, member_val_probas, member_names = run_one_setting_local(
                    ens_name, members, use_smote=sm, agg=agg, weights=None, save_dir=out_dir
                )
                all_val_rows.extend(rows)

# ================== VAL Top-20 & TEST Top-5 ==================
val_df = pd.DataFrame(all_val_rows)
val_df = val_df.sort_values(["macro_f1","auc_macro","mcc"], ascending=[False, False, False]).reset_index(drop=True)
val_df.to_csv(SAVE_ROOT/"VAL_all_results.csv", index=False)
print("\n=== TOP-20 by VAL Macro-F1 ===")
print(val_df.head(20).to_string(index=False))
val_df.head(20).to_csv(SAVE_ROOT/"VAL_top20.csv", index=False)

# Persist registry (append-friendly)
if REGISTRY_JSON.exists():
    reg = json.load(open(REGISTRY_JSON))
else:
    reg = {"runs":[]}
reg["runs"].append({
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "space": SEARCH_SPACE,
    "val_top20_path": str(SAVE_ROOT/"VAL_top20.csv"),
})
json.dump(reg, open(REGISTRY_JSON,"w"), indent=2)

# Select Top-5 (by VAL Macro-F1) for TEST; refit on Train∪Val with same transform/smote/voting/weights
print("\n=== Selected Top-5 for TEST ===")
top5 = val_df.head(5).copy()
print(top5[["ensemble","voting","transform","smote","weights","macro_f1","auc_macro","mcc"]].to_string(index=False))
top5.to_csv(SAVE_ROOT/"TEST_selected_top5_from_VAL.csv", index=False)

# For refit we must recreate transform per-feature, then aggregate with chosen voting/weights
test_rows=[]
for i, r in top5.iterrows():
    ens_name = r["ensemble"]
    # parse members
    m = [tuple(x.split("|")) for x in r["members"].split("+")]  # ("SWIN","MLP") etc.
    members = [(a, b) for (a,b) in m]
    trspec = r["transform"]
    smote_flag = bool(r["smote"])
    voting = r["voting"]
    weights = None if pd.isna(r["weights"]) or r["weights"] is None else eval(str(r["weights"]))

    # Rebuild per-feature transformers on Train∪Val? -> No. Leak-free requires: fit transformer on TRAIN only, then
    # for TEST we are allowed to refit on TRAIN∪VAL consistently? To keep parity with Stage-1b, we refit ML on Train∪Val,
    # but keep the representation transform as TRAIN∪VAL as well ONLY if we did so in Stage-1b. To stay conservative:
    # We'll FIT scaler+PCA/DEA on TRAIN∪VAL for TEST stage (common in deployment when VAL joins training).
    # If you prefer TRAIN-only, change use_trainval_rep = False.
    use_trainval_rep = True

    per_feat_cache = {}
    def build_transform_for_feat(feat_key):
        Xtr = DATA[feat_key]["Xtr"]; Xva = DATA[feat_key]["Xva"]
        Xfit = np.vstack([Xtr, Xva]) if use_trainval_rep else Xtr
        if trspec.startswith("PCA"):
            pvar = float(trspec.split("PCA-")[1])
            tr, tag = get_transformer("pca", Xfit, pca_var=pvar)
        elif trspec.startswith("DEA"):
            bn = int(trspec.split("DEA-")[1])
            tr, tag = get_transformer("dea", Xfit, dea_bottleneck=bn)
        else:
            raise ValueError("unknown transform spec")
        return tr

    # Wrapper to apply per-feature transform
    def transform_feat(feat_key, X):
        if feat_key not in per_feat_cache:
            per_feat_cache[feat_key] = build_transform_for_feat(feat_key)
        return per_feat_cache[feat_key](X)

    # Refit&Test
    out_dir = SAVE_ROOT/f"TEST/{ens_name}/{trspec}_SMOTE={smote_flag}_{voting}"
    out_dir.mkdir(parents=True, exist_ok=True)

    member_test_probas=[]
    for feat, model_name in members:
        Xtr, Xva, Xte = DATA[feat]["Xtr"], DATA[feat]["Xva"], DATA[feat]["Xte"]
        Ztr = transform_feat(feat, Xtr); Zva = transform_feat(feat, Xva); Zte = transform_feat(feat, Xte)
        Xfit = np.vstack([Ztr, Zva]); yfit = np.concatenate([Y_TR, Y_VA])
        if smote_flag:
            try:
                sm = SMOTE(k_neighbors=5, random_state=RANDOM_STATE)
                Xfit, yfit = sm.fit_resample(Xfit, yfit)
            except Exception as e:
                print(f"[WARN] SMOTE failed on TEST-refit: {e}")
        mtype, obj = build_classifier(model_name, Xfit.shape[1], NC, feat)
        Pte = refit_predict(mtype, obj, Xfit, yfit, Zte, NC, feat)
        member_test_probas.append(Pte)

    if voting=="HARD":
        P = agg_hard(member_test_probas)
    elif voting=="AVG":
        P = agg_avg(member_test_probas)
    elif voting=="SOFT":
        P = agg_soft(member_test_probas)
    elif voting=="WEIGHTED":
        P = agg_soft(member_test_probas, weights)
    met = eval_block(Y_TE, P)
    pd.DataFrame([met]).to_csv(out_dir/"TEST_metrics.csv", index=False)
    save_cm_cr("TEST", Y_TE, P, CLASS_NAMES, out_dir, print_console=True)

    test_rows.append({
        "validation_rank": int(i) + 1,
        "ensemble": ens_name, "transform": trspec, "smote": smote_flag, "voting": voting,
        "weights": weights, **met
    })

# Save TEST summary & macro-ROC of top-5
test_df = pd.DataFrame(test_rows).sort_values("validation_rank", ascending=True).reset_index(drop=True)
test_df.to_csv(SAVE_ROOT/"TEST_top5_summary.csv", index=False)
print("\n=== TEST summary in validation-selected order (not ranked by TEST metrics) ===")
print(test_df.to_string(index=False))

# Macro-ROC overlay for top-5 (macro-averaged per model) — optional quick plot using OvR
# (We reuse saved probas if needed; here we only aggregate-level probs kept in memory;
# for brevity we skip per-class curve overlay and plot bars of Macro-AUC.)
plt.figure(figsize=(8,4))
plt.bar(np.arange(len(test_df)), test_df["auc_macro"])
plt.xticks(np.arange(len(test_df)), [f"{r['ensemble']}|{r['voting']}" for _,r in test_df.iterrows()], rotation=30, ha='right')
plt.ylabel("Macro AUC (OvR)"); plt.title("Top-5 (from VAL) — TEST Macro-AUC")
plt.tight_layout()
plt.savefig(SAVE_ROOT/"TEST_top5_macro_auc.png", dpi=300)
plt.close()

print(f"\nAll artifacts saved under: {SAVE_ROOT}")

## Section 3A — Fixed-weight visual ensemble with validation calibration

**Research question:** How does the pre-specified three-member SWIN/EVA/COAT ensemble perform, and what ensemble temperature minimizes validation NLL?

**Selection rule:** the member list and global weights are treated as fixed for this comparison; temperature is chosen on validation NLL. The code reports the final ensemble on test but does not use test labels for oracle or rescue analysis. Verify that any fixed weights used in the paper were determined independently of test results.

In [ ]:
# STAGE 3A
# -*- coding: utf-8 -*-
# ============================================================
# Per-feature models, optional metadata, and leakage-aware ensemble evaluation
# Fixed members and pre-specified ensemble weights
#  w[SWIN|MLP(SwinMLP)]=0.55, w[EVA|MLP(EvaMLP)]=0.25, w[COAT|configured model]=0.20
# ============================================================

import os, sys, json, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# User configuration
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
SPLIT_DIR = str(PROJECT_ROOT / "data" / "splits")  # train_split.csv / val_split.csv / test_split.csv
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin_ft768"),
    "EVA": str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet_ft672"),
}
META_CSV_PATH = str(PROJECT_ROOT / "data" / "metadata" / "ISIC_2019_Training_Metadata.csv")

SAVE_DIR = str(PROJECT_ROOT / "outputs" / "ensemble_meta_runs")
RANDOM_STATE = 42
DEVICE = "auto"  # "cuda" | "cpu" | "auto"
PCA_VARIANCE = 0.99
CALIBRATION_BINS = 15

# Metadata feature flags
META_FLAGS = {
    "INCLUDE_AGE_2BIT"     : False,
    "AGE_THRESHOLD"        : 30,
    "INCLUDE_REGION_9BIT"  : False,
    "INCLUDE_GROUP_SCHEMA" : False,
    "GROUP_SCHEMA"         : "B",
    "INCLUDE_SEX_2BIT"     : False,
}

# ---- Selected ensemble members ----
MODELS_PER_FEATURE = {
    "SWIN": ["MLP(SwinMLP)"],
    "EVA" : ["MLP(EvaMLP)"],
    "COAT": ["LogReg"]
}

# ---- MLP presets ----
NAMED_PRESETS = {
    "SwinMLP": {
        "hidden": [2048], "dropout": 0.1605757337,
        "lr": 0.000341409509895, "weight_decay": 0.0001488348195284,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
    "EvaMLP": {
        "hidden": [2048], "dropout": 0.3203382207442766,
        "lr": 0.0009765455593461, "weight_decay": 8.212522451000109e-05,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
}

# Additional model hyperparameters
HP_DEFAULTS = {
    "LogReg": {"C": 2.3402196063650043, "max_iter": 587, "tol": 3.6058193649597426e-04,
               "solver": "lbfgs", "class_weight": None},
    "SVM"   : {"C": 2.0, "max_iter": 20000},
    "KNN"   : {"n_neighbors": 15, "weights": "distance", "p": 2},
    "XGB"   : {"n_estimators": 400, "max_depth": 6, "learning_rate": 0.08,
               "subsample": 0.9, "colsample_bytree": 0.7, "tree_method": "hist",
               "random_state": RANDOM_STATE},
    "LGBM"  : {"n_estimators": 600, "num_leaves": 48, "learning_rate": 0.05,
               "subsample": 0.9, "colsample_bytree": 0.7, "reg_lambda": 1.0,
               "random_state": RANDOM_STATE,"verbosity" : -1 },
}

# Fixed ensemble weights
GLOBAL_WEIGHTS = {
    "SWIN|MLP(SwinMLP)": 0.55,
    "EVA|MLP(EvaMLP)"  : 0.25,
    "COAT|LGBM"        : 0.20,
}
ENABLE_CLASSWISE = False   
TEMP_GRID = [0.70,0.75,0.80,0.85,0.90,0.95,1.00,1.05,1.10,1.15,1.20,1.25,1.30]

# ---- Determinism / CUDA fix ----
os.environ["CUBLAS_WORKSPACE_CONFIG"] = os.environ.get("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

def _pip(*pkgs):
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
        return True
    except Exception as e:
        print(f"[WARN] pip install failed for {pkgs}: {e}")
        return False

# -------------------- Torch --------------------
try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn
except Exception:
    _pip("torch","torchvision","torchaudio")
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn

# -------------------- Sklearn vs. --------------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, log_loss
from sklearn.model_selection import StratifiedKFold

# Optional XGBoost / LightGBM models
def _lazy_import_xgb():
    try:
        import xgboost as xgb
    except Exception:
        _pip("xgboost")
        try:
            import xgboost as xgb
        except Exception:
            xgb = None
    return xgb

def _lazy_import_lgbm():
    try:
        import lightgbm as lgb
    except Exception:
        _pip("lightgbm")
        try:
            import lightgbm as lgb
        except Exception:
            lgb = None
    return lgb

# -------------------- Util --------------------
def set_seed(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = True
set_seed(RANDOM_STATE)

def choose_device():
    if DEVICE == "cuda": return "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":  return "cpu"
    return "cuda" if torch.cuda.is_available() else "cpu"
DEV = choose_device()
print("Device:", DEV); os.makedirs(SAVE_DIR, exist_ok=True)

def _glob_any(p: Path, names):
    for nm in names:
        f = p / nm
        if f.exists(): return str(f)
    return None

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True, flags=0)
    return s.values

def _read_global_split_csv(split_csv_path):
    df = pd.read_csv(split_csv_path)
    id_col = None
    for cand in ["image_id","image","Image","filename","fname","img_id","name"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None: raise ValueError(f"No image_id/image column found in {split_csv_path}.")
    ids = _normalize_ids(df[id_col])

    if "label" in df.columns:
        y = df["label"].astype(int).values
    elif "class_name" in df.columns:
        classes = sorted(df["class_name"].unique().tolist())
        mp = {c:i for i,c in enumerate(classes)}
        y = df["class_name"].map(mp).astype(int).values
    else:
        raise ValueError(f"No label/class_name column found in {split_csv_path}.")
    return ids, y, df

def load_global_splits(split_dir):
    paths = {
        "train": os.path.join(split_dir, "train_split.csv"),
        "val"  : os.path.join(split_dir, "val_split.csv"),
        "test" : os.path.join(split_dir, "test_split.csv"),
    }
    out = {}
    for sp, pth in paths.items():
        ids, y, df = _read_global_split_csv(pth)
        out[sp] = {"ids": ids, "y": y, "df": df}
    df_tr = out["train"]["df"]
    if "class_name" in df_tr.columns:
        class_names = sorted(df_tr["class_name"].unique().tolist())
    else:
        class_names = [f"class{i}" for i in sorted(pd.Series(out["train"]["y"]).unique())]
    return out, class_names

def _load_features(dir_path: str, split: str):
    p = Path(dir_path)
    X_path = _glob_any(p, [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"])
    if X_path is None: raise FileNotFoundError(f"No {split} feature array found in {dir_path}.")
    return np.load(X_path)

def brier_score(y_true, proba):
    C = proba.shape[1]; Y = np.eye(C)[y_true]
    return float(np.mean(np.sum((proba - Y)**2, axis=1)))

def ece_score(y_true, proba, n_bins=CALIBRATION_BINS):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); accs = (pred == y_true).astype(float)
    bins = np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        if m.sum(): ece += m.mean() * abs(accs[m].mean() - conf[m].mean())
    return float(ece)

def topk_capture(y_true, proba, k=3):
    topk = np.argpartition(-proba, kth=k-1, axis=1)[:, :k]
    return float(np.any(topk == y_true[:, None], axis=1).mean())

def reliability_bins(y_true, proba, n_bins=CALIBRATION_BINS):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); correct = (pred == y_true).astype(int)
    bins = np.linspace(0,1,n_bins+1); rows=[]
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        rows.append({"bin": i, "count": int(m.sum()),
                     "avg_conf": float(np.nan if m.sum()==0 else conf[m].mean()),
                     "acc": float(np.nan if m.sum()==0 else correct[m].mean()),
                     "gap": float(np.nan if m.sum()==0 else abs(correct[m].mean()-conf[m].mean()))})
    return pd.DataFrame(rows)

def save_reports(prefix, y_true, proba, out_dir, class_names, print_console=True):
    os.makedirs(out_dir, exist_ok=True)
    y_pred = proba.argmax(1)
    row = {"split":prefix,
           "Acc": accuracy_score(y_true, y_pred),
           "MacroF1": f1_score(y_true, y_pred, average="macro", zero_division=0),
           "Brier": brier_score(y_true, proba),
           "ECE": ece_score(y_true, proba),
           "Top3": topk_capture(y_true, proba, 3),
           "N": len(y_true), "classes": proba.shape[1]}
    pd.DataFrame([row]).to_csv(os.path.join(out_dir, f"{prefix}_analysis_report.csv"), index=False)
    reliability_bins(y_true, proba).to_csv(os.path.join(out_dir, f"{prefix}_reliability_bins.csv"), index=False)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in class_names],
                         columns=[f"pred_{c}" for c in class_names])
    cm_df.to_csv(os.path.join(out_dir, f"{prefix}_confusion_matrix.csv"))
    cr = classification_report(y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True)
    cr_df = pd.DataFrame(cr); cr_df.to_csv(os.path.join(out_dir, f"{prefix}_classification_report.csv"))

    if print_console:
        print(f"\n[{out_dir}] {prefix.upper()} — Acc={row['Acc']:.4f} | MacroF1={row['MacroF1']:.4f} | "
              f"Brier={row['Brier']:.4f} | ECE={row['ECE']:.4f} | Top3={row['Top3']:.4f}")
        print("\nConfusion Matrix:\n", cm_df.to_string())
        print("\nClassification Report:\n", cr_df.to_string())
    return row

# -------------------- Meta Engineering --------------------
def read_meta_local(csv_path):
    df = pd.read_csv(csv_path)
    id_col = None
    for cand in ["image","image_id","Image","filename","fname","name","img_id"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None:
        raise ValueError("The metadata CSV must contain an 'image' or 'image_id' column.")
    df["image_id"] = _normalize_ids(df[id_col])
    for c in ["anatom_site_general","age_approx","sex"]:
        if c not in df.columns:
            df[c] = np.nan
    return df

REGION9_LIST = [
    "head/neck","torso_anterior","torso_posterior","torso_lateral",
    "upper_extremity","lower_extremity","acral","mucosal","other_or_missing"
]
def map_region9(anat: str):
    if pd.isna(anat): return "other_or_missing"
    s = str(anat).strip().lower()
    if "head" in s or "neck" in s: return "head/neck"
    if "torso" in s and "anterior" in s: return "torso_anterior"
    if "torso" in s and "posterior" in s: return "torso_posterior"
    if "torso" in s and "lateral" in s: return "torso_lateral"
    if "upper" in s: return "upper_extremity"
    if "lower" in s: return "lower_extremity"
    if "palms" in s or "soles" in s or "acral" in s: return "acral"
    if "oral" in s or "genital" in s or "mucosal" in s: return "mucosal"
    return "other_or_missing"

def map_group(schema: str, anat: str):
    if pd.isna(anat): base = "missing"
    else: base = str(anat).strip().lower()

    headneck = ("head" in base or "neck" in base)
    upper    = ("upper" in base)
    lower    = ("lower" in base)
    torso    = ("torso" in base)
    lateral  = ("lateral" in base and "torso" in base)
    acral    = ("palms" in base or "soles" in base or "acral" in base)
    mucosal  = ("oral" in base or "genital" in base or "mucosal" in base)

    if schema.upper()=="A":
        if headneck: return "head/neck"
        if upper:    return "upper_extremity"
        if lower:    return "lower_extremity"
        if torso:    return "torso"
        if acral or mucosal: return "special"
        return "missing"
    elif schema.upper()=="B":
        if headneck: return "head/neck"
        if upper:    return "upper_extremity"
        if lower:    return "lower_extremity"
        if torso and not lateral: return "torso"
        if lateral:  return "lateral_torso"
        if acral:    return "acral"
        if mucosal:  return "mucosal"
        return "missing"
    else:
        if headneck: return "head/neck"
        if upper or lower: return "extremity"
        if torso:    return "torso"
        if acral or mucosal: return "special"
        return "other"

GROUP_CATS = {
    "A": ["head/neck","upper_extremity","lower_extremity","torso","special","missing"],
    "B": ["head/neck","upper_extremity","lower_extremity","torso","lateral_torso","acral","mucosal","missing"],
    "C": ["head/neck","extremity","torso","special","other"],
}

def build_meta_matrix(meta_df, ids, flags):
    df = pd.DataFrame({"image_id": ids}).merge(meta_df, on="image_id", how="left")
    cols = []

    A = []
    if flags["INCLUDE_AGE_2BIT"]:
        thr = float(flags["AGE_THRESHOLD"])
        age = pd.to_numeric(df["age_approx"], errors="coerce")
        a_lt = (age < thr).astype(float).fillna(0.0).values.reshape(-1,1)
        a_ge = (age >= thr).astype(float).fillna(0.0).values.reshape(-1,1)
        A = [a_lt, a_ge]; cols += ["age_lt","age_ge"]

    R9 = []
    if flags["INCLUDE_REGION_9BIT"]:
        r = df["anatom_site_general"].apply(map_region9)
        for cat in REGION9_LIST:
            R9.append((r == cat).astype(float).values.reshape(-1,1))
        cols += [f"reg9_{c}" for c in REGION9_LIST]

    G = []
    if flags["INCLUDE_GROUP_SCHEMA"]:
        schema = flags["GROUP_SCHEMA"].upper()
        r = df["anatom_site_general"].apply(lambda s: map_group(schema, s))
        cats = GROUP_CATS[schema]
        for cat in cats:
            G.append((r == cat).astype(float).values.reshape(-1,1))
        cols += [f"group_{schema}_{c}" for c in cats]

    S = []
    if flags["INCLUDE_SEX_2BIT"]:
        sx = df["sex"].astype(str).str.lower()
        male  = (sx == "male").astype(float).values.reshape(-1,1)
        female= (sx == "female").astype(float).values.reshape(-1,1)
        S = [male, female]; cols += ["sex_male","sex_female"]

    mats = [m for m in (A+R9+G+S) if len(m)]
    if mats:
        M = np.concatenate(mats, axis=1).astype(np.float32)
    else:
        M = np.zeros((len(df),0), dtype=np.float32)
    return M, cols

# -------------------- MLP --------------------
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(1024,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def _make_loader(X, y, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=0, drop_last=False)

def train_mlp_fixed_epochs(Xfit, yfit, n_classes, hp, device, log_tag):
    from sklearn.metrics import f1_score, accuracy_score
    loader = _make_loader(Xfit, yfit, hp["batch_size"], shuffle=True)
    model = RobustMLP(Xfit.shape[1], n_classes, tuple(hp["hidden"]), hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
    model.train()
    for ep in range(1, hp["epochs"]+1):
        for xb,yb in loader:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss=crit(model(xb), yb); loss.backward(); opt.step()
        if ep % max(1,hp.get("log_every",5))==0 or ep==1:
            model.eval()
            with torch.no_grad():
                lg = model(torch.tensor(Xfit, dtype=torch.float32, device=device)).cpu().numpy()
            proba = torch.softmax(torch.tensor(lg), dim=1).numpy()
            y_hat = proba.argmax(1)
            acc = accuracy_score(yfit, y_hat); f1 = f1_score(yfit, y_hat, average="macro", zero_division=0)
            print(f"[MLP] ep {ep:03d} | {log_tag} Acc {acc:.4f} | MacroF1 {f1:.4f}")
            model.train()
    return model

def logits_and_proba(model, X, device):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X), dtype=torch.long))
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=0, drop_last=False)
    model.eval(); out=[]
    with torch.no_grad():
        for (xb,_) in loader:
            xb=xb.to(device); out.append(model(xb).cpu())
    lg = torch.cat(out,0).numpy()
    P  = torch.softmax(torch.tensor(lg), dim=1).numpy()
    return lg, P

def apply_temp_logits(logits, T):
    return torch.softmax(torch.tensor(logits)/T, dim=1).numpy()

# Fit/transform helpers
def fit_transform_train_then_val(Xtr, Xva, pca_var):
    scaler = StandardScaler().fit(Xtr)
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva)
    if pca_var is None: return scaler, None, Ztr, Zva
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva)

def fit_transform_trainval_then_test(Xtr, Xva, Xte, pca_var):
    scaler = StandardScaler().fit(np.vstack([Xtr, Xva]))
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva); Zte = scaler.transform(Xte)
    if pca_var is None: return scaler, None, Ztr, Zva, Zte
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(np.vstack([Ztr,Zva]))
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)

# Member-model training
def parse_model_token(tok: str):
    if tok.startswith("MLP(") and tok.endswith(")"):
        name = tok[4:-1].strip()
        return "MLP", name
    return tok, None

def build_and_fit_val_predict(model_key, preset_name, Ztr, ytr, Zva, n_class):
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        model = train_mlp_fixed_epochs(Ztr, ytr, n_class, hp, DEV, "TRAIN")
        lg_va, _ = logits_and_proba(model, Zva, DEV)
        P_va = apply_temp_logits(lg_va, hp["T"])
        return ("MLP", hp, model, P_va)
    elif model_key=="LGBM":
        lgb = _lazy_import_lgbm()
        if lgb is None:
            print("[WARN] LightGBM is unavailable; skipping this model.")
            return ("LGBM", None, None, None)
        hp = HP_DEFAULTS["LGBM"].copy()
        clf = lgb.LGBMClassifier(**hp)
        clf.fit(Ztr, ytr)
        P_va = clf.predict_proba(Zva)
        return ("LGBM", hp, clf, P_va)
    else:
        raise ValueError(f"Unknown model: {model_key}")

def refit_and_predict_test(model_key, preset_name, trained, Xfit, yfit, Zte, n_class):
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        model = train_mlp_fixed_epochs(Xfit, yfit, n_class, hp, DEV, "TRAIN∪VAL")
        lg_te, _ = logits_and_proba(model, Zte, DEV)
        P_te = apply_temp_logits(lg_te, hp["T"])
        return P_te
    elif model_key=="LGBM":
        lgb = _lazy_import_lgbm()
        if lgb is None:
            print("[WARN] LightGBM is unavailable; skipping this test evaluation.")
            return None
        hp = HP_DEFAULTS["LGBM"].copy()
        clf = lgb.LGBMClassifier(**hp); clf.fit(Xfit, yfit)
        return clf.predict_proba(Zte)
    else:
        raise ValueError(f"Unknown model: {model_key}")

# -------------------- Ensemble helpers --------------------
def soft_mean(P_list, weights=None):
    P_list = [P for P in P_list if P is not None]
    if not P_list: return None
    if weights is None: weights = np.ones(len(P_list), dtype=float)
    else: weights = np.array(weights[:len(P_list)], dtype=float)
    S = np.zeros_like(P_list[0], dtype=np.float64)
    for P, w in zip(P_list, weights): S += w * P
    S = S / max(1e-12, weights.sum())
    return S


def hard_preds_from_proba(P_list):
    return [P.argmax(1) for P in P_list if P is not None]

def accuracy(y, yhat):
    return float((yhat == y).mean())

def macroF1(y, yhat):
    return float(f1_score(y, yhat, average="macro", zero_division=0))

def per_class_f1(y, yhat, n_cls):
    arr = f1_score(y, yhat, average=None, labels=list(range(n_cls)), zero_division=0)
    return arr.astype(float)

def majority_vote(votes, n_class):
    K, N = votes.shape
    out = np.zeros(N, dtype=int)
    for i in range(N):
        cnt = np.bincount(votes[:, i], minlength=n_class)
        out[i] = int(np.argmax(cnt))
    return out

def compute_rescue_table(y, ensemble_pred, member_preds, member_keys):
    rows = []
    wrong_mask = (ensemble_pred != y)
    right_mask = (ensemble_pred == y)
    for key, p in zip(member_keys, member_preds):
        rescued = int(((p == y) & wrong_mask).sum())
        rescued_ratio = 100.0 * rescued / max(1, wrong_mask.sum())
        solo_correct = int((p == y).sum())
        hurt = int(((p != y) & right_mask).sum())
        rows.append({
            "member": key,
            "rescued": rescued,
            "rescued_ratio_%": round(rescued_ratio, 2),
            "solo_correct": solo_correct,
            "hurt_when_ens_right": hurt
        })
    df = pd.DataFrame(rows).sort_values(["rescued","solo_correct"], ascending=[False, False]).reset_index(drop=True)
    return df

def oracle_any_correct(y, preds_list, n_class, fallback_pred=None):
    K = len(preds_list); N = len(y)
    any_ok = np.zeros(N, dtype=bool)
    for p in preds_list:
        any_ok |= (p == y)
    if fallback_pred is None:
        votes = np.stack(preds_list, axis=0)
        fallback_pred = majority_vote(votes, n_class)
    oracle_pred = fallback_pred.copy()
    oracle_pred[any_ok] = y[any_ok]
    return oracle_pred, any_ok

def k_of_m_distribution(y, preds_list):
    K = len(preds_list); N = len(y)
    mat = np.stack([(p == y) for p in preds_list], axis=0)
    correct_counts = mat.sum(axis=0)
    hist = np.bincount(correct_counts, minlength=K+1)
    dist = [{"k_correct": k, "count": int(hist[k]), "ratio_%": round(100.0*hist[k]/N, 2)} for k in range(K+1)]
    return pd.DataFrame(dist)

# Main workflow
def main():
    set_seed(RANDOM_STATE)
    root = SAVE_DIR; os.makedirs(root, exist_ok=True)

    
    splits, class_names = load_global_splits(SPLIT_DIR)
    n_class = len(class_names)
    print(f"[Global splits] classes={class_names}")

    # Load feature arrays
    data = {}
    for feat, path in FEATURE_DIRS.items():
        Xtr = _load_features(path, "train"); Xva = _load_features(path, "val"); Xte = _load_features(path, "test")
        ids_tr = splits["train"]["ids"]; y_tr = splits["train"]["y"]
        ids_va = splits["val"]["ids"];   y_va = splits["val"]["y"]
        ids_te = splits["test"]["ids"];  y_te = splits["test"]["y"]
        assert len(Xtr)==len(y_tr)==len(ids_tr)
        assert len(Xva)==len(y_va)==len(ids_va)
        assert len(Xte)==len(y_te)==len(ids_te)
        data[feat] = {"Xtr":Xtr,"ytr":y_tr,"ids_tr":ids_tr,
                      "Xva":Xva,"yva":y_va,"ids_va":ids_va,
                      "Xte":Xte,"yte":y_te,"ids_te":ids_te}

    
    meta_all = read_meta_local(META_CSV_PATH)
    M_tr, cols_tr = build_meta_matrix(meta_all, data["SWIN"]["ids_tr"], META_FLAGS)
    M_va, cols_va = build_meta_matrix(meta_all, data["SWIN"]["ids_va"], META_FLAGS)
    M_te, cols_te = build_meta_matrix(meta_all, data["SWIN"]["ids_te"], META_FLAGS)
    assert cols_tr==cols_va==cols_te
    print(f"[META] columns: {len(cols_tr)} -> {cols_tr if len(cols_tr)<=10 else str(cols_tr[:10])+' ...'}")

    # 4) Append metadata features to the feature matrices
    if M_tr.shape[1] > 0:
        for k in data.keys():
            data[k]["Xtr"] = np.concatenate([data[k]["Xtr"], M_tr], axis=1).astype(np.float32)
            data[k]["Xva"] = np.concatenate([data[k]["Xva"], M_va], axis=1).astype(np.float32)
            data[k]["Xte"] = np.concatenate([data[k]["Xte"], M_te], axis=1).astype(np.float32)

    # Selected model members
    members = []  # dict: key, feat, model, P_val, P_te, y_val, y_te
    for feat, model_list in MODELS_PER_FEATURE.items():
        print(f"\n===== {feat}: {len(model_list)} model =====")
        Xtr,ytr,Xva,yva,Xte,yte = data[feat]["Xtr"],data[feat]["ytr"],data[feat]["Xva"],data[feat]["yva"],data[feat]["Xte"],data[feat]["yte"]

        
        sc1, pca1, Ztr, Zva = fit_transform_train_then_val(Xtr, Xva, PCA_VARIANCE)

        for tok in model_list:
            model_key, preset = parse_model_token(tok)
            print(f"[{feat}] model = {tok}")
            mdl_name, hp, trained, P_va = build_and_fit_val_predict(model_key, preset, Ztr, ytr, Zva, n_class)
            if P_va is None:
                print(f"[WARN] {feat}|{tok}: validation probabilities could not be generated; skipping this model.")
                continue

            out_dir = os.path.join(SAVE_DIR, feat, tok.replace("/","_"), "VAL")
            save_reports("val", yva, P_va, out_dir, class_names, print_console=True)

            
            sc2, pca2, Ztr2, Zva2, Zte2 = fit_transform_trainval_then_test(Xtr, Xva, Xte, PCA_VARIANCE)
            Xfit = np.vstack([Ztr2, Zva2]); yfit = np.concatenate([ytr, yva])

            P_te = refit_and_predict_test(model_key, preset, trained, Xfit, yfit, Zte2, n_class)
            if P_te is None:
                print(f"[WARN] {feat}|{tok}: test probabilities could not be generated; skipping this model.")
                continue
            out_dir_te = os.path.join(SAVE_DIR, feat, tok.replace("/","_"), "TEST")
            save_reports("test", yte, P_te, out_dir_te, class_names, print_console=True)
            np.save(os.path.join(out_dir_te, "test_proba.npy"), P_te)

            members.append({"key": f"{feat}|{tok}", "feat": feat, "model": tok,
                            "P_val": P_va, "P_te": P_te, "y_val": yva, "y_te": yte})

    if not members:
        raise RuntimeError("No ensemble members were created. Check MODELS_PER_FEATURE.")

    
    y_val = members[0]["y_val"]; y_te = members[0]["y_te"]
    for m in members:
        assert np.array_equal(m["y_val"], y_val) and np.array_equal(m["y_te"], y_te)

    P_val_members = [m["P_val"] for m in members]
    P_te_members  = [m["P_te"]  for m in members]
    member_keys   = [m["key"]   for m in members]

    # Align weights with member order.
    w_init = np.array([GLOBAL_WEIGHTS.get(k, 0.0) for k in member_keys], dtype=np.float64)
    if not np.isclose(w_init.sum(), 1.0):
        w_init = w_init / max(1e-12, w_init.sum())
        print(f"[INFO] Weights were normalized → {dict(zip(member_keys, np.round(w_init,4)))}")

    print("[Ensemble] Members and weights:")
    for k, w in zip(member_keys, w_init):
        print(f" - {k:>24s} : {w:.4f}")

    P_val_final = soft_mean(P_val_members, w_init)
    P_te_final  = soft_mean(P_te_members,  w_init)

    # Validation-only diagnostic analysis. Oracle outputs are not deployable model results.
    val_member_preds = hard_preds_from_proba(P_val_members)
    val_ens_pred = P_val_final.argmax(1)
    df_rescue_val = compute_rescue_table(y_val, val_ens_pred, val_member_preds, member_keys)
    df_rescue_val.to_csv(os.path.join(SAVE_DIR, "rescue_val.csv"), index=False)
    print("\n=== RESCUE diagnostic — VALIDATION ONLY ===")
    print(df_rescue_val.to_string(index=False))

    val_oracle_pred, val_any_ok = oracle_any_correct(
        y_val, val_member_preds, len(class_names), fallback_pred=val_ens_pred
    )
    print("\n=== ORACLE diagnostic — VALIDATION ONLY (not a deployable model) ===")
    print(f"Acc={accuracy(y_val, val_oracle_pred):.4f} | MacroF1={macroF1(y_val, val_oracle_pred):.4f} | "
          f"rescued={int(val_any_ok.sum())}/{len(y_val)} ({100.0*val_any_ok.mean():.2f}%)")
    kdist_val = k_of_m_distribution(y_val, val_member_preds)
    print("Correct-member-count distribution (VALIDATION):")
    print(kdist_val.to_string(index=False))
    kdist_val.to_csv(os.path.join(SAVE_DIR, "oracle_kdist_val.csv"), index=False)

    
    print("\n=== Pick temperature T on VAL (NLL) ===")
    bestT, bestNLL = 1.0, float("inf")
    for T in [0.70,0.75,0.80,0.85,0.90,0.95,1.00,1.05,1.10,1.15,1.20,1.25,1.30]:
        Pv = torch.softmax(torch.log(torch.tensor(P_val_final+1e-12))/T, dim=1).numpy()
        nll = log_loss(y_val, Pv, labels=list(range(len(class_names))))
        print(f"T={T:.2f} | VAL NLL={nll:.6f}")
        if nll < bestNLL: bestNLL, bestT = nll, T

    P_test = torch.softmax(torch.log(torch.tensor(P_te_final+1e-12))/bestT, dim=1).numpy()

    out_dir = os.path.join(SAVE_DIR, "ENSEMBLE_FINAL"); os.makedirs(out_dir, exist_ok=True)
    from sklearn.metrics import f1_score as _f1
    save_reports("test", y_te, P_test, out_dir, class_names, print_console=True)
    np.save(os.path.join(out_dir, "test_proba.npy"), P_test)
    with open(os.path.join(out_dir, "members.json"), "w") as f:
        json.dump(member_keys, f, indent=2)
    with open(os.path.join(out_dir, "ensemble_meta.json"), "w") as f:
        json.dump({"strategy":"GLOBAL_FIXED",
                   "weights": dict(zip(member_keys, [float(x) for x in w_init])),
                   "T": float(bestT)}, f, indent=2)

    print(f"\nDone. Strategy=GLOBAL_FIXED | Weights={dict(zip(member_keys, np.round(w_init,4)))} | T={bestT:.2f} | Outputs → {out_dir}")


def k_of_m_distribution(y, preds_list):
    K = len(preds_list); N = len(y)
    mat = np.stack([(p == y) for p in preds_list], axis=0)
    correct_counts = mat.sum(axis=0)
    hist = np.bincount(correct_counts, minlength=K+1)
    dist = [{"k_correct": k, "count": int(hist[k]), "ratio_%": round(100.0*hist[k]/N, 2)} for k in range(K+1)]
    return pd.DataFrame(dist)

if __name__ == "__main__":
    main()

## Section 3 — A/B/C/D ensemble variants

**Research question:** How do a fixed-weight reference, validation-tuned soft weights, alternative voting rules, and an expanded member set compare?

**Selection rule:** dynamic weights are searched on validation Macro-F1, and calibration temperatures are selected using validation NLL. Variant A uses a pre-specified fixed-weight configuration. Test results are reporting-only and do not select the winner.

In [ ]:


# -*- coding: utf-8 -*-
# ============================================================






# Common components:



# ============================================================

import os, sys, json, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# User configuration
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
SPLIT_DIR = str(PROJECT_ROOT / "data" / "splits")
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin_ft768"),
    "EVA": str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet_ft672"),
}
SAVE_DIR = str(PROJECT_ROOT / "outputs" / "ensemble_meta_runs" / "Stage3")
RANDOM_STATE = 42
DEVICE = "auto"  # "cuda" | "cpu" | "auto"


PCA_VARIANCE = 0.99          # None | 0.95 | 0.99
DEA_TARGET = None            
SMOTE_ON = False             


NAMED_PRESETS = {
    "SwinMLP": {
        "hidden": [2048], "dropout": 0.1605757337,
        "lr": 0.000341409509895, "weight_decay": 0.0001488348195284,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
    "EvaMLP": {
        "hidden": [2048], "dropout": 0.3203382207442766,
        "lr": 0.0009765455593461, "weight_decay": 8.212522451000109e-05,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
    
    "SwinRobust": {
        "hidden": [2048], "dropout": 0.1605757337,
        "lr": 0.000341409509895, "weight_decay": 0.0001488348195284,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
    "EvaRobust": {
        "hidden": [2048], "dropout": 0.3203382207442766,
        "lr": 0.0009765455593461, "weight_decay": 8.212522451000109e-05,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
}

HP_DEFAULTS = {
    "LogReg": {"C": 2.3402196063650043, "max_iter": 587, "tol": 3.6058193649597426e-04,
               "solver": "lbfgs", "class_weight": None},
    "LGBM"  : {"n_estimators": 600, "num_leaves": 48, "learning_rate": 0.05,
               "subsample": 0.9, "colsample_bytree": 0.7, "reg_lambda": 1.0,
               "random_state": RANDOM_STATE, "verbosity": -1 },
}


GLOBAL_FIXED_WEIGHTS_A = {
    "SWIN|MLP(SwinMLP)": 0.55,
    "EVA|MLP(EvaMLP)"  : 0.25,
    "COAT|LGBM"        : 0.20,
}

TEMP_GRID = [0.70,0.75,0.80,0.85,0.90,0.95,1.00,1.05,1.10,1.15,1.20,1.25,1.30]

# -------------------- Torch --------------------
def _pip(*pkgs):
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
        return True
    except Exception as e:
        print(f"[WARN] pip install failed for {pkgs}: {e}")
        return False

try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn
except Exception:
    _pip("torch","torchvision","torchaudio")
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn

# -------------------- Sklearn --------------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report, log_loss)

def _lazy_import_lgbm():
    try:
        import lightgbm as lgb
    except Exception:
        _pip("lightgbm")
        try:
            import lightgbm as lgb
        except Exception:
            lgb = None
    return lgb

# -------------------- Util --------------------
def set_seed(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = True

def choose_device():
    if DEVICE == "cuda": return "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":  return "cpu"
    return "cuda" if torch.cuda.is_available() else "cpu"

set_seed(RANDOM_STATE)
DEV = choose_device()
print("Device:", DEV); os.makedirs(SAVE_DIR, exist_ok=True)

def _glob_any(p: Path, names):
    for nm in names:
        f = p / nm
        if f.exists(): return str(f)
    return None

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True, flags=0)
    return s.values

def _read_split_csv(split_csv_path):
    df = pd.read_csv(split_csv_path)
    id_col = None
    for cand in ["image_id","image","Image","filename","fname","img_id","name"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None: raise ValueError(f"No image_id/image column found in {split_csv_path}.")
    ids = _normalize_ids(df[id_col])

    if "label" in df.columns:
        y = df["label"].astype(int).values
    elif "class_name" in df.columns:
        classes = sorted(df["class_name"].unique().tolist())
        mp = {c:i for i,c in enumerate(classes)}
        y = df["class_name"].map(mp).astype(int).values
    else:
        raise ValueError(f"No label/class_name column found in {split_csv_path}.")
    return ids, y, df

def load_global_splits(split_dir):
    paths = {
        "train": os.path.join(split_dir, "train_split.csv"),
        "val"  : os.path.join(split_dir, "val_split.csv"),
        "test" : os.path.join(split_dir, "test_split.csv"),
    }
    out = {}
    for sp, pth in paths.items():
        ids, y, df = _read_split_csv(pth)
        out[sp] = {"ids": ids, "y": y, "df": df}
    df_tr = out["train"]["df"]
    if "class_name" in df_tr.columns:
        class_names = sorted(df_tr["class_name"].unique().tolist())
    else:
        class_names = [f"class{i}" for i in sorted(pd.Series(out["train"]["y"]).unique())]
    return out, class_names

def _load_features(dir_path: str, split: str):
    p = Path(dir_path)
    X_path = _glob_any(p, [f"{split}_features.npy", f"features_{split}.npy",
                           f"X_{split}.npy", f"{split}.npy"])
    if X_path is None: raise FileNotFoundError(f"No {split} feature array found in {dir_path}.")
    return np.load(X_path)


def print_cm_cr(prefix, y_true, y_pred, out_dir, class_names, print_console=True):
    os.makedirs(out_dir, exist_ok=True)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in class_names],
                         columns=[f"pred_{c}" for c in class_names])
    cr = classification_report(y_true, y_pred, target_names=class_names,
                               zero_division=0, output_dict=True)
    cr_df = pd.DataFrame(cr)

    acc = accuracy_score(y_true, y_pred)
    mac = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mic = f1_score(y_true, y_pred, average="micro", zero_division=0)

    pd.DataFrame([{"split":prefix,"Acc":acc,"MacroF1":mac,"MicroF1":mic}])\
      .to_csv(os.path.join(out_dir, f"{prefix}_summary.csv"), index=False)
    cm_df.to_csv(os.path.join(out_dir, f"{prefix}_CM.csv"))
    cr_df.to_csv(os.path.join(out_dir, f"{prefix}_CR.csv"))

    if print_console:
        print(f"{prefix.upper()} — Acc={acc:.4f} | Macro-F1={mac:.4f} | Micro-F1={mic:.4f}")
        print("CM:")
        print(cm_df.to_string())
        print("CR:")
        print(cr_df.to_string())

def save_proba(prefix, proba, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    np.save(os.path.join(out_dir, f"{prefix}_proba.npy"), proba)


class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(1024,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def _make_loader(X, y, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g,
                      num_workers=0, drop_last=False)

def train_mlp_fixed_epochs(Xfit, yfit, n_classes, hp, device, log_tag):
    loader = _make_loader(Xfit, yfit, hp["batch_size"], shuffle=True)
    model = RobustMLP(Xfit.shape[1], n_classes, tuple(hp["hidden"]), hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
    model.train()
    for ep in range(1, hp["epochs"]+1):
        for xb,yb in loader:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss=crit(model(xb), yb); loss.backward(); opt.step()
        if ep % max(1,hp.get("log_every",5))==0 or ep==1:
            model.eval()
            with torch.no_grad():
                lg = model(torch.tensor(Xfit, dtype=torch.float32, device=device)).cpu().numpy()
            proba = torch.softmax(torch.tensor(lg), dim=1).numpy()
            y_hat = proba.argmax(1)
            acc = accuracy_score(yfit, y_hat)
            mac = f1_score(yfit, y_hat, average="macro", zero_division=0)
            print(f"[MLP] ep {ep:03d} | {log_tag} Acc {acc:.4f} | MacroF1 {mac:.4f}")
            model.train()
    return model

def logits_and_proba(model, X, device):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X), dtype=torch.long))
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=0, drop_last=False)
    model.eval(); out=[]
    with torch.no_grad():
        for (xb,_) in loader:
            xb=xb.to(device); out.append(model(xb).cpu())
    lg = torch.cat(out,0).numpy()
    P  = torch.softmax(torch.tensor(lg), dim=1).numpy()
    return lg, P

def apply_temp_logits(logits, T):
    return torch.softmax(torch.tensor(logits)/T, dim=1).numpy()


def pca_safe_fit_transform(Xtr, Xva, pca_var):
    scaler = StandardScaler().fit(Xtr)
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva)
    if pca_var is None: return scaler, None, Ztr, Zva
    
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva)

def pca_safe_fit_transform_test(Xtr, Xva, Xte, pca_var):
    scaler = StandardScaler().fit(np.vstack([Xtr, Xva]))
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva); Zte = scaler.transform(Xte)
    if pca_var is None: return scaler, None, Ztr, Zva, Zte
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(np.vstack([Ztr, Zva]))
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)


def parse_model_token(tok: str):
    if tok.startswith("MLP(") and tok.endswith(")"):
        name = tok[4:-1].strip()
        return "MLP", name
    return tok, None

def print_and_save(prefix, y_true, proba, out_dir, class_names, print_console=True):
    save_proba(prefix, proba, out_dir)
    print_cm_cr(prefix, y_true, proba.argmax(1), out_dir, class_names, print_console)

def fit_val_and_test(model_key, preset_name,
                     Ztr, ytr, Zva, yva,
                     Ztr2, Zva2, Zte2, yte,
                     n_class, out_root, feat, tok, class_names):
    """VAL (Train-only PCA) + TEST (Train∪Val PCA)."""
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        # VAL
        model = train_mlp_fixed_epochs(Ztr, ytr, n_class, hp, DEV, f"{feat}|{tok} VAL")
        lg_va, _ = logits_and_proba(model, Zva, DEV)
        P_va = apply_temp_logits(lg_va, hp["T"])
        out_val = os.path.join(out_root, feat, tok.replace("/","_"), "VAL")
        print_and_save("val", yva, P_va, out_val, class_names, print_console=True)
        # TEST (refit Train∪Val)
        model2 = train_mlp_fixed_epochs(np.vstack([Ztr2, Zva2]),
                                        np.concatenate([ytr, yva]),
                                        n_class, hp, DEV, f"{feat}|{tok} TEST-fit")
        lg_te, _ = logits_and_proba(model2, Zte2, DEV)
        P_te = apply_temp_logits(lg_te, hp["T"])
        out_test = os.path.join(out_root, feat, tok.replace("/","_"), "TEST")
        print_and_save("test", yte, P_te, out_test, class_names, print_console=True)
        return P_va, P_te

    elif model_key=="LGBM":
        lgb = _lazy_import_lgbm()
        if lgb is None:
            print("[WARN] LightGBM is unavailable; skipping this model.")
            return None, None
        hp = HP_DEFAULTS["LGBM"].copy()
        clf = lgb.LGBMClassifier(**hp)
        clf.fit(Ztr, ytr)
        P_va = clf.predict_proba(Zva)
        out_val = os.path.join(out_root, feat, tok.replace("/","_"), "VAL")
        print_and_save("val", yva, P_va, out_val, class_names, print_console=True)

        clf2 = lgb.LGBMClassifier(**hp).fit(np.vstack([Ztr2, Zva2]),
                                            np.concatenate([ytr, yva]))
        P_te = clf2.predict_proba(Zte2)
        out_test = os.path.join(out_root, feat, tok.replace("/","_"), "TEST")
        print_and_save("test", yte, P_te, out_test, class_names, print_console=True)
        return P_va, P_te

    elif model_key=="LogReg":
        hp = HP_DEFAULTS["LogReg"].copy()
        clf = LogisticRegression(**hp)
        clf.fit(Ztr, ytr)
        P_va = clf.predict_proba(Zva)
        out_val = os.path.join(out_root, feat, tok.replace("/","_"), "VAL")
        print_and_save("val", yva, P_va, out_val, class_names, print_console=True)

        clf2 = LogisticRegression(**hp).fit(np.vstack([Ztr2, Zva2]),
                                            np.concatenate([ytr, yva]))
        P_te = clf2.predict_proba(Zte2)
        out_test = os.path.join(out_root, feat, tok.replace("/","_"), "TEST")
        print_and_save("test", yte, P_te, out_test, class_names, print_console=True)
        return P_va, P_te

    else:
        raise ValueError(f"Unknown model: {model_key}")

# -------------------- Ensemble --------------------
def soft_mix(P_list, weights):
    P_list = [P for P in P_list if P is not None]
    if not P_list: return None
    w = np.array(weights[:len(P_list)], dtype=float)
    S = np.zeros_like(P_list[0], dtype=np.float64)
    for P, a in zip(P_list, w): S += a * P
    S /= max(1e-12, w.sum())
    return S

def hard_vote(P_list):
    preds = [P.argmax(1) for P in P_list if P is not None]
    votes = np.stack(preds, axis=0)
    N = votes.shape[1]
    out = np.zeros(N, dtype=int)
    for i in range(N):
        cnt = np.bincount(votes[:, i])
        out[i] = int(np.argmax(cnt))
    return out

def pick_T_by_val_nll(P_val, y_val, t_grid=TEMP_GRID):
    bestT, bestNLL = 1.0, float("inf")
    for T in t_grid:
        Pv = torch.softmax(torch.log(torch.tensor(P_val+1e-12))/T, dim=1).numpy()
        nll = log_loss(y_val, Pv, labels=list(range(P_val.shape[1])))
        if nll < bestNLL: bestNLL, bestT = nll, T
    return float(bestT)


def grid_weights_step_05_for_3():
    ws = []
    step = 0.05
    vals = np.arange(0.0, 1.0+1e-9, step)
    for a in vals:
        for b in vals:
            c = 1.0 - a - b
            if c < -1e-9: continue
            if abs(round(c/step)*step - c) < 1e-9 and 0.0 <= c <= 1.0:
                ws.append((round(a,2), round(b,2), round(c,2)))
    return ws

def coord_ascent_weights(P_list, y_val, init=None, step=0.05, max_pass=4):
    """Deterministic coordinate ascent (suitable for up to five members)."""
    K = len(P_list)
    if init is None:
        w = np.ones(K, dtype=float) / K
    else:
        w = np.array(init, dtype=float); w /= w.sum()
    best = -1.0
    
    Pv = soft_mix(P_list, w)
    base = f1_score(y_val, Pv.argmax(1), average="macro", zero_division=0)
    best = base
    improved = True; passes=0
    while improved and passes < max_pass:
        improved = False; passes += 1
        for i in range(K):
            
            for delta in (+step, -step):
                w_new = w.copy()
                w_new[i] = np.clip(w_new[i] + delta, 0.0, 1.0)
                # Normalize the weights to sum to one.
                if w_new.sum() == 0: continue
                w_new /= w_new.sum()
                Pv = soft_mix(P_list, w_new)
                mac = f1_score(y_val, Pv.argmax(1), average="macro", zero_division=0)
                if mac > best + 1e-8:
                    best = mac; w = w_new; improved = True
        
    return w


def run_experiment(variant_name, members_spec, dynamic=False, try_hard_soft_avg=False,
                   fixed_weights=None):
    """
    members_spec: dict(feature -> [tokens]), e.g.:
      {"SWIN": ["MLP(SwinMLP)"], "EVA":["MLP(EvaMLP)"], "COAT":["LGBM"]}
    dynamic: optimize soft-voting weights on validation (variants B and D)
    try_hard_soft_avg: compare hard, soft, and average voting for variant C
    fixed_weights: dict("feat|tok" -> weight) for variant A
    """
    print("\n" + "="*95)
    print(f"[STAGE-3 / {variant_name}] RUNNING...")
    print("="*95)

    
    splits, class_names = load_global_splits(SPLIT_DIR)
    n_class = len(class_names)

    # Load feature arrays
    data = {}
    for feat, path in FEATURE_DIRS.items():
        Xtr = _load_features(path, "train"); Xva = _load_features(path, "val"); Xte = _load_features(path, "test")
        ids_tr = splits["train"]["ids"]; y_tr = splits["train"]["y"]
        ids_va = splits["val"]["ids"];   y_va = splits["val"]["y"]
        ids_te = splits["test"]["ids"];  y_te = splits["test"]["y"]
        assert len(Xtr)==len(y_tr)==len(ids_tr)
        assert len(Xva)==len(y_va)==len(ids_va)
        assert len(Xte)==len(y_te)==len(ids_te)
        data[feat] = {"Xtr":Xtr,"ytr":y_tr,"ids_tr":ids_tr,
                      "Xva":Xva,"yva":y_va,"ids_va":ids_va,
                      "Xte":Xte,"yte":y_te,"ids_te":ids_te}

    out_root = os.path.join(SAVE_DIR, variant_name)
    os.makedirs(out_root, exist_ok=True)

    
    members = []
    for feat, model_list in members_spec.items():
        Xtr,ytr = data[feat]["Xtr"], data[feat]["ytr"]
        Xva,yva = data[feat]["Xva"], data[feat]["yva"]
        Xte,yte = data[feat]["Xte"], data[feat]["yte"]

        # VAL: Train-only PCA
        sc1, pca1, Ztr, Zva = pca_safe_fit_transform(Xtr, Xva, PCA_VARIANCE)
        # TEST: Train∪Val PCA
        sc2, pca2, Ztr2, Zva2, Zte2 = pca_safe_fit_transform_test(Xtr, Xva, Xte, PCA_VARIANCE)

        for tok in model_list:
            mk, preset = parse_model_token(tok)
            print(f"--- TRAIN/VAL member: {feat} | {mk if preset is None else f'{mk}({preset})'} ---")
            P_va, P_te = fit_val_and_test(mk, preset, Ztr, ytr, Zva, yva,
                                          Ztr2, Zva2, Zte2, yte,
                                          n_class, out_root, feat, tok, class_names)
            if P_va is None: continue
            members.append({"key": f"{feat}|{tok}", "feat": feat, "model": tok,
                            "P_val": P_va, "P_test": P_te})
    assert len(members) > 0, "No ensemble members were created."

    y_val = data["SWIN"]["yva"]; y_test = data["SWIN"]["yte"]  
    member_keys = [m["key"] for m in members]
    Pval_list   = [m["P_val"]  for m in members]
    Ptest_list  = [m["P_test"] for m in members]

    
    def eval_and_dump(tag, P_final, y_true, split_tag):
        Topt = pick_T_by_val_nll(P_final if split_tag=="val" else P_val_final,  
                                 y_val)
        P_cal = torch.softmax(torch.log(torch.tensor(P_final+1e-12))/Topt, dim=1).numpy()
        out_dir = os.path.join(out_root, "ENSEMBLE", tag.upper(), split_tag.upper())
        print_and_save(split_tag, y_true, P_cal, out_dir, class_names, print_console=True)
        return Topt, P_cal

    
    if fixed_weights is not None:
        
        w = np.array([fixed_weights.get(k, 0.0) for k in member_keys], dtype=float)
        if w.sum() == 0: w = np.ones_like(w) / len(w)
        w = w / w.sum()
        print("[A] Fixed weights:", dict(zip(member_keys, np.round(w,3))))
        P_val_final  = soft_mix(Pval_list,  w)
        P_test_final = soft_mix(Ptest_list, w)

        
        print("\n[ENSEMBLE-A | SOFT (fixed w)]")
        T_val, _ = eval_and_dump("soft_fixed", P_val_final,  y_val,  "val")
        T_te,  _ = eval_and_dump("soft_fixed", P_test_final, y_test, "test")

        results = {"weights": dict(zip(member_keys, [float(x) for x in w])),
                   "T_from_VAL": float(T_val)}
        with open(os.path.join(out_root, "ENSEMBLE", "A_soft_fixed_meta.json"), "w") as f:
            json.dump(results, f, indent=2)
        return {"member_keys": member_keys, "weights": w, "T": T_val}

    
    P_val_final = None
    P_test_final = None

    if try_hard_soft_avg:
        
        print("\n[C] Starting the voting-rule comparison...")
        # --- HARD
        yv_hard = hard_vote(Pval_list)
        out_dir = os.path.join(out_root, "ENSEMBLE", "HARD")
        print_cm_cr("val", y_val, yv_hard, os.path.join(out_dir,"VAL"), class_names, True)
        yt_hard = hard_vote(Ptest_list)
        print_cm_cr("test", y_test, yt_hard, os.path.join(out_dir,"TEST"), class_names, True)

        
        if len(Pval_list) == 3:
            
            best_mac, best_w = -1.0, None
            for a,b,c in grid_weights_step_05_for_3():
                Pv = soft_mix(Pval_list, [a,b,c])
                mac = f1_score(y_val, Pv.argmax(1), average="macro", zero_division=0)
                if mac > best_mac:
                    best_mac, best_w = mac, np.array([a,b,c])
            w_soft = best_w
        else:
            
            w_soft = coord_ascent_weights(Pval_list, y_val, init=None, step=0.05, max_pass=4)
        print("[C|SOFT] Best weights:", dict(zip(member_keys, np.round(w_soft,3))))
        P_val_soft  = soft_mix(Pval_list,  w_soft)
        P_test_soft = soft_mix(Ptest_list, w_soft)
        T_val = pick_T_by_val_nll(P_val_soft, y_val)
        P_val_soft_cal  = torch.softmax(torch.log(torch.tensor(P_val_soft+1e-12))/T_val, dim=1).numpy()
        P_test_soft_cal = torch.softmax(torch.log(torch.tensor(P_test_soft+1e-12))/T_val, dim=1).numpy()
        print_cm_cr("val",  y_val,  P_val_soft_cal.argmax(1),  os.path.join(out_root,"ENSEMBLE","SOFT","VAL"),  class_names, True)
        print_cm_cr("test", y_test, P_test_soft_cal.argmax(1), os.path.join(out_root,"ENSEMBLE","SOFT","TEST"), class_names, True)

        
        w_avg = np.ones(len(Pval_list), dtype=float) / len(Pval_list)
        P_val_avg  = soft_mix(Pval_list,  w_avg)
        P_test_avg = soft_mix(Ptest_list, w_avg)
        T_val2 = pick_T_by_val_nll(P_val_avg, y_val)
        P_val_avg_cal  = torch.softmax(torch.log(torch.tensor(P_val_avg+1e-12))/T_val2, dim=1).numpy()
        P_test_avg_cal = torch.softmax(torch.log(torch.tensor(P_test_avg+1e-12))/T_val2, dim=1).numpy()
        print_cm_cr("val",  y_val,  P_val_avg_cal.argmax(1),  os.path.join(out_root,"ENSEMBLE","AVG","VAL"),  class_names, True)
        print_cm_cr("test", y_test, P_test_avg_cal.argmax(1), os.path.join(out_root,"ENSEMBLE","AVG","TEST"), class_names, True)

        return {"member_keys": member_keys, "w_soft": w_soft, "T_soft": T_val}

    else:
        
        if len(Pval_list) == 3:
            best_mac, best_w = -1.0, None
            for a,b,c in grid_weights_step_05_for_3():
                Pv = soft_mix(Pval_list, [a,b,c])
                mac = f1_score(y_val, Pv.argmax(1), average="macro", zero_division=0)
                if mac > best_mac:
                    best_mac, best_w = mac, np.array([a,b,c])
            w = best_w
        else:
            w = coord_ascent_weights(Pval_list, y_val, init=None, step=0.05, max_pass=5)
        print(f"[{variant_name}] Best weights:", dict(zip(member_keys, np.round(w,3))))
        P_val_final  = soft_mix(Pval_list,  w)
        P_test_final = soft_mix(Ptest_list, w)

        
        print(f"\n[ENSEMBLE-{variant_name} | SOFT (dynamic w)]")
        T_val, _ = eval_and_dump("soft_dynamic", P_val_final,  y_val,  "val")
        T_te,  _ = eval_and_dump("soft_dynamic", P_test_final, y_test, "test")
        results = {"weights": dict(zip(member_keys, [float(x) for x in w])),
                   "T_from_VAL": float(T_val)}
        with open(os.path.join(out_root, "ENSEMBLE", f"{variant_name}_soft_dynamic_meta.json"), "w") as f:
            json.dump(results, f, indent=2)
        return {"member_keys": member_keys, "weights": w, "T": T_val}

# -------------------- MAIN --------------------
def main():
    
    A_spec = {
        "SWIN": ["MLP(SwinMLP)"],
        "EVA" : ["MLP(EvaMLP)"],
        "COAT": ["LGBM"]
    }
    top_A = run_experiment("A", A_spec, dynamic=False, try_hard_soft_avg=False,
                           fixed_weights=GLOBAL_FIXED_WEIGHTS_A)

    
    top_B = run_experiment("B", A_spec, dynamic=True, try_hard_soft_avg=False)

    
    top_C = run_experiment("C", A_spec, dynamic=True, try_hard_soft_avg=True)

    
    D_spec = {
        "SWIN": ["MLP(SwinMLP)", "MLP(SwinRobust)"],
        "EVA" : ["MLP(EvaMLP)", "MLP(EvaRobust)"],
        "COAT": ["LGBM", "LogReg"]
    }
    top_D = run_experiment("D", D_spec, dynamic=True, try_hard_soft_avg=False)

    
    print("\n" + "="*80)
    print("Stage 3 completed. A/B/C/D results were saved under SAVE_DIR:")
    print(" -", SAVE_DIR)
    print("="*80)

if __name__ == "__main__":
    main()

## Section A — Validation-tuned ensemble weights with metadata options

**Research question:** For the configured metadata options (region, age, and sex), which three-member weight combination maximizes validation Macro-F1?

**Selection rule:** the weight grid is evaluated against validation predictions only. The winning weights are frozen; the ensemble temperature is then selected on validation NLL. Only after both decisions are fixed are these settings applied to the test predictions.

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================




# ============================================================

import os, sys, json, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# User configuration
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
SPLIT_DIR = str(PROJECT_ROOT / "data" / "splits")  # train_split.csv / val_split.csv / test_split.csv
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin_ft768"),
    "EVA": str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet_ft672"),
}
META_CSV_PATH = str(PROJECT_ROOT / "data" / "metadata" / "ISIC_2019_Training_Metadata.csv")

SAVE_DIR = str(PROJECT_ROOT / "outputs" / "ensemble_meta_runs_A_dynamic")
RANDOM_STATE = 42
DEVICE = "auto"  # "cuda" | "cpu" | "auto"
PCA_VARIANCE = 0.99
CALIBRATION_BINS = 15


COAT_BASE = "LGBM"   # "LogReg" or "LGBM"

# Metadata feature flags
META_FLAGS = {
    "INCLUDE_AGE_2BIT"     : False,
    "AGE_THRESHOLD"        : 30,
    "INCLUDE_REGION_9BIT"  : False,
    "INCLUDE_GROUP_SCHEMA" : False,
    "GROUP_SCHEMA"         : "B",
    "INCLUDE_SEX_2BIT"     : False,
}

# Selected model members
MODELS_PER_FEATURE = {
    "SWIN": ["MLP(SwinMLP)"],
    "EVA" : ["MLP(EvaMLP)"],
    "COAT": [COAT_BASE]   # LogReg or LGBM
}

# ---- MLP presets ----
NAMED_PRESETS = {
    "SwinMLP": {
        "hidden": [2048], "dropout": 0.1605757337,
        "lr": 0.000341409509895, "weight_decay": 0.0001488348195284,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
    "EvaMLP": {
        "hidden": [2048], "dropout": 0.3203382207442766,
        "lr": 0.0009765455593461, "weight_decay": 8.212522451000109e-05,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
}

# Additional model hyperparameters
HP_DEFAULTS = {
    "LogReg": {"C": 2.3402196063650043, "max_iter": 600, "tol": 3.6e-04,
               "solver": "lbfgs", "class_weight": None},
    "LGBM"  : {"n_estimators": 600, "num_leaves": 48, "learning_rate": 0.05,
               "subsample": 0.9, "colsample_bytree": 0.7, "reg_lambda": 1.0,
               "random_state": RANDOM_STATE, "verbosity": -1},
}

# Temperature grid; select calibration temperature using validation NLL.
TEMP_GRID = [0.70,0.75,0.80,0.85,0.90,0.95,1.00,1.05,1.10,1.15,1.20,1.25,1.30]

# ---- Determinism / CUDA fix ----
os.environ["CUBLAS_WORKSPACE_CONFIG"] = os.environ.get("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

# -------------------- Torch --------------------
def _pip(*pkgs):
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
        return True
    except Exception as e:
        print(f"[WARN] pip install failed for {pkgs}: {e}")
        return False

try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn
except Exception:
    _pip("torch","torchvision","torchaudio")
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn

# -------------------- Sklearn vs. --------------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, log_loss

# Optional XGBoost / LightGBM models
def _lazy_import_lgbm():
    try:
        import lightgbm as lgb
    except Exception:
        ok = _pip("lightgbm")
        try:
            import lightgbm as lgb
        except Exception:
            lgb = None
    return lgb

# -------------------- Util --------------------
def set_seed(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = True
set_seed(RANDOM_STATE)

def choose_device():
    if DEVICE == "cuda": return "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":  return "cpu"
    return "cuda" if torch.cuda.is_available() else "cpu"
DEV = choose_device()
print("Device:", DEV); os.makedirs(SAVE_DIR, exist_ok=True)

def _glob_any(p: Path, names):
    for nm in names:
        f = p / nm
        if f.exists(): return str(f)
    return None

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True, flags=0)
    return s.values

def _read_global_split_csv(split_csv_path):
    df = pd.read_csv(split_csv_path)
    id_col = None
    for cand in ["image_id","image","Image","filename","fname","img_id","name"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None: raise ValueError(f"No image_id/image column found in {split_csv_path}.")
    ids = _normalize_ids(df[id_col])

    if "label" in df.columns:
        y = df["label"].astype(int).values
    elif "class_name" in df.columns:
        classes = sorted(df["class_name"].unique().tolist())
        mp = {c:i for i,c in enumerate(classes)}
        y = df["class_name"].map(mp).astype(int).values
    else:
        raise ValueError(f"No label/class_name column found in {split_csv_path}.")
    return ids, y, df

def load_global_splits(split_dir):
    paths = {
        "train": os.path.join(split_dir, "train_split.csv"),
        "val"  : os.path.join(split_dir, "val_split.csv"),
        "test" : os.path.join(split_dir, "test_split.csv"),
    }
    out = {}
    for sp, pth in paths.items():
        ids, y, df = _read_global_split_csv(pth)
        out[sp] = {"ids": ids, "y": y, "df": df}
    df_tr = out["train"]["df"]
    if "class_name" in df_tr.columns:
        class_names = sorted(df_tr["class_name"].unique().tolist())
    else:
        class_names = [f"class{i}" for i in sorted(pd.Series(out["train"]["y"]).unique())]
    return out, class_names

def _load_features(dir_path: str, split: str):
    p = Path(dir_path)
    X_path = _glob_any(p, [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"])
    if X_path is None: raise FileNotFoundError(f"No {split} feature array found in {dir_path}.")
    return np.load(X_path)

def brier_score(y_true, proba):
    C = proba.shape[1]; Y = np.eye(C)[y_true]
    return float(np.mean(np.sum((proba - Y)**2, axis=1)))

def ece_score(y_true, proba, n_bins=CALIBRATION_BINS):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); accs = (pred == y_true).astype(float)
    bins = np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        if m.sum(): ece += m.mean() * abs(accs[m].mean() - conf[m].mean())
    return float(ece)

def topk_capture(y_true, proba, k=3):
    topk = np.argpartition(-proba, kth=k-1, axis=1)[:, :k]
    return float(np.any(topk == y_true[:, None], axis=1).mean())

def reliability_bins(y_true, proba, n_bins=CALIBRATION_BINS):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); correct = (pred == y_true).astype(int)
    bins = np.linspace(0,1,n_bins+1); rows=[]
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        rows.append({"bin": i, "count": int(m.sum()),
                     "avg_conf": float(np.nan if m.sum()==0 else conf[m].mean()),
                     "acc": float(np.nan if m.sum()==0 else correct[m].mean()),
                     "gap": float(np.nan if m.sum()==0 else abs(correct[m].mean()-conf[m].mean()))})
    return pd.DataFrame(rows)

def save_reports(prefix, y_true, proba, out_dir, class_names, print_console=True):
    os.makedirs(out_dir, exist_ok=True)
    y_pred = proba.argmax(1)
    row = {"split":prefix,
           "Acc": accuracy_score(y_true, y_pred),
           "MacroF1": f1_score(y_true, y_pred, average="macro", zero_division=0),
           "Brier": brier_score(y_true, proba),
           "ECE": ece_score(y_true, proba),
           "Top3": topk_capture(y_true, proba, 3),
           "N": len(y_true), "classes": proba.shape[1]}
    pd.DataFrame([row]).to_csv(os.path.join(out_dir, f"{prefix}_analysis_report.csv"), index=False)
    reliability_bins(y_true, proba).to_csv(os.path.join(out_dir, f"{prefix}_reliability_bins.csv"), index=False)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in class_names],
                         columns=[f"pred_{c}" for c in class_names])
    cm_df.to_csv(os.path.join(out_dir, f"{prefix}_confusion_matrix.csv"))
    cr = classification_report(y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True)
    cr_df = pd.DataFrame(cr); cr_df.to_csv(os.path.join(out_dir, f"{prefix}_classification_report.csv"))

    if print_console:
        print(f"\n[{out_dir}] {prefix.upper()} — Acc={row['Acc']:.4f} | MacroF1={row['MacroF1']:.4f} | "
              f"Brier={row['Brier']:.4f} | ECE={row['ECE']:.4f} | Top3={row['Top3']:.4f}")
        print("\nCM:\n", cm_df.to_string())
        print("\nCR:\n", cr_df.to_string())
    return row

# -------------------- Meta Engineering --------------------
def read_meta_local(csv_path):
    df = pd.read_csv(csv_path)
    id_col = None
    for cand in ["image","image_id","Image","filename","fname","name","img_id"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None:
        raise ValueError("The metadata CSV must contain an 'image' or 'image_id' column.")
    df["image_id"] = _normalize_ids(df[id_col])
    for c in ["anatom_site_general","age_approx","sex"]:
        if c not in df.columns:
            df[c] = np.nan
    return df

REGION9_LIST = [
    "head/neck","torso_anterior","torso_posterior","torso_lateral",
    "upper_extremity","lower_extremity","acral","mucosal","other_or_missing"
]
def map_region9(anat: str):
    if pd.isna(anat): return "other_or_missing"
    s = str(anat).strip().lower()
    if "head" in s or "neck" in s: return "head/neck"
    if "torso" in s and "anterior" in s: return "torso_anterior"
    if "torso" in s and "posterior" in s: return "torso_posterior"
    if "torso" in s and "lateral" in s: return "torso_lateral"
    if "upper" in s: return "upper_extremity"
    if "lower" in s: return "lower_extremity"
    if "palms" in s or "soles" in s or "acral" in s: return "acral"
    if "oral" in s or "genital" in s or "mucosal" in s: return "mucosal"
    return "other_or_missing"

def map_group(schema: str, anat: str):
    if pd.isna(anat): base = "missing"
    else: base = str(anat).strip().lower()

    headneck = ("head" in base or "neck" in base)
    upper    = ("upper" in base)
    lower    = ("lower" in base)
    torso    = ("torso" in base)
    lateral  = ("lateral" in base and "torso" in base)
    acral    = ("palms" in base or "soles" in base or "acral" in base)
    mucosal  = ("oral" in base or "genital" in base or "mucosal" in base)

    if schema.upper()=="B":
        if headneck: return "head/neck"
        if upper:    return "upper_extremity"
        if lower:    return "lower_extremity"
        if torso and not lateral: return "torso"
        if lateral:  return "lateral_torso"
        if acral:    return "acral"
        if mucosal:  return "mucosal"
        return "missing"
    elif schema.upper()=="A":
        if headneck: return "head/neck"
        if upper:    return "upper_extremity"
        if lower:    return "lower_extremity"
        if torso:    return "torso"
        if acral or mucosal: return "special"
        return "missing"
    else:
        if headneck: return "head/neck"
        if upper or lower: return "extremity"
        if torso:    return "torso"
        if acral or mucosal: return "special"
        return "other"

GROUP_CATS = {
    "A": ["head/neck","upper_extremity","lower_extremity","torso","special","missing"],
    "B": ["head/neck","upper_extremity","lower_extremity","torso","lateral_torso","acral","mucosal","missing"],
    "C": ["head/neck","extremity","torso","special","other"],
}

def build_meta_matrix(meta_df, ids, flags):
    df = pd.DataFrame({"image_id": ids}).merge(meta_df, on="image_id", how="left")
    cols = []

    mats = []

    if flags["INCLUDE_AGE_2BIT"]:
        thr = float(flags["AGE_THRESHOLD"])
        age = pd.to_numeric(df["age_approx"], errors="coerce")
        mats += [(age < thr).astype(float).fillna(0.0).values.reshape(-1,1),
                 (age >= thr).astype(float).fillna(0.0).values.reshape(-1,1)]
        cols += ["age_lt","age_ge"]

    if flags["INCLUDE_REGION_9BIT"]:
        r = df["anatom_site_general"].apply(map_region9)
        for cat in REGION9_LIST:
            mats.append((r == cat).astype(float).values.reshape(-1,1))
        cols += [f"reg9_{c}" for c in REGION9_LIST]

    if flags["INCLUDE_GROUP_SCHEMA"]:
        schema = flags["GROUP_SCHEMA"].upper()
        r = df["anatom_site_general"].apply(lambda s: map_group(schema, s))
        cats = GROUP_CATS[schema]
        for cat in cats:
            mats.append((r == cat).astype(float).values.reshape(-1,1))
        cols += [f"group_{schema}_{c}" for c in cats]

    if flags["INCLUDE_SEX_2BIT"]:
        sx = df["sex"].astype(str).str.lower()
        mats += [(sx == "male").astype(float).values.reshape(-1,1),
                 (sx == "female").astype(float).values.reshape(-1,1)]
        cols += ["sex_male","sex_female"]

    if mats:
        M = np.concatenate(mats, axis=1).astype(np.float32)
    else:
        M = np.zeros((len(df),0), dtype=np.float32)
    return M, cols

# -------------------- MLP --------------------
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(1024,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def _make_loader(X, y, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=0, drop_last=False)

def train_mlp_fixed_epochs(Xfit, yfit, n_classes, hp, device, log_tag):
    loader = _make_loader(Xfit, yfit, hp["batch_size"], shuffle=True)
    model = RobustMLP(Xfit.shape[1], n_classes, tuple(hp["hidden"]), hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
    model.train()
    for ep in range(1, hp["epochs"]+1):
        for xb,yb in loader:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss=crit(model(xb), yb); loss.backward(); opt.step()
        if ep % max(1,hp.get("log_every",5))==0 or ep==1:
            model.eval()
            with torch.no_grad():
                lg = model(torch.tensor(Xfit, dtype=torch.float32, device=device)).cpu().numpy()
            proba = torch.softmax(torch.tensor(lg), dim=1).numpy()
            y_hat = proba.argmax(1)
            acc = accuracy_score(yfit, y_hat); f1 = f1_score(yfit, y_hat, average="macro", zero_division=0)
            print(f"[MLP] ep {ep:03d} | {log_tag} Acc {acc:.4f} | MacroF1 {f1:.4f}")
            model.train()
    return model

def logits_and_proba(model, X, device):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X), dtype=torch.long))
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=0, drop_last=False)
    model.eval(); out=[]
    with torch.no_grad():
        for (xb,_) in loader:
            xb=xb.to(device); out.append(model(xb).cpu())
    lg = torch.cat(out,0).numpy()
    P  = torch.softmax(torch.tensor(lg), dim=1).numpy()
    return lg, P

def apply_temp_logits(logits, T):
    return torch.softmax(torch.tensor(logits)/T, dim=1).numpy()

# Fit/transform helpers
def fit_transform_train_then_val(Xtr, Xva, pca_var):
    scaler = StandardScaler().fit(Xtr)
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva)
    if pca_var is None: return scaler, None, Ztr, Zva
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva)

def fit_transform_trainval_then_test(Xtr, Xva, Xte, pca_var):
    scaler = StandardScaler().fit(np.vstack([Xtr, Xva]))
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva); Zte = scaler.transform(Xte)
    if pca_var is None: return scaler, None, Ztr, Zva, Zte
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(np.vstack([Ztr,Zva]))
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)

# Member-model training
def parse_model_token(tok: str):
    if tok.startswith("MLP(") and tok.endswith(")"):
        name = tok[4:-1].strip()
        return "MLP", name
    return tok, None

def build_and_fit_val_predict(model_key, preset_name, Ztr, ytr, Zva, n_class):
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        model = train_mlp_fixed_epochs(Ztr, ytr, n_class, hp, DEV, "TRAIN")
        lg_va, _ = logits_and_proba(model, Zva, DEV)
        P_va = apply_temp_logits(lg_va, hp["T"])
        return ("MLP", hp, model, P_va)
    elif model_key=="LGBM":
        lgb = _lazy_import_lgbm()
        if lgb is None:
            print("[WARN] LightGBM is unavailable; skipping this model.")
            return ("LGBM", None, None, None)
        hp = HP_DEFAULTS["LGBM"].copy()
        clf = lgb.LGBMClassifier(**hp)
        clf.fit(Ztr, ytr)
        P_va = clf.predict_proba(Zva)
        return ("LGBM", hp, clf, P_va)
    elif model_key=="LogReg":
        hp = HP_DEFAULTS["LogReg"].copy()
        clf = LogisticRegression(**hp, multi_class="multinomial", n_jobs=None)
        clf.fit(Ztr, ytr)
        P_va = clf.predict_proba(Zva)
        return ("LogReg", hp, clf, P_va)
    else:
        raise ValueError(f"Unknown model: {model_key}")

def refit_and_predict_test(model_key, preset_name, trained, Xfit, yfit, Zte, n_class):
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        model = train_mlp_fixed_epochs(Xfit, yfit, n_class, hp, DEV, "TRAIN∪VAL")
        lg_te, _ = logits_and_proba(model, Zte, DEV)
        P_te = apply_temp_logits(lg_te, hp["T"])
        return P_te
    elif model_key=="LGBM":
        lgb = _lazy_import_lgbm()
        if lgb is None:
            print("[WARN] LightGBM is unavailable; skipping this test evaluation.")
            return None
        hp = HP_DEFAULTS["LGBM"].copy()
        clf = lgb.LGBMClassifier(**hp); clf.fit(Xfit, yfit)
        return clf.predict_proba(Zte)
    elif model_key=="LogReg":
        hp = HP_DEFAULTS["LogReg"].copy()
        clf = LogisticRegression(**hp, multi_class="multinomial", n_jobs=None)
        clf.fit(Xfit, yfit)
        return clf.predict_proba(Zte)
    else:
        raise ValueError(f"Unknown model: {model_key}")

# -------------------- Ensemble helpers --------------------
def soft_mean(P_list, weights=None):
    P_list = [P for P in P_list if P is not None]
    if not P_list: return None
    if weights is None: weights = np.ones(len(P_list), dtype=float)
    else: weights = np.array(weights[:len(P_list)], dtype=float)
    S = np.zeros_like(P_list[0], dtype=np.float64)
    for P, w in zip(P_list, weights): S += w * P
    S = S / max(1e-12, weights.sum())
    return S

def hard_preds_from_proba(P_list):
    return [P.argmax(1) for P in P_list if P is not None]


def weight_grid_search_3(Pv_list, y_val, step=0.05, metric="macroF1"):
    """
    w1,w2 in {0,step,2*step,...,1}; w3 = 1-w1-w2; w3>=0.
    Validation Macro-F1 is maximized by default.
    """
    assert len(Pv_list) == 3, "This grid is intended for three members."
    best = {"w": np.array([1/3,1/3,1/3],dtype=float), "score": -1.0}
    grid = np.arange(0, 1+1e-9, step)
    for w1 in grid:
        for w2 in grid:
            w3 = 1.0 - w1 - w2
            if w3 < -1e-9: continue
            w = np.array([w1,w2,max(0.0,w3)], dtype=float)
            if w.sum() <= 0: continue
            P = soft_mean(Pv_list, w)
            yhat = P.argmax(1)
            if metric=="macroF1":
                sc = f1_score(y_val, yhat, average="macro", zero_division=0)
            else:
                sc = accuracy_score(y_val, yhat)
            if sc > best["score"]:
                best = {"w": w / w.sum(), "score": float(sc)}
    return best

# Main workflow
def main():
    set_seed(RANDOM_STATE)
    root = SAVE_DIR; os.makedirs(root, exist_ok=True)

    
    splits, class_names = load_global_splits(SPLIT_DIR)
    n_class = len(class_names)
    print(f"[Global splits] classes={class_names}")

    # Load feature arrays
    data = {}
    for feat, path in FEATURE_DIRS.items():
        Xtr = _load_features(path, "train"); Xva = _load_features(path, "val"); Xte = _load_features(path, "test")
        ids_tr = splits["train"]["ids"]; y_tr = splits["train"]["y"]
        ids_va = splits["val"]["ids"];   y_va = splits["val"]["y"]
        ids_te = splits["test"]["ids"];  y_te = splits["test"]["y"]
        assert len(Xtr)==len(y_tr)==len(ids_tr)
        assert len(Xva)==len(y_va)==len(ids_va)
        assert len(Xte)==len(y_te)==len(ids_te)
        data[feat] = {"Xtr":Xtr,"ytr":y_tr,"ids_tr":ids_tr,
                      "Xva":Xva,"yva":y_va,"ids_va":ids_va,
                      "Xte":Xte,"yte":y_te,"ids_te":ids_te}

    
    meta_all = read_meta_local(META_CSV_PATH)
    M_tr, cols_tr = build_meta_matrix(meta_all, data["SWIN"]["ids_tr"], META_FLAGS)
    M_va, cols_va = build_meta_matrix(meta_all, data["SWIN"]["ids_va"], META_FLAGS)
    M_te, cols_te = build_meta_matrix(meta_all, data["SWIN"]["ids_te"], META_FLAGS)
    assert cols_tr==cols_va==cols_te
    print(f"[META] columns: {len(cols_tr)} -> {cols_tr if len(cols_tr)<=10 else str(cols_tr[:10])+' ...'}")

    # 4) Append metadata features to the feature matrices
    if M_tr.shape[1] > 0:
        for k in data.keys():
            data[k]["Xtr"] = np.concatenate([data[k]["Xtr"], M_tr], axis=1).astype(np.float32)
            data[k]["Xva"] = np.concatenate([data[k]["Xva"], M_va], axis=1).astype(np.float32)
            data[k]["Xte"] = np.concatenate([data[k]["Xte"], M_te], axis=1).astype(np.float32)

    
    members = []  # dict: key, feat, model, P_val, P_te, y_val, y_te
    for feat, model_list in MODELS_PER_FEATURE.items():
        print(f"\n===== {feat}: {len(model_list)} model =====")
        Xtr,ytr,Xva,yva,Xte,yte = data[feat]["Xtr"],data[feat]["ytr"],data[feat]["Xva"],data[feat]["yva"],data[feat]["Xte"],data[feat]["yte"]

        
        sc1, pca1, Ztr, Zva = fit_transform_train_then_val(Xtr, Xva, PCA_VARIANCE)

        for tok in model_list:
            model_key, preset = parse_model_token(tok)
            print(f"[{feat}] model = {tok}")
            mdl_name, hp, trained, P_va = build_and_fit_val_predict(model_key, preset, Ztr, ytr, Zva, n_class)
            if P_va is None:
                print(f"[WARN] {feat}|{tok}: validation probabilities could not be generated; skipping this model.")
                continue

            out_dir = os.path.join(SAVE_DIR, feat, tok.replace("/","_"), "VAL")
            save_reports("val", yva, P_va, out_dir, class_names, print_console=True)

            
            sc2, pca2, Ztr2, Zva2, Zte2 = fit_transform_trainval_then_test(Xtr, Xva, Xte, PCA_VARIANCE)
            Xfit = np.vstack([Ztr2, Zva2]); yfit = np.concatenate([ytr, yva])

            P_te = refit_and_predict_test(model_key, preset, trained, Xfit, yfit, Zte2, n_class)
            if P_te is None:
                print(f"[WARN] {feat}|{tok}: test probabilities could not be generated; skipping this model.")
                continue
            out_dir_te = os.path.join(SAVE_DIR, feat, tok.replace("/","_"), "TEST")
            save_reports("test", yte, P_te, out_dir_te, class_names, print_console=True)
            np.save(os.path.join(out_dir_te, "test_proba.npy"), P_te)

            members.append({"key": f"{feat}|{tok}", "feat": feat, "model": tok,
                            "P_val": P_va, "P_te": P_te, "y_val": yva, "y_te": yte})

    if len(members) != 3:
        raise RuntimeError(f"Scenario A requires exactly three ensemble members, but {len(members)} were created. Check MODELS_PER_FEATURE.")

    
    y_val = members[0]["y_val"]; y_te = members[0]["y_te"]
    for m in members:
        assert np.array_equal(m["y_val"], y_val) and np.array_equal(m["y_te"], y_te)

    P_val_members = [m["P_val"] for m in members]
    P_te_members  = [m["P_te"]  for m in members]
    member_keys   = [m["key"]   for m in members]

    print("\n[ENSEMBLE-A | SOFT (dynamic w) — Grid step=0.05, criterion=VAL Macro-F1]")
    best = weight_grid_search_3(P_val_members, y_val, step=0.05, metric="macroF1")
    w_best = best["w"]; score = best["score"]
    w_dict = dict(zip(member_keys, [float(x) for x in w_best]))
    print(f"[A] Best weights: {w_dict} | VAL Macro-F1={score:.4f}")

    # --- Best w ile proba'lar ---
    P_val_final = soft_mean(P_val_members, w_best)
    P_te_final  = soft_mean(P_te_members,  w_best)

    
    bestT, bestNLL = 1.0, float("inf")
    for T in TEMP_GRID:
        Pv = torch.softmax(torch.log(torch.tensor(P_val_final+1e-12))/T, dim=1).numpy()
        nll = log_loss(y_val, Pv, labels=list(range(len(class_names))))
        if nll < bestNLL: bestNLL, bestT = nll, T
    print(f"[A] Selected T={bestT:.2f} (VAL-NLL={bestNLL:.6f})")

    P_test = torch.softmax(torch.log(torch.tensor(P_te_final+1e-12))/bestT, dim=1).numpy()

    out_dir = os.path.join(SAVE_DIR, "ENSEMBLE_A_DYNAMIC"); os.makedirs(out_dir, exist_ok=True)
    save_reports("test", y_te, P_test, out_dir, class_names, print_console=True)
    np.save(os.path.join(out_dir, "test_proba.npy"), P_test)
    with open(os.path.join(out_dir, "members.json"), "w") as f:
        json.dump(member_keys, f, indent=2)
    with open(os.path.join(out_dir, "ensemble_meta.json"), "w") as f:
        json.dump({"strategy":"A_DYNAMIC",
                   "weights": w_dict,
                   "T": float(bestT)}, f, indent=2)

    print(f"\nDone. Strategy=A_DYNAMIC | Weights={w_dict} | T={bestT:.2f} | Outputs → {out_dir}")

if __name__ == "__main__":
    main()

## Section A — Fixed global weights and validation-selected temperature

**Research question:** What does the pre-specified fixed-weight ensemble achieve when its ensemble temperature is calibrated on validation NLL?

**Selection rule:** global weights are fixed configuration values, not estimated from test metrics; only the temperature is selected on validation. Treat this cell as a pre-specified reference, not as a model-selection step.

In [ ]:

# -*- coding: utf-8 -*-
# ============================================================




#  - Reports: Accuracy/Macro-F1/Brier/ECE/Top-3 + confusion matrix/classification report CSV files
# ============================================================


import os, sys, json, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# User configuration
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
SPLIT_DIR = str(PROJECT_ROOT / "data" / "splits")  # train_split.csv / val_split.csv / test_split.csv
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin_ft768"),
    "EVA": str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet_ft672"),
}
META_CSV_PATH = str(PROJECT_ROOT / "data" / "metadata" / "ISIC_2019_Training_Metadata.csv")

SAVE_DIR = str(PROJECT_ROOT / "outputs" / "ensemble_meta_runs_FIXED")
RANDOM_STATE = 42
DEVICE = "auto"  # "cuda" | "cpu" | "auto"
PCA_VARIANCE = 0.99
CALIBRATION_BINS = 15

# Metadata feature flags
META_FLAGS = {
    "INCLUDE_AGE_2BIT"     : True,
    "AGE_THRESHOLD"        : 30,
    "INCLUDE_REGION_9BIT"  : False,
    "INCLUDE_GROUP_SCHEMA" : True,
    "GROUP_SCHEMA"         : "B",
    "INCLUDE_SEX_2BIT"     : False,
}


COAT_MODEL = "LogReg"   # "LGBM" or "LogReg"


MODELS_PER_FEATURE = {
    "SWIN": ["MLP(SwinMLP)"],
    "EVA" : ["MLP(EvaMLP)"],
    "COAT": [COAT_MODEL],
}

# ---- MLP presets ----
NAMED_PRESETS = {
    "SwinMLP": {
        "hidden": [2048], "dropout": 0.1605757337,
        "lr": 0.000341409509895, "weight_decay": 0.0001488348195284,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
    "EvaMLP": {
        "hidden": [2048], "dropout": 0.3203382207442766,
        "lr": 0.0009765455593461, "weight_decay": 8.212522451000109e-05,
        "batch_size": 128, "epochs": 50, "T": 0.9, "log_every": 5
    },
}

# Additional model hyperparameters
HP_DEFAULTS = {
    "LogReg": {"C": 2.3402196063650043, "max_iter": 587, "tol": 3.6058193649597426e-04,
               "solver": "lbfgs", "class_weight": None},
    "LGBM"  : {"n_estimators": 600, "num_leaves": 48, "learning_rate": 0.05,
               "subsample": 0.9, "colsample_bytree": 0.7, "reg_lambda": 1.0,
               "random_state": RANDOM_STATE, "verbosity": -1 },
}

# Fixed ensemble weights
GLOBAL_WEIGHTS = {
    "SWIN|MLP(SwinMLP)": 0.55,
    "EVA|MLP(EvaMLP)"  : 0.25,
    f"COAT|{COAT_MODEL}" : 0.20,
}
TEMP_GRID = [0.70,0.75,0.80,0.85,0.90,0.95,1.00,1.05,1.10,1.15,1.20,1.25,1.30]

# ---- Determinism / CUDA fix ----
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = os.environ.get("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

def _pip(*pkgs):
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
        return True
    except Exception as e:
        print(f"[WARN] pip install failed for {pkgs}: {e}")
        return False

# -------------------- Torch --------------------
try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn
except Exception:
    _pip("torch","torchvision","torchaudio")
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn

# -------------------- Sklearn vs. --------------------
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, log_loss

# Optional LightGBM dependency
def _lazy_import_lgbm():
    try:
        import lightgbm as lgb
    except Exception:
        _pip("lightgbm")
        try:
            import lightgbm as lgb
        except Exception:
            lgb = None
    return lgb

# -------------------- Util --------------------
def set_seed(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = True
set_seed(RANDOM_STATE)

def choose_device():
    if DEVICE == "cuda": return "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":  return "cpu"
    return "cuda" if torch.cuda.is_available() else "cpu"
DEV = choose_device()
print("Device:", DEV)

def _glob_any(p: Path, names):
    for nm in names:
        f = p / nm
        if f.exists(): return str(f)
    return None

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True, flags=0)
    return s.values

def _read_global_split_csv(split_csv_path):
    df = pd.read_csv(split_csv_path)
    id_col = None
    for cand in ["image_id","image","Image","filename","fname","img_id","name"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None: raise ValueError(f"No image_id/image column found in {split_csv_path}.")
    ids = _normalize_ids(df[id_col])
    if "label" in df.columns:
        y = df["label"].astype(int).values
    elif "class_name" in df.columns:
        classes = sorted(df["class_name"].unique().tolist())
        mp = {c:i for i,c in enumerate(classes)}
        y = df["class_name"].map(mp).astype(int).values
    else:
        raise ValueError(f"No label/class_name column found in {split_csv_path}.")
    return ids, y, df

def load_global_splits(split_dir):
    paths = {
        "train": os.path.join(split_dir, "train_split.csv"),
        "val"  : os.path.join(split_dir, "val_split.csv"),
        "test" : os.path.join(split_dir, "test_split.csv"),
    }
    out = {}
    for sp, pth in paths.items():
        ids, y, df = _read_global_split_csv(pth)
        out[sp] = {"ids": ids, "y": y, "df": df}
    df_tr = out["train"]["df"]
    if "class_name" in df_tr.columns:
        class_names = sorted(df_tr["class_name"].unique().tolist())
    else:
        class_names = [f"class{i}" for i in sorted(pd.Series(out["train"]["y"]).unique())]
    return out, class_names

def _load_features(dir_path: str, split: str):
    p = Path(dir_path)
    X_path = _glob_any(p, [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"])
    if X_path is None: raise FileNotFoundError(f"No {split} feature array found in {dir_path}.")
    return np.load(X_path)

def brier_score(y_true, proba):
    C = proba.shape[1]; Y = np.eye(C)[y_true]
    return float(np.mean(np.sum((proba - Y)**2, axis=1)))

def ece_score(y_true, proba, n_bins=CALIBRATION_BINS):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); accs = (pred == y_true).astype(float)
    bins = np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        if m.sum(): ece += m.mean() * abs(accs[m].mean() - conf[m].mean())
    return float(ece)

def topk_capture(y_true, proba, k=3):
    topk = np.argpartition(-proba, kth=k-1, axis=1)[:, :k]
    return float(np.any(topk == y_true[:, None], axis=1).mean())

def reliability_bins(y_true, proba, n_bins=CALIBRATION_BINS):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); correct = (pred == y_true).astype(int)
    bins = np.linspace(0,1,n_bins+1); rows=[]
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        rows.append({"bin": i, "count": int(m.sum()),
                     "avg_conf": float(np.nan if m.sum()==0 else conf[m].mean()),
                     "acc": float(np.nan if m.sum()==0 else correct[m].mean()),
                     "gap": float(np.nan if m.sum()==0 else abs(correct[m].mean()-conf[m].mean()))})
    return pd.DataFrame(rows)

def save_reports(prefix, y_true, proba, out_dir, class_names, print_console=True):
    os.makedirs(out_dir, exist_ok=True)
    y_pred = proba.argmax(1)
    row = {"split":prefix,
           "Acc": accuracy_score(y_true, y_pred),
           "MacroF1": f1_score(y_true, y_pred, average="macro", zero_division=0),
           "Brier": brier_score(y_true, proba),
           "ECE": ece_score(y_true, proba),
           "Top3": topk_capture(y_true, proba, 3),
           "N": len(y_true), "classes": proba.shape[1]}
    pd.DataFrame([row]).to_csv(os.path.join(out_dir, f"{prefix}_analysis_report.csv"), index=False)
    reliability_bins(y_true, proba).to_csv(os.path.join(out_dir, f"{prefix}_reliability_bins.csv"), index=False)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
    cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in class_names],
                         columns=[f"pred_{c}" for c in class_names])
    cm_df.to_csv(os.path.join(out_dir, f"{prefix}_confusion_matrix.csv"))
    cr = classification_report(y_true, y_pred, target_names=class_names, zero_division=0, output_dict=True)
    cr_df = pd.DataFrame(cr); cr_df.to_csv(os.path.join(out_dir, f"{prefix}_classification_report.csv"))

    if print_console:
        print(f"\n[{out_dir}] {prefix.upper()} — Acc={row['Acc']:.4f} | MacroF1={row['MacroF1']:.4f} | "
              f"Brier={row['Brier']:.4f} | ECE={row['ECE']:.4f} | Top3={row['Top3']:.4f}")
        print("\nCM:\n", cm_df.to_string())
        print("\nCR:\n", cr_df.to_string())
    return row

# -------------------- Meta --------------------
def read_meta_local(csv_path):
    df = pd.read_csv(csv_path)
    id_col = None
    for cand in ["image","image_id","Image","filename","fname","name","img_id"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None:
        raise ValueError("The metadata CSV must contain an 'image' or 'image_id' column.")
    df["image_id"] = _normalize_ids(df[id_col])
    for c in ["anatom_site_general","age_approx","sex"]:
        if c not in df.columns:
            df[c] = np.nan
    return df

def build_meta_matrix(meta_df, ids, flags):
    df = pd.DataFrame({"image_id": ids}).merge(meta_df, on="image_id", how="left")
    cols = []; mats = []

    if flags["INCLUDE_AGE_2BIT"]:
        thr = float(flags["AGE_THRESHOLD"])
        age = pd.to_numeric(df["age_approx"], errors="coerce")
        a_lt = (age < thr).astype(float).fillna(0.0).values.reshape(-1,1)
        a_ge = (age >= thr).astype(float).fillna(0.0).values.reshape(-1,1)
        mats += [a_lt, a_ge]; cols += ["age_lt","age_ge"]

    if flags["INCLUDE_SEX_2BIT"]:
        sx = df["sex"].astype(str).str.lower()
        male  = (sx == "male").astype(float).values.reshape(-1,1)
        female= (sx == "female").astype(float).values.reshape(-1,1)
        mats += [male, female]; cols += ["sex_male","sex_female"]

    if mats:
        M = np.concatenate(mats, axis=1).astype(np.float32)
    else:
        M = np.zeros((len(df),0), dtype=np.float32)
    return M, cols

# -------------------- MLP --------------------
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(1024,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def _make_loader(X, y, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=0, drop_last=False)

def train_mlp_fixed_epochs(Xfit, yfit, n_classes, hp, device, log_tag):
    from sklearn.metrics import f1_score, accuracy_score
    loader = _make_loader(Xfit, yfit, hp["batch_size"], shuffle=True)
    model = RobustMLP(Xfit.shape[1], n_classes, tuple(hp["hidden"]), hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
    model.train()
    for ep in range(1, hp["epochs"]+1):
        for xb,yb in loader:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss=crit(model(xb), yb); loss.backward(); opt.step()
        if ep % max(1,hp.get("log_every",5))==0 or ep==1:
            model.eval()
            with torch.no_grad():
                lg = model(torch.tensor(Xfit, dtype=torch.float32, device=device)).cpu().numpy()
            proba = torch.softmax(torch.tensor(lg), dim=1).numpy()
            y_hat = proba.argmax(1)
            acc = accuracy_score(yfit, y_hat); f1 = f1_score(yfit, y_hat, average="macro", zero_division=0)
            print(f"[MLP] ep {ep:03d} | {log_tag} Acc {acc:.4f} | MacroF1 {f1:.4f}")
            model.train()
    return model

def logits_and_proba(model, X, device):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X), dtype=torch.long))
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=0, drop_last=False)
    model.eval(); out=[]
    with torch.no_grad():
        for (xb,_) in loader:
            xb=xb.to(device); out.append(model(xb).cpu())
    lg = torch.cat(out,0).numpy()
    P  = torch.softmax(torch.tensor(lg), dim=1).numpy()
    return lg, P

def apply_temp_logits(logits, T):
    return torch.softmax(torch.tensor(logits)/T, dim=1).numpy()

# Fit/transform helpers
def fit_transform_train_then_val(Xtr, Xva, pca_var):
    scaler = StandardScaler().fit(Xtr)
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva)
    if pca_var is None: return scaler, None, Ztr, Zva
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva)

def fit_transform_trainval_then_test(Xtr, Xva, Xte, pca_var):
    scaler = StandardScaler().fit(np.vstack([Xtr, Xva]))
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva); Zte = scaler.transform(Xte)
    if pca_var is None: return scaler, None, Ztr, Zva, Zte
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(np.vstack([Ztr,Zva]))
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)

# Member-model training
def parse_model_token(tok: str):
    if tok.startswith("MLP(") and tok.endswith(")"):
        name = tok[4:-1].strip()
        return "MLP", name
    return tok, None  # LGBM, LogReg

def build_and_fit_val_predict(model_key, preset_name, Ztr, ytr, Zva, n_class):
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        model = train_mlp_fixed_epochs(Ztr, ytr, n_class, hp, DEV, "TRAIN")
        lg_va, _ = logits_and_proba(model, Zva, DEV)
        P_va = apply_temp_logits(lg_va, hp["T"])
        return ("MLP", hp, model, P_va)
    elif model_key=="LGBM":
        lgb = _lazy_import_lgbm()
        if lgb is None:
            print("[WARN] LightGBM is unavailable; skipping this model.")
            return ("LGBM", None, None, None)
        hp = HP_DEFAULTS["LGBM"].copy()
        clf = lgb.LGBMClassifier(**hp)
        clf.fit(Ztr, ytr)
        P_va = clf.predict_proba(Zva)
        return ("LGBM", hp, clf, P_va)
    elif model_key=="LogReg":
        hp = HP_DEFAULTS["LogReg"].copy()
        clf = LogisticRegression(**hp)
        clf.fit(Ztr, ytr)
        P_va = clf.predict_proba(Zva)
        return ("LogReg", hp, clf, P_va)
    else:
        raise ValueError(f"Unknown model: {model_key}")

def refit_and_predict_test(model_key, preset_name, trained, Xfit, yfit, Zte, n_class):
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        model = train_mlp_fixed_epochs(Xfit, yfit, n_class, hp, DEV, "TRAIN∪VAL")
        lg_te, _ = logits_and_proba(model, Zte, DEV)
        P_te = apply_temp_logits(lg_te, hp["T"])
        return P_te
    elif model_key=="LGBM":
        lgb = _lazy_import_lgbm()
        if lgb is None:
            print("[WARN] LightGBM is unavailable; skipping this test evaluation.")
            return None
        hp = HP_DEFAULTS["LGBM"].copy()
        clf = lgb.LGBMClassifier(**hp); clf.fit(Xfit, yfit)
        return clf.predict_proba(Zte)
    elif model_key=="LogReg":
        hp = HP_DEFAULTS["LogReg"].copy()
        clf = LogisticRegression(**hp); clf.fit(Xfit, yfit)
        return clf.predict_proba(Zte)
    else:
        raise ValueError(f"Unknown model: {model_key}")

# -------------------- Ensemble helpers --------------------
def soft_mean(P_list, weights=None):
    P_list = [P for P in P_list if P is not None]
    if not P_list: return None
    if weights is None: weights = np.ones(len(P_list), dtype=float)
    else: weights = np.array(weights[:len(P_list)], dtype=float)
    S = np.zeros_like(P_list[0], dtype=np.float64)
    for P, w in zip(P_list, weights): S += w * P
    S = S / max(1e-12, weights.sum())
    return S

# Main workflow
def main():
    set_seed(RANDOM_STATE)
    root = SAVE_DIR; os.makedirs(root, exist_ok=True)

    
    splits, class_names = load_global_splits(SPLIT_DIR)
    n_class = len(class_names)
    print(f"[Global splits] classes={class_names}")

    # Load feature arrays
    data = {}
    for feat, path in FEATURE_DIRS.items():
        Xtr = _load_features(path, "train"); Xva = _load_features(path, "val"); Xte = _load_features(path, "test")
        ids_tr = splits["train"]["ids"]; y_tr = splits["train"]["y"]
        ids_va = splits["val"]["ids"];   y_va = splits["val"]["y"]
        ids_te = splits["test"]["ids"];  y_te = splits["test"]["y"]
        assert len(Xtr)==len(y_tr)==len(ids_tr)
        assert len(Xva)==len(y_va)==len(ids_va)
        assert len(Xte)==len(y_te)==len(ids_te)
        data[feat] = {"Xtr":Xtr,"ytr":y_tr,"ids_tr":ids_tr,
                      "Xva":Xva,"yva":y_va,"ids_va":ids_va,
                      "Xte":Xte,"yte":y_te,"ids_te":ids_te}

    # 3) Meta
    meta_all = read_meta_local(META_CSV_PATH)
    M_tr, cols_tr = build_meta_matrix(meta_all, data["SWIN"]["ids_tr"], META_FLAGS)
    M_va, cols_va = build_meta_matrix(meta_all, data["SWIN"]["ids_va"], META_FLAGS)
    M_te, cols_te = build_meta_matrix(meta_all, data["SWIN"]["ids_te"], META_FLAGS)
    assert cols_tr==cols_va==cols_te
    print(f"[META] columns: {len(cols_tr)} -> {cols_tr if len(cols_tr)<=10 else str(cols_tr[:10])+' ...'}")

    # 4) Append metadata features to the feature matrices
    if M_tr.shape[1] > 0:
        for k in data.keys():
            data[k]["Xtr"] = np.concatenate([data[k]["Xtr"], M_tr], axis=1).astype(np.float32)
            data[k]["Xva"] = np.concatenate([data[k]["Xva"], M_va], axis=1).astype(np.float32)
            data[k]["Xte"] = np.concatenate([data[k]["Xte"], M_te], axis=1).astype(np.float32)

    # Train ensemble members without leakage
    members = []  # dict: key, feat, model, P_val, P_te, y_val, y_te
    for feat, model_list in MODELS_PER_FEATURE.items():
        print(f"\n===== {feat}: {len(model_list)} model =====")
        Xtr,ytr,Xva,yva,Xte,yte = data[feat]["Xtr"],data[feat]["ytr"],data[feat]["Xva"],data[feat]["yva"],data[feat]["Xte"],data[feat]["yte"]

        
        sc1, pca1, Ztr, Zva = fit_transform_train_then_val(Xtr, Xva, PCA_VARIANCE)

        for tok in model_list:
            model_key, preset = parse_model_token(tok)
            print(f"[{feat}] model = {tok}")
            mdl_name, hp, trained, P_va = build_and_fit_val_predict(model_key, preset, Ztr, ytr, Zva, n_class)
            if P_va is None:
                print(f"[WARN] {feat}|{tok}: validation probabilities could not be generated; skipping this model.")
                continue

            out_dir = os.path.join(SAVE_DIR, feat, tok.replace("/","_"), "VAL")
            save_reports("val", yva, P_va, out_dir, class_names, print_console=True)

            
            sc2, pca2, Ztr2, Zva2, Zte2 = fit_transform_trainval_then_test(Xtr, Xva, Xte, PCA_VARIANCE)
            Xfit = np.vstack([Ztr2, Zva2]); yfit = np.concatenate([ytr, yva])

            P_te = refit_and_predict_test(model_key, preset, trained, Xfit, yfit, Zte2, n_class)
            if P_te is None:
                print(f"[WARN] {feat}|{tok}: test probabilities could not be generated; skipping this model.")
                continue
            out_dir_te = os.path.join(SAVE_DIR, feat, tok.replace("/","_"), "TEST")
            save_reports("test", yte, P_te, out_dir_te, class_names, print_console=True)
            np.save(os.path.join(out_dir_te, "test_proba.npy"), P_te)

            members.append({"key": f"{feat}|{tok}", "feat": feat, "model": tok,
                            "P_val": P_va, "P_te": P_te, "y_val": yva, "y_te": yte})

    if not members:
        raise RuntimeError("No ensemble members were created. Check MODELS_PER_FEATURE.")

    
    y_val = members[0]["y_val"]; y_te = members[0]["y_te"]
    for m in members:
        assert np.array_equal(m["y_val"], y_val) and np.array_equal(m["y_te"], y_te)

    P_val_members = [m["P_val"] for m in members]
    P_te_members  = [m["P_te"]  for m in members]
    member_keys   = [m["key"]   for m in members]

    # Align weights with member order and normalize them as a safety check.
    w_init = np.array([GLOBAL_WEIGHTS.get(k, 0.0) for k in member_keys], dtype=np.float64)
    if not np.isclose(w_init.sum(), 1.0):
        w_init = w_init / max(1e-12, w_init.sum())
        print(f"[INFO] Weights were normalized → {dict(zip(member_keys, np.round(w_init,4)))}")

    print("\n[Ensemble] Members and fixed weights:")
    for k, w in zip(member_keys, w_init):
        print(f" - {k:>24s} : {w:.4f}")

    
    P_val_final = soft_mean(P_val_members, w_init)
    P_te_final  = soft_mean(P_te_members,  w_init)

    
    print("\n=== Ensemble Temperature Pick on VAL (NLL) ===")
    bestT, bestNLL = 1.0, float("inf")
    for T in TEMP_GRID:
        Pv = torch.softmax(torch.log(torch.tensor(P_val_final+1e-12))/T, dim=1).numpy()
        nll = log_loss(y_val, Pv, labels=list(range(Pv.shape[1])))
        print(f"T={T:.2f} | VAL NLL={nll:.6f}")
        if nll < bestNLL: bestNLL, bestT = nll, T

    P_val_cal = torch.softmax(torch.log(torch.tensor(P_val_final+1e-12))/bestT, dim=1).numpy()
    P_test    = torch.softmax(torch.log(torch.tensor(P_te_final+1e-12))/bestT, dim=1).numpy()

    out_dir = os.path.join(SAVE_DIR, "ENSEMBLE_FINAL_FIXED"); os.makedirs(out_dir, exist_ok=True)
    save_reports("val",  y_val, P_val_cal, out_dir, class_names, print_console=True)
    save_reports("test", y_te,  P_test,    out_dir, class_names, print_console=True)
    np.save(os.path.join(out_dir, "val_proba.npy"),  P_val_cal)
    np.save(os.path.join(out_dir, "test_proba.npy"), P_test)
    with open(os.path.join(out_dir, "members.json"), "w") as f:
        json.dump(member_keys, f, indent=2)
    with open(os.path.join(out_dir, "ensemble_meta.json"), "w") as f:
        json.dump({"strategy":"GLOBAL_FIXED",
                   "weights": dict(zip(member_keys, [float(x) for x in w_init])),
                   "T": float(bestT)}, f, indent=2)

    print(f"\n✅ Completed. Strategy=GLOBAL_FIXED | Weights={dict(zip(member_keys, np.round(w_init,4)))} | T={bestT:.2f} | Outputs → {out_dir}")

if __name__ == "__main__":
    main()

## Metadata-aware ensemble — validation-only weight selection

**Research question:** For the configured metadata options (region, age, and sex), which three-member weight combination maximizes validation Macro-F1?

**Selection rule:** the grid is evaluated only with validation probabilities and labels. The chosen weights are frozen; the ensemble temperature is selected using validation NLL. The frozen configuration is then evaluated once on the test split. This replaces the original TEST-based weight search.

In [ ]:
# -*- coding: utf-8 -*-
# ============================================================





# ============================================================

import os, sys, json, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# User configuration
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
SPLIT_DIR = str(PROJECT_ROOT / "data" / "splits")  # train_split.csv / val_split.csv / test_split.csv
FEATURE_DIRS = {
    "SWIN": str(PROJECT_ROOT / "features" / "swin_ft768"),
    "EVA": str(PROJECT_ROOT / "features" / "eva02"),
    "COAT": str(PROJECT_ROOT / "features" / "coatnet_ft672"),
}
META_CSV_PATH = str(PROJECT_ROOT / "data" / "metadata" / "ISIC_2019_Training_Metadata.csv")
SAVE_DIR = str(PROJECT_ROOT / "outputs" / "ensemble_meta_runs_validation_weights")
os.makedirs(SAVE_DIR, exist_ok=True)


ENSEMBLE_T = 0.9  # Legacy fallback only; final ensemble temperature is selected on validation.
MEMBER_T = 0.9            
PCA_VARIANCE = 0.99
CALIBRATION_BINS = 15
RANDOM_STATE = 42
DEVICE = "auto"  # "cuda" | "cpu" | "auto"


META_FLAGS = {
    # Region
    "INCLUDE_REGION": True,
    # Region grouping mode:
    
    
    "REGION_MODE": "raw",   # "raw" | "map"

    # Sex
    "INCLUDE_SEX": False,
    "SEX_ADD_MISSING_FLAG": False,   
                                    

    # Age
    "INCLUDE_AGE": True,
    "AGE_MODE": "quantile_5",       # "none" | "z" | "fixed_bins" | "quantile_5" | "splines"
    "AGE_ADD_MISSING_FLAG": True,   
    "AGE_FIXED_BINS": [18, 40, 60, 80],  
    "AGE_QUANTILE_K": 5,            
    "AGE_SPLINE_KNOTS": 5,          
}



REGION_MAPPING = {
    
    # "anterior torso":"torso", "posterior torso":"torso", "lateral torso":"torso"
    # "palms/soles":"acral", "oral/genital":"mucosal"
}


GRID_STEP = 0.05
SWIN_MIN = 0.50      # SWIN ≥ 0.50
VALIDATION_SELECTION_METRIC = "MacroF1"  # Selection criterion; computed on validation only
TOP_K_SHOW = 15

# ============================================================

# ============================================================
os.environ["CUBLAS_WORKSPACE_CONFIG"] = os.environ.get("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

def _pip(*pkgs):
    import subprocess
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
        return True
    except Exception as e:
        print(f"[WARN] pip install failed for {pkgs}: {e}")
        return False

# Torch
try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn
except Exception:
    _pip("torch","torchvision","torchaudio")
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    import torch.backends.cudnn as cudnn

# Sklearn
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer, SplineTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, log_loss

def set_seed(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    cudnn.benchmark = True
set_seed(RANDOM_STATE)

def choose_device():
    if DEVICE == "cuda": return "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cpu":  return "cpu"
    return "cuda" if torch.cuda.is_available() else "cpu"
DEV = choose_device()
print("Device:", DEV)

# -------------------- file helpers --------------------
def _glob_any(p: Path, names):
    for nm in names:
        f = p / nm
        if f.exists(): return str(f)
    return None

def _normalize_ids(arr_like):
    s = pd.Series(arr_like, dtype=str)
    s = s.str.replace(r"^.*[\\/]", "", regex=True)
    s = s.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True)
    return s.values

def _read_global_split_csv(split_csv_path):
    df = pd.read_csv(split_csv_path)
    id_col = None
    for cand in ["image_id","image","Image","filename","fname","img_id","name"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None: raise ValueError(f"No image_id/image column found in {split_csv_path}.")
    ids = _normalize_ids(df[id_col])

    if "label" in df.columns:
        y = df["label"].astype(int).values
    elif "class_name" in df.columns:
        classes = sorted(df["class_name"].unique().tolist())
        mp = {c:i for i,c in enumerate(classes)}
        y = df["class_name"].map(mp).astype(int).values
    else:
        raise ValueError(f"No label/class_name column found in {split_csv_path}.")
    return ids, y, df

def load_global_splits(split_dir):
    paths = {
        "train": os.path.join(split_dir, "train_split.csv"),
        "val"  : os.path.join(split_dir, "val_split.csv"),
        "test" : os.path.join(split_dir, "test_split.csv"),
    }
    out = {}
    for sp, pth in paths.items():
        ids, y, df = _read_global_split_csv(pth)
        out[sp] = {"ids": ids, "y": y, "df": df}
    df_tr = out["train"]["df"]
    if "class_name" in df_tr.columns:
        class_names = sorted(df_tr["class_name"].unique().tolist())
    else:
        class_names = [f"class{i}" for i in sorted(pd.Series(out["train"]["y"]).unique())]
    return out, class_names

def _load_features(dir_path: str, split: str):
    p = Path(dir_path)
    X_path = _glob_any(p, [f"{split}_features.npy", f"features_{split}.npy", f"X_{split}.npy", f"{split}.npy"])
    if X_path is None: raise FileNotFoundError(f"No {split} feature array found in {dir_path}.")
    return np.load(X_path)

# -------------------- metrics --------------------
def brier_score(y_true, proba):
    C = proba.shape[1]; Y = np.eye(C)[y_true]
    return float(np.mean(np.sum((proba - Y)**2, axis=1)))

def ece_score(y_true, proba, n_bins=CALIBRATION_BINS):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); accs = (pred == y_true).astype(float)
    bins = np.linspace(0,1,n_bins+1); ece=0.0
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        if m.sum(): ece += m.mean() * abs(accs[m].mean() - conf[m].mean())
    return float(ece)

def reliability_bins(y_true, proba, n_bins=CALIBRATION_BINS):
    conf = proba.max(axis=1); pred = proba.argmax(axis=1); correct = (pred == y_true).astype(int)
    bins = np.linspace(0,1,n_bins+1); rows=[]
    for i in range(n_bins):
        m = (conf >= bins[i]) & (conf < bins[i+1] if i<n_bins-1 else conf <= bins[i+1])
        rows.append({"bin": i, "count": int(m.sum()),
                     "avg_conf": float(np.nan if m.sum()==0 else conf[m].mean()),
                     "acc": float(np.nan if m.sum()==0 else correct[m].mean()),
                     "gap": float(np.nan if m.sum()==0 else abs(correct[m].mean()-conf[m].mean()))})
    return pd.DataFrame(rows)

def save_reports(prefix, y_true, proba, out_dir, class_names, print_console=True):
    os.makedirs(out_dir, exist_ok=True)
    y_pred = proba.argmax(1)
    row = {"split":prefix,
           "Acc": accuracy_score(y_true, y_pred),
           "MacroF1": f1_score(y_true, y_pred, average="macro", zero_division=0),
           "Brier": brier_score(y_true, proba),
           "ECE": ece_score(y_true, proba),
           "Top3": float(np.any(np.argpartition(-proba,2,axis=1)[:,:3]==y_true[:,None],axis=1).mean()),
           "N": len(y_true), "classes": proba.shape[1]}
    pd.DataFrame([row]).to_csv(os.path.join(out_dir, f"{prefix}_analysis_report.csv"), index=False)
    reliability_bins(y_true, proba).to_csv(os.path.join(out_dir, f"{prefix}_reliability_bins.csv"), index=False)

    cm = confusion_matrix(y_true, y_pred, labels=list(range(proba.shape[1])))
    pd.DataFrame(cm).to_csv(os.path.join(out_dir, f"{prefix}_confusion_matrix.csv"))
    cr = classification_report(y_true, y_pred, zero_division=0, output_dict=True)
    pd.DataFrame(cr).to_csv(os.path.join(out_dir, f"{prefix}_classification_report.csv"))

    if print_console:
        print(f"[{out_dir}] {prefix.upper()} — Acc={row['Acc']:.4f} | MacroF1={row['MacroF1']:.4f} | Brier={row['Brier']:.4f} | ECE={row['ECE']:.4f}")
    return row

# ============================================================
#  Meta (region/sex/age)
# ============================================================
def read_meta_local(csv_path):
    df = pd.read_csv(csv_path)
    id_col = None
    for cand in ["image","image_id","Image","filename","fname","name","img_id"]:
        if cand in df.columns: id_col = cand; break
    if id_col is None:
        raise ValueError("The metadata CSV must contain an 'image' or 'image_id' column.")
    df["image_id"] = _normalize_ids(df[id_col])
    for c in ["anatom_site_general","age_approx","sex"]:
        if c not in df.columns: df[c] = np.nan
    
    return df[["image_id","anatom_site_general","age_approx","sex"]].copy()

def norm_region(x):
    if pd.isna(x): return "missing"
    s = str(x).strip().lower()
    mp = {
        "anterior torso":"anterior torso",
        "posterior torso":"posterior torso",
        "lateral torso":"lateral torso",
        "upper extremity":"upper extremity",
        "lower extremity":"lower extremity",
        "head/neck":"head/neck",
        "palms/soles":"palms/soles",
        "oral/genital":"oral/genital",
        "torso anterior":"anterior torso",
        "torso posterior":"posterior torso",
        "torso lateral":"lateral torso",
        "none":"missing","nan":"missing","": "missing"
    }
    if s in mp: return mp[s]
    if "anterior" in s and "torso" in s: return "anterior torso"
    if "posterior" in s and "torso" in s: return "posterior torso"
    if "lateral" in s and "torso" in s: return "lateral torso"
    if "upper" in s: return "upper extremity"
    if "lower" in s: return "lower extremity"
    if "head" in s or "neck" in s: return "head/neck"
    if "palm" in s or "sole" in s or "acral" in s: return "palms/soles"
    if "oral" in s or "genital" in s or "mucosal" in s: return "oral/genital"
    if "torso" in s: return "anterior torso"
    return "missing"

def region_group(x):
    """Return the normalized raw region as-is or group it according to REGION_MAPPING, based on REGION_MODE."""
    r = norm_region(x)
    if META_FLAGS["REGION_MODE"] == "raw":
        return r
    # "map"
    return REGION_MAPPING.get(r, r)

from sklearn.base import BaseEstimator, TransformerMixin

class MetaEncoder(BaseEstimator, TransformerMixin):
    """Fit on TRAIN and transform VAL/TEST. Region, sex, and age features are modular."""
    def __init__(self, flags):
        self.flags = flags
        self.enc_region = None
        self.region_cols = []
        self.enc_sex = None
        self.sex_cols = []
        self.age_pack = None
        self.age_cols = []
        self.age_med = None

    def fit(self, ids, meta_df):
        mtr = pd.DataFrame({"image_id": ids}).merge(meta_df, on="image_id", how="left")

        # ---- REGION
        if self.flags["INCLUDE_REGION"]:
            rgrp = mtr["anatom_site_general"].apply(region_group)
            self.enc_region = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
            self.enc_region.fit(rgrp.values.reshape(-1,1))
            self.region_cols = [f"reg_{c}" for c in self.enc_region.categories_[0]]
        else:
            self.enc_region = None; self.region_cols = []

        # ---- SEX
        if self.flags["INCLUDE_SEX"]:
            sx = mtr["sex"].astype(str).str.lower().replace({"nan":"missing","none":"missing","": "missing"})
            self.enc_sex = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
            self.enc_sex.fit(sx.values.reshape(-1,1))
            self.sex_cols = [f"sex_{c}" for c in self.enc_sex.categories_[0]]
            if self.flags["SEX_ADD_MISSING_FLAG"]:
                self.sex_cols += ["sex_missing_flag"]
        else:
            self.enc_sex = None; self.sex_cols = []

        # ---- AGE
        if self.flags["INCLUDE_AGE"] and self.flags["AGE_MODE"]!="none":
            age_raw = pd.to_numeric(mtr["age_approx"], errors="coerce")
            self.age_med = np.nanmedian(age_raw.values) if not np.isnan(age_raw.values).all() else 50.0
            age_filled = age_raw.fillna(self.age_med).values.reshape(-1,1)
            mode = self.flags["AGE_MODE"]

            if mode == "z":
                self.age_pack = ("z", StandardScaler().fit(age_filled))
                self.age_cols = ["age_std"]
            elif mode == "fixed_bins":
                edges = np.array(self.flags["AGE_FIXED_BINS"], dtype=float)
                bins = np.digitize(age_filled.ravel(), edges, right=False).reshape(-1,1)
                ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore").fit(bins)
                self.age_pack = ("fixed_bins", {"edges":edges, "ohe":ohe})
                self.age_cols = [f"agebin_{i}" for i in range(ohe.categories_[0].size)]
            elif mode == "quantile_5":
                k = int(self.flags["AGE_QUANTILE_K"])
                kb = KBinsDiscretizer(n_bins=k, encode="ordinal", strategy="quantile").fit(age_filled)
                cat = kb.transform(age_filled).astype(int)
                ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore").fit(cat)
                self.age_pack = ("quantile", {"kb":kb, "ohe":ohe})
                self.age_cols = [f"ageq_{i}" for i in range(ohe.categories_[0].size)]
            elif mode == "splines":
                st = SplineTransformer(degree=3, n_knots=int(self.flags["AGE_SPLINE_KNOTS"]), include_bias=False)
                st.fit(age_filled)
                self.age_pack = ("splines", st)
                self.age_cols = [f"age_spline_{i}" for i in range(st.n_features_out_)]
            else:
                raise ValueError(f"Bilinmeyen AGE_MODE: {mode}")

            if self.flags["AGE_ADD_MISSING_FLAG"]:
                self.age_cols += ["age_missing_flag"]
        else:
            self.age_pack = None; self.age_cols = []; self.age_med = None

        return self

    def _encode_age(self, age_series):
        if self.age_pack is None:
            return None
        age = pd.to_numeric(age_series, errors="coerce")
        miss_flag = age.isna().astype(float).values.reshape(-1,1)
        age = age.fillna(self.age_med).values.reshape(-1,1)
        mode = self.age_pack[0]; obj = self.age_pack[1]

        if mode == "z":
            A = obj.transform(age)
        elif mode == "fixed_bins":
            edges = obj["edges"]; ohe = obj["ohe"]
            bins = np.digitize(age.ravel(), edges, right=False).reshape(-1,1)
            A = ohe.transform(bins).toarray()
        elif mode == "quantile":
            kb=obj["kb"]; ohe=obj["ohe"]
            cat = kb.transform(age).astype(int)
            A = ohe.transform(cat).toarray()
        elif mode == "splines":
            A = obj.transform(age)
        else:
            raise ValueError("unknown age mode")

        if META_FLAGS["AGE_ADD_MISSING_FLAG"]:
            A = np.concatenate([A, miss_flag], axis=1)
        return A

    def transform(self, ids, meta_df):
        df = pd.DataFrame({"image_id": ids}).merge(meta_df, on="image_id", how="left")
        mats=[]; cols=[]

        # region
        if self.enc_region is not None:
            rgrp = df["anatom_site_general"].apply(region_group)
            R = self.enc_region.transform(rgrp.values.reshape(-1,1)).toarray()
            mats.append(R); cols += self.region_cols

        # sex
        if self.enc_sex is not None:
            sx = df["sex"].astype(str).str.lower().replace({"nan":"missing","none":"missing","": "missing"})
            S = self.enc_sex.transform(sx.values.reshape(-1,1)).toarray()
            if self.flags["SEX_ADD_MISSING_FLAG"]:
                miss = (sx=="missing").astype(float).values.reshape(-1,1)
                S = np.concatenate([S, miss], axis=1)
            mats.append(S); cols += self.sex_cols

        # age
        if self.age_pack is not None:
            A = self._encode_age(df["age_approx"])
            mats.append(A); cols += self.age_cols

        if mats:
            M = np.concatenate(mats, axis=1).astype(np.float32)
        else:
            M = np.zeros((len(df),0), dtype=np.float32)
        return M, cols

# ============================================================

# ============================================================
class RobustMLP(nn.Module):
    def __init__(self, in_dim, n_classes, hidden=(2048,), dropout=0.3):
        super().__init__()
        layers=[]; last=in_dim
        for h in hidden:
            layers += [nn.Linear(last,h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last=h
        layers += [nn.Linear(last, n_classes)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def _make_loader(X, y, bs, shuffle=True):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
    g = torch.Generator(device="cpu"); g.manual_seed(RANDOM_STATE)
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=0, drop_last=False)

NAMED_PRESETS = {
    "SwinMLP": {
        "hidden": [2048], "dropout": 0.1605757337,
        "lr": 0.000341409509895, "weight_decay": 0.0001488348195284,
        "batch_size": 128, "epochs": 50, "log_every": 5
    },
    "EvaMLP": {
        "hidden": [2048], "dropout": 0.3203382207442766,
        "lr": 0.0009765455593461, "weight_decay": 8.212522451000109e-05,
        "batch_size": 128, "epochs": 50, "log_every": 5
    },
}
HP_LOGREG = {"C": 2.3402196063650043, "max_iter": 587, "tol": 3.6e-04,
             "solver": "lbfgs", "class_weight": None, "n_jobs": -1}

def train_mlp_fixed_epochs(Xfit, yfit, n_classes, hp, device, log_tag):
    loader = _make_loader(Xfit, yfit, hp["batch_size"], shuffle=True)
    model = RobustMLP(Xfit.shape[1], n_classes, tuple(hp["hidden"]), hp["dropout"]).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    crit = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
    model.train()
    for ep in range(1, hp["epochs"]+1):
        for xb,yb in loader:
            xb=xb.to(device); yb=yb.to(device)
            opt.zero_grad(set_to_none=True)
            loss=crit(model(xb), yb); loss.backward(); opt.step()
        if ep % max(1,hp.get("log_every",5))==0 or ep==1:
            model.eval()
            with torch.no_grad():
                lg = model(torch.tensor(Xfit, dtype=torch.float32, device=device)).cpu().numpy()
            proba = torch.softmax(torch.tensor(lg), dim=1).numpy()
            y_hat = proba.argmax(1)
            acc = accuracy_score(yfit, y_hat); f1 = f1_score(yfit, y_hat, average="macro", zero_division=0)
            print(f"[MLP] ep {ep:03d} | {log_tag} Acc {acc:.4f} | MacroF1 {f1:.4f}")
            model.train()
    return model

def logits_and_proba(model, X, device):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.zeros(len(X), dtype=torch.long))
    loader = DataLoader(ds, batch_size=256, shuffle=False, num_workers=0, drop_last=False)
    model.eval(); out=[]
    with torch.no_grad():
        for (xb,_) in loader:
            xb=xb.to(device); out.append(model(xb).cpu())
    lg = torch.cat(out,0).numpy()
    P  = torch.softmax(torch.tensor(lg), dim=1).numpy()
    return lg, P

def apply_temp_logits(logits, T):
    return torch.softmax(torch.tensor(logits)/T, dim=1).numpy()

def fit_transform_train_then_val(Xtr, Xva, pca_var):
    scaler = StandardScaler().fit(Xtr)
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva)
    if pca_var is None: return scaler, None, Ztr, Zva
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(Ztr)
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva)

def fit_transform_trainval_then_test(Xtr, Xva, Xte, pca_var):
    scaler = StandardScaler().fit(np.vstack([Xtr, Xva]))
    Ztr = scaler.transform(Xtr); Zva = scaler.transform(Xva); Zte = scaler.transform(Xte)
    if pca_var is None: return scaler, None, Ztr, Zva, Zte
    pca = PCA(n_components=pca_var, svd_solver="full", random_state=RANDOM_STATE).fit(np.vstack([Ztr,Zva]))
    return scaler, pca, pca.transform(Ztr), pca.transform(Zva), pca.transform(Zte)

def parse_model_token(tok: str):
    if tok.startswith("MLP(") and tok.endswith(")"):
        name = tok[4:-1].strip()
        return "MLP", name
    return tok, None

def build_and_fit_val_predict(model_key, preset_name, Ztr, ytr, Zva, n_class):
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        model = train_mlp_fixed_epochs(Ztr, ytr, n_class, hp, DEV, "TRAIN")
        lg_va, _ = logits_and_proba(model, Zva, DEV)
        P_va = apply_temp_logits(lg_va, MEMBER_T)  
        return ("MLP", hp, model, P_va)
    elif model_key=="LogReg":
        clf = LogisticRegression(**HP_LOGREG).fit(Ztr, ytr)
        return ("LogReg", HP_LOGREG, clf, clf.predict_proba(Zva))
    else:
        raise ValueError(f"Unknown model: {model_key}")

def refit_and_predict_test(model_key, preset_name, trained, Xfit, yfit, Zte, n_class):
    if model_key=="MLP":
        hp = NAMED_PRESETS[preset_name]
        model = train_mlp_fixed_epochs(Xfit, yfit, n_class, hp, DEV, "TRAIN∪VAL")
        lg_te, _ = logits_and_proba(model, Zte, DEV)
        return apply_temp_logits(lg_te, MEMBER_T)
    elif model_key=="LogReg":
        clf = LogisticRegression(**HP_LOGREG).fit(Xfit, yfit)
        return clf.predict_proba(Zte)
    else:
        raise ValueError(f"Unknown model: {model_key}")

def soft_mean(P_list, weights):
    S = np.zeros_like(P_list[0], dtype=np.float64)
    for P, w in zip(P_list, weights): S += w * P
    return S / max(1e-12, np.sum(weights))

# ============================================================
# Validation-only ensemble weight search
# ============================================================
def gen_weight_grid(step=0.05, swin_min=0.50):
    """Generate non-negative three-member weights that sum to one."""
    vals = np.arange(0.0, 1.0 + 1e-9, step)
    combos = []
    for ws in vals:
        if ws + 1e-9 < swin_min:
            continue
        for we in vals:
            wc = 1.0 - ws - we
            if wc < -1e-9:
                continue
            if abs(round(wc / step) * step - wc) > 1e-6:
                continue
            wc = round(wc / step) * step
            if wc < -1e-9 or ws + we + wc > 1.000001:
                continue
            if abs((ws + we + wc) - 1.0) > 1e-6:
                continue
            combos.append((round(ws, 2), round(we, 2), round(wc, 2)))
    return combos

def evaluate_weights_on_validation(P_list_val, y_val, weight_grid):
    """Score candidate weights on validation only; TEST labels are not accepted here."""
    rows = []
    for ws, we, wc in weight_grid:
        weights = np.asarray([ws, we, wc], dtype=np.float64)
        probabilities = soft_mean(P_list_val, weights)
        y_pred = probabilities.argmax(1)
        rows.append({
            "w_swin": ws, "w_eva": we, "w_coat": wc,
            "VAL_Accuracy": accuracy_score(y_val, y_pred),
            "VAL_MacroF1": f1_score(y_val, y_pred, average="macro", zero_division=0),
            "VAL_Brier": brier_score(y_val, probabilities),
            "VAL_ECE": ece_score(y_val, probabilities),
        })
    return pd.DataFrame(rows).sort_values(
        ["VAL_MacroF1", "VAL_Accuracy"], ascending=[False, False]
    ).reset_index(drop=True)

def select_temperature_on_validation(P_val, y_val, temperature_grid):
    """Select a single ensemble temperature by validation NLL."""
    best_t, best_nll = 1.0, float("inf")
    for temperature in temperature_grid:
        calibrated = torch.softmax(
            torch.log(torch.as_tensor(P_val + 1e-12, dtype=torch.float64)) / temperature, dim=1
        ).cpu().numpy()
        nll = log_loss(y_val, calibrated, labels=list(range(calibrated.shape[1])))
        if nll < best_nll:
            best_t, best_nll = float(temperature), float(nll)
    return best_t, best_nll

# ============================================================
#  MAIN
# ============================================================
def main():
    set_seed(RANDOM_STATE)
    splits, class_names = load_global_splits(SPLIT_DIR)
    print("Device:", DEV)
    print("[Global splits] classes=", class_names)

    # 1) Feature representations
    data = {}
    for feat, path in FEATURE_DIRS.items():
        Xtr = _load_features(path, "train"); Xva = _load_features(path, "val"); Xte = _load_features(path, "test")
        ids_tr = splits["train"]["ids"]; y_tr = splits["train"]["y"]
        ids_va = splits["val"]["ids"];   y_va = splits["val"]["y"]
        ids_te = splits["test"]["ids"];  y_te = splits["test"]["y"]
        assert len(Xtr)==len(y_tr)==len(ids_tr)
        assert len(Xva)==len(y_va)==len(ids_va)
        assert len(Xte)==len(y_te)==len(ids_te)
        data[feat] = {"Xtr":Xtr,"ytr":y_tr,"ids_tr":ids_tr,
                      "Xva":Xva,"yva":y_va,"ids_va":ids_va,
                      "Xte":Xte,"yte":y_te,"ids_te":ids_te}

    # 2) Fit metadata encoders on TRAIN and transform TRAIN/VAL/TEST
    meta_df = read_meta_local(META_CSV_PATH)
    enc = MetaEncoder(META_FLAGS).fit(data["SWIN"]["ids_tr"], meta_df)
    M_tr, meta_cols = enc.transform(data["SWIN"]["ids_tr"], meta_df)
    M_va, _         = enc.transform(data["SWIN"]["ids_va"], meta_df)
    M_te, _         = enc.transform(data["SWIN"]["ids_te"], meta_df)
    print(f"[META] cols: {len(meta_cols)} -> {meta_cols[:12] + (['...'] if len(meta_cols)>12 else [])}")

    # 3) Append metadata features to the feature matrices
    for k in data.keys():
        data[k]["Xtr"] = np.concatenate([data[k]["Xtr"], M_tr], axis=1).astype(np.float32)
        data[k]["Xva"] = np.concatenate([data[k]["Xva"], M_va], axis=1).astype(np.float32)
        data[k]["Xte"] = np.concatenate([data[k]["Xte"], M_te], axis=1).astype(np.float32)

    # Train ensemble members without leakage
    members = []
    MODELS_PER_FEATURE = {"SWIN": ["MLP(SwinMLP)"], "EVA": ["MLP(EvaMLP)"], "COAT": ["LogReg"]}
    n_class = len(class_names)

    out_root = os.path.join(SAVE_DIR, "RUN_FIXED_T")
    for feat, model_list in MODELS_PER_FEATURE.items():
        print(f"\n===== {feat}: {len(model_list)} model =====")
        Xtr,ytr,Xva,yva,Xte,yte = data[feat]["Xtr"],data[feat]["ytr"],data[feat]["Xva"],data[feat]["yva"],data[feat]["Xte"],data[feat]["yte"]

        
        _, _, Ztr, Zva = fit_transform_train_then_val(Xtr, Xva, PCA_VARIANCE)

        # TEST: scaler/PCA TRAIN∪VAL
        _, _, Ztr2, Zva2, Zte2 = fit_transform_trainval_then_test(Xtr, Xva, Xte, PCA_VARIANCE)
        Xfit = np.vstack([Ztr2, Zva2]); yfit = np.concatenate([ytr, yva])

        for tok in model_list:
            model_key, preset = parse_model_token(tok)
            print(f"[{feat}] model = {tok}")
            mdl_name, hp, trained, P_va = build_and_fit_val_predict(model_key, preset, Ztr, ytr, Zva, n_class)
            out_dir = os.path.join(out_root, feat, tok.replace("/","_"), "VAL")
            save_reports("val", yva, P_va, out_dir, class_names, print_console=True)

            P_te = refit_and_predict_test(model_key, preset, trained, Xfit, yfit, Zte2, n_class)
            out_dir_te = os.path.join(out_root, feat, tok.replace("/","_"), "TEST")
            save_reports("test", yte, P_te, out_dir_te, class_names, print_console=True)
            np.save(os.path.join(out_dir_te, "test_proba.npy"), P_te)

            members.append({"key": f"{feat}|{tok}", "feat": feat, "model": tok,
                            "P_val": P_va, "P_te": P_te, "y_val": yva, "y_te": yte})

    assert len(members)==3, "Expected exactly three members (SWIN/EVA/COAT)."

    # 5) Select ensemble weights using VALIDATION predictions only.
    def get_member(key):
        for member in members:
            if member["key"] == key:
                return member
        raise KeyError(f"Required ensemble member not found: {key}")

    swin = get_member("SWIN|MLP(SwinMLP)")
    eva = get_member("EVA|MLP(EvaMLP)")
    coat = get_member("COAT|LogReg")
    member_keys = ["SWIN|MLP(SwinMLP)", "EVA|MLP(EvaMLP)", "COAT|LogReg"]
    P_list_val = [swin["P_val"], eva["P_val"], coat["P_val"]]
    P_list_test = [swin["P_te"], eva["P_te"], coat["P_te"]]
    y_val = swin["y_val"]
    y_test = swin["y_te"]
    assert all(np.array_equal(member["y_val"], y_val) for member in [swin, eva, coat])
    assert all(np.array_equal(member["y_te"], y_test) for member in [swin, eva, coat])

    weight_grid = gen_weight_grid(step=GRID_STEP, swin_min=SWIN_MIN)
    print(f"\nValidation weight search: {len(weight_grid)} candidate combinations.")
    df_search = evaluate_weights_on_validation(P_list_val, y_val, weight_grid)
    val_search_path = os.path.join(out_root, "WEIGHT_SEARCH_VALIDATION_sorted_by_VAL_MacroF1.csv")
    df_search.to_csv(val_search_path, index=False)
    print("\nTop candidate weights on VALIDATION only:")
    print(df_search.head(TOP_K_SHOW).to_string(index=False))

    # Freeze the validation-selected weights before accessing TEST metrics.
    best_row = df_search.iloc[0]
    selected_weights = np.asarray(
        [best_row["w_swin"], best_row["w_eva"], best_row["w_coat"]], dtype=np.float64
    )
    print("\nFrozen weights selected on validation:", dict(zip(member_keys, selected_weights.tolist())))

    P_val_ensemble = soft_mean(P_list_val, selected_weights)
    P_test_ensemble = soft_mean(P_list_test, selected_weights)
    temperature_grid = [0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.25, 1.30]
    selected_t, selected_nll = select_temperature_on_validation(P_val_ensemble, y_val, temperature_grid)
    print(f"Selected ensemble temperature on validation: T={selected_t:.2f}; VAL NLL={selected_nll:.6f}")

    P_val_calibrated = torch.softmax(
        torch.log(torch.as_tensor(P_val_ensemble + 1e-12, dtype=torch.float64)) / selected_t, dim=1
    ).cpu().numpy()
    P_test_calibrated = torch.softmax(
        torch.log(torch.as_tensor(P_test_ensemble + 1e-12, dtype=torch.float64)) / selected_t, dim=1
    ).cpu().numpy()

    final_out_dir = os.path.join(out_root, "FINAL_VALIDATION_SELECTED_ENSEMBLE")
    os.makedirs(final_out_dir, exist_ok=True)
    save_reports("val", y_val, P_val_calibrated, final_out_dir, class_names, print_console=True)
    save_reports("test", y_test, P_test_calibrated, final_out_dir, class_names, print_console=True)
    np.save(os.path.join(final_out_dir, "val_proba.npy"), P_val_calibrated)
    np.save(os.path.join(final_out_dir, "test_proba.npy"), P_test_calibrated)
    with open(os.path.join(final_out_dir, "ensemble_selection.json"), "w") as file:
        json.dump({
            "selection_split": "validation",
            "selection_metric": "macro_f1",
            "member_keys": member_keys,
            "weights": dict(zip(member_keys, [float(value) for value in selected_weights])),
            "temperature_selection_split": "validation",
            "temperature_selection_metric": "negative_log_likelihood",
            "temperature": float(selected_t),
        }, file, indent=2)
    print("\nFinal ensemble evaluation complete. Test metrics were not used to select weights or temperature.")
    print(f"Validation weight-search table: {val_search_path}")
    print(f"Final reports: {final_out_dir}")

if __name__ == "__main__":
    main()

## Baseline comparison — validation-selected KNN/Logistic Regression

**Research question:** For each backbone representation, which of two pre-defined classical baselines is strongest on validation?

**Selection rule:** choose between KNN and Logistic Regression using validation Macro-F1, then Macro-AUC and MCC. Refit that selected estimator on train + validation, then evaluate once on test. Test scores do not choose the estimator.

In [ ]:
import os, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# User-configurable paths. Keep datasets and feature arrays outside the repository.
PROJECT_ROOT = Path(os.environ.get("SKIN_LESION_PROJECT_ROOT", "/content/skin_lesion_project"))
OUT_DIR = PROJECT_ROOT / "outputs" / "validation_selected_baselines"
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
FEATURE_DIRS = {
    "SWIN": PROJECT_ROOT / "features" / "swin_ft768",
    "EVA": PROJECT_ROOT / "features" / "eva02",
    "COAT": PROJECT_ROOT / "features" / "coatnet_ft672",
}
CLASS_NAMES = ["AK", "BCC", "BKL", "DF", "MEL", "NV", "SCC", "VASC"]
RNG_SEED = 42

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix, matthews_corrcoef
from IPython.display import display


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)


def normalize_ids(values):
    series = pd.Series(values, dtype=str)
    series = series.str.replace(r"^.*[\\/]", "", regex=True)
    series = series.str.replace(r"\.(jpg|jpeg|png|bmp|tif|tiff|gif)$", "", regex=True)
    return series.values


def read_split_csv(path):
    frame = pd.read_csv(path)
    id_column = next((column for column in ["image_id", "image", "Image", "filename", "fname", "img_id", "name"] if column in frame.columns), None)
    if id_column is None:
        raise ValueError(f"No image identifier column found in {path}.")
    image_ids = normalize_ids(frame[id_column])
    if "label" in frame.columns:
        labels = frame["label"].astype(int).values
    elif "class_name" in frame.columns:
        mapping = {name: index for index, name in enumerate(CLASS_NAMES)}
        unknown = sorted(set(frame["class_name"].dropna()) - set(mapping))
        if unknown:
            raise ValueError(f"Unexpected class labels in {path}: {unknown}")
        labels = frame["class_name"].map(mapping).astype(int).values
    else:
        raise ValueError(f"No label or class_name column found in {path}.")
    return image_ids, labels


def load_features(directory, split_name):
    directory = Path(directory)
    candidates = [f"{split_name}_features.npy", f"features_{split_name}.npy", f"X_{split_name}.npy", f"{split_name}.npy"]
    for filename in candidates:
        candidate = directory / filename
        if candidate.exists():
            return np.load(candidate)
    raise FileNotFoundError(f"Could not find {split_name} features in {directory}.")


def macro_auc_ovr(y_true, probabilities):
    return roc_auc_score(y_true, probabilities, multi_class="ovr", average="macro", labels=np.arange(probabilities.shape[1]))


def class_report_frame(y_true, y_pred, class_names):
    report = classification_report(y_true, y_pred, labels=np.arange(len(class_names)), target_names=class_names, zero_division=0, output_dict=True)
    report_frame = pd.DataFrame(report).T.reset_index().rename(columns={"index": "class"})
    return report_frame[report_frame["class"].isin(class_names)][["class", "precision", "recall", "f1-score", "support"]].reset_index(drop=True)


def fit_scaler_transform(train, validation, test):
    scaler = StandardScaler().fit(train)
    return scaler.transform(train), scaler.transform(validation), scaler.transform(test)


def make_baseline_models():
    # These baseline configurations are fixed in advance; no hyperparameter search is run on TEST.
    return {
        "KNN": KNeighborsClassifier(n_neighbors=15, weights="distance", p=2),
        "LOGREG": LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs", n_jobs=-1, multi_class="auto"),
    }


def main():
    random.seed(RNG_SEED)
    np.random.seed(RNG_SEED)
    ensure_dir(OUT_DIR)
    split_root = Path(SPLIT_DIR)
    ids_train, y_train = read_split_csv(split_root / "train_split.csv")
    ids_val, y_val = read_split_csv(split_root / "val_split.csv")
    ids_test, y_test = read_split_csv(split_root / "test_split.csv")

    # Confirm that the three split feature arrays align with their corresponding split files.
    feature_data = {}
    for feature_name, feature_dir in FEATURE_DIRS.items():
        X_train = load_features(feature_dir, "train")
        X_val = load_features(feature_dir, "val")
        X_test = load_features(feature_dir, "test")
        assert len(X_train) == len(y_train) == len(ids_train), f"Train rows do not align for {feature_name}."
        assert len(X_val) == len(y_val) == len(ids_val), f"Validation rows do not align for {feature_name}."
        assert len(X_test) == len(y_test) == len(ids_test), f"Test rows do not align for {feature_name}."
        feature_data[feature_name] = (X_train, X_val, X_test)

    validation_records = []
    selected_models = {}
    for feature_name, (X_train, X_val, X_test) in feature_data.items():
        Z_train, Z_val, _ = fit_scaler_transform(X_train, X_val, X_val)
        candidates = []
        for model_name, model in make_baseline_models().items():
            model.fit(Z_train, y_train)
            val_probabilities = model.predict_proba(Z_val)
            val_predictions = val_probabilities.argmax(1)
            record = {
                "feature": feature_name,
                "model": model_name,
                "VAL_accuracy": accuracy_score(y_val, val_predictions),
                "VAL_macro_f1": f1_score(y_val, val_predictions, average="macro", zero_division=0),
                "VAL_macro_auc": macro_auc_ovr(y_val, val_probabilities),
                "VAL_mcc": matthews_corrcoef(y_val, val_predictions),
            }
            validation_records.append(record)
            candidates.append((record, model_name))

        # Select the baseline for this feature using validation Macro-F1 only.
        chosen_record, chosen_name = sorted(candidates, key=lambda item: (item[0]["VAL_macro_f1"], item[0]["VAL_macro_auc"], item[0]["VAL_mcc"]), reverse=True)[0]
        selected_models[feature_name] = (chosen_name, chosen_record)

    validation_table = pd.DataFrame(validation_records).sort_values(["feature", "VAL_macro_f1", "VAL_macro_auc", "VAL_mcc"], ascending=[True, False, False, False])
    validation_table.to_csv(OUT_DIR / "baseline_validation_metrics.csv", index=False)
    print("Validation results for all pre-specified baseline candidates:")
    display(validation_table.round(4))

    # TEST is evaluated only after the selected baseline for each backbone is fixed by validation.
    test_records = []
    for feature_name, (X_train, X_val, X_test) in feature_data.items():
        model_name, val_record = selected_models[feature_name]
        X_fit = np.concatenate([X_train, X_val], axis=0)
        y_fit = np.concatenate([y_train, y_val], axis=0)
        scaler = StandardScaler().fit(X_fit)
        Z_fit = scaler.transform(X_fit)
        Z_test = scaler.transform(X_test)
        model = make_baseline_models()[model_name]
        model.fit(Z_fit, y_fit)
        probabilities = model.predict_proba(Z_test)
        predictions = probabilities.argmax(1)
        test_records.append({
            "feature": feature_name,
            "validation_selected_model": model_name,
            "validation_macro_f1": val_record["VAL_macro_f1"],
            "TEST_accuracy": accuracy_score(y_test, predictions),
            "TEST_macro_f1": f1_score(y_test, predictions, average="macro", zero_division=0),
            "TEST_macro_auc": macro_auc_ovr(y_test, probabilities),
            "TEST_mcc": matthews_corrcoef(y_test, predictions),
        })
        cm = confusion_matrix(y_test, predictions, labels=np.arange(len(CLASS_NAMES)))
        row_sums = cm.sum(axis=1, keepdims=True)
        normalized_cm = np.divide(cm, row_sums, out=np.zeros_like(cm, dtype=float), where=row_sums != 0)
        pd.DataFrame(normalized_cm, index=[f"true_{name}" for name in CLASS_NAMES], columns=[f"pred_{name}" for name in CLASS_NAMES]).to_csv(OUT_DIR / f"{feature_name.lower()}_test_confusion_matrix_normalized.csv")
        class_report_frame(y_test, predictions, CLASS_NAMES).to_csv(OUT_DIR / f"{feature_name.lower()}_test_classification_report.csv", index=False)

    test_table = pd.DataFrame(test_records).sort_values("feature")
    test_table.to_csv(OUT_DIR / "validation_selected_baseline_test_metrics.csv", index=False)
    print("\nFinal test comparison for baselines selected using validation:")
    display(test_table.round(4))
    print(f"\nReports saved under: {OUT_DIR}")


if __name__ == "__main__":
    main()
